# Crawl corpus ViBioMIR trên Kaggle — tự upload Hugging Face

Notebook tự chứa code crawl, không cần clone repo hoặc upload file Python.
Phần chẩn đoán đã tích hợp: Output tự tạo `<run>_diagnostics.zip` để debug.
Bật **Internet** trong cài đặt notebook; dùng CPU, không cần GPU.
Chỉnh ô **1. Cấu hình**, rồi chạy các ô theo thứ tự.

ID bắt đầu và kết thúc đều được tính vào khoảng chọn. Đây là **ID gốc**,
không phải số thứ tự dòng. Giữ nguyên ID/URL nguồn, không dịch hoặc sinh lại text.
Trích HTML/PDF có text; Selenium thử lại URL lỗi, trang JS hoặc text rỗng/QC lỗi và render JavaScript. Chưa có OCR.
Dùng chung engine với `crawl_domain_samples.ipynb`: header trình duyệt,
timeout/retry có ngân sách, giữ lỗi theo từng lần thử và kiểm tra challenge.
Nguồn mới: `links_corpus_vi_en_zh.parquet`, **1.220.598 URL**, cùng snapshot với domain samples.
ID gốc từ 583 đến 7.466.161 và không liên tục; số URL không phải ID cuối.
Tài liệu nguồn: [ViBioMIR](https://huggingface.co/datasets/AIGuruTinix/ViBioMIR).

**Bản tự upload:** `crawl_corpus_kaggle_hf_upload.ipynb`.
Mục 1–9 chạy quy trình crawl/xuất; mục 10 tự upload output đã hoàn tất
lên [AncientWings_R2AI_STAGE3_ds](https://huggingface.co/datasets/AncientWings/AncientWings_R2AI_STAGE3_ds).
Trước Run All: mở **Add-ons > Secrets**, tạo secret **HF_TOKEN** với token Hugging Face
có quyền ghi vào dataset này và bật secret cho notebook. Token không nằm trong code/output.

PubMed dùng EFetch GET theo PMID; các website khác dùng Requests/Selenium.


## 1. Cấu hình — chỉnh ID tại đây

Để cứu run đã crawl xong nhưng audit/export lỗi, đặt POSTPROCESS_ONLY = True,
chỉnh đúng khoảng ID và RESUME_RUN_PATH tới thư mục run đầy đủ trong Kaggle Input
(hoặc giữ None nếu run vẫn ở session hiện tại). Chọn SOURCE_PARQUET nếu nguồn nằm trong Input.
Chạy các ô từ trên xuống: mục 6 tự bỏ qua crawl; mục 7 chạy lại audit/export.
Snapshot crawler/runner/probe, cấu hình crawl và manifest nguồn được giữ nguyên.
ZIP diagnostics chỉ có một số mẫu, không đủ để hậu xử lý toàn run.

Mặc định chọn ID **583–1582**. Để thử trước, đặt `MAX_NEW = 10`;
sau đó đổi thành `None` và chạy lại ô crawl để làm tiếp.
Giữ nguyên cấu hình tải trang/browser và phiên bản code khi resume cùng run.
Với khoảng lớn, chia thành nhiều run nhỏ: crawler hiện tại nạp danh sách công việc vào RAM.

`SOURCE_PARQUET` để trống sẽ tải nguồn đã chốt từ Hugging Face.
Nếu đã thêm Parquet vào Kaggle Input, điền đường dẫn `/kaggle/input/.../links_corpus_vi_en_zh.parquet`.
Run mới dùng nguồn 1.220.598 URL. Resume phải khớp nguồn; nâng cấp từ run nguồn cũ dùng `RECOVERY_RUN_PATH`.
`POSTPROCESS_ONLY` nhận diện hash nguồn cũ trong manifest để audit/export đúng snapshot của run đó.
`RESUME_RUN_PATH` tùy chọn trỏ tới **thư mục run đã giải nén** từ notebook này trong Kaggle Input.
Để nâng cấp run cũ hoặc thử lại lỗi, đặt `RECOVERY_RUN_PATH` và chọn `RUN_TAG` mới.
Recovery dùng lại raw 2xx đã kiểm tra hash, chạy lại extractor/QC, rồi crawl URL còn lỗi.
`RESPECT_ROBOTS = False` tải trực tiếp giống `domain_samples`; đặt `True` để
kiểm tra robots trước các request trang. Trạng thái robots không được chuyển sang Selenium.

Khôi phục checkpoint PubMed 6.000/110.000: giải nén ZIP đầy đủ
`ids_5979207_7061937_synced_vi_en_zh_efetch.zip`, đặt `RESUME_RUN_PATH` tới thư mục chứa manifest,
`ID_FROM=5979207`, `ID_TO=7061937`, `POSTPROCESS_ONLY=False`, `PUBMED_USE_EFETCH=True`, `MAX_NEW=None`.
Giữ `PUBMED_UPGRADE_EFETCH_SNAPSHOT=True`: chỉ nâng cấp transport EFetch; XML parser/schema, core/runner/probe
và dữ liệu cũ giữ nguyên. Log sẽ ghi completed/pending thực tế. Không dùng ZIP diagnostics để resume.
Checkpoint full ZIP được tạo mỗi 10.000 URL mới hoặc 15 phút, và cuối lượt. Nếu có Secret HF_TOKEN quyền ghi,
upload ZIP/checksum vào `checkpoints/<run>/latest.zip` và `latest.json` trong cùng commit. Đây là backup;
run còn pending không upload corpus như hoàn tất. Có thể tắt upload, ZIP local vẫn được tạo.


In [ ]:
# ===== CHỈNH CẤU HÌNH TẠI ĐÂY =====
ID_FROM = 583
ID_TO = 1582              # Bao gồm ID này
WORKERS = 4
DELAY = 1.0
LOG_EVERY = 300          # Tiến độ mỗi 300 URL; URL lỗi luôn in ngay
ATTEMPTS = 3
MAX_NEW = None            # None: toàn bộ còn lại; 10: thêm tối đa 10 URL
REQUESTS_TRUST_ENV = False  # False như lượt thử Kaggle; True nếu môi trường bắt buộc proxy
USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0.0.0 Safari/537.36'
PAGE_CONNECT_TIMEOUT = 20.0
PAGE_READ_TIMEOUT = 45.0
PAGE_REQUEST_BUDGET = 180.0
RESPECT_ROBOTS = False    # False: giống domain_samples; True: kiểm tra robots
ROBOTS_ATTEMPTS = 3
ROBOTS_CONNECT_TIMEOUT = 15.0
ROBOTS_READ_TIMEOUT = 30.0
ROBOTS_RETRY_BUDGET = 90.0
ROBOTS_FAILURE_TTL = 30.0
VOV_USE_CURL = True  # vov.vn và các subdomain dùng curl/Chrome
LONGCHAU_USE_CURL = True
LONGCHAU_WORKERS = 1
LONGCHAU_DELAY = 5.0
LONGCHAU_ATTEMPTS = 3
LONGCHAU_CONNECT_TIMEOUT = 10.0
LONGCHAU_READ_TIMEOUT = 45.0
LONGCHAU_REQUEST_BUDGET = 180.0
LONGCHAU_RETRY_FAILED_ON_RESUME = True

CNKANG_REQUEST_GAP = 1.5  # 1 worker, robots luôn bật, không chạy Selenium
CNKANG_REQUEST_DEADLINE = 45.0  # DNS + connect + headers + body
CNKANG_URL_DEADLINE = 120.0  # robots + gap + redirects + retries
CNKANG_ATTEMPTS = 2
CNKANG_CONNECT_TIMEOUT = 10.0
CNKANG_READ_TIMEOUT = 30.0
CNKANG_USER_AGENT = 'ViBioMIR-CnkangCheck/1.0'

BROWSER_TEST_MODE = 'failed_only'  # 'off', 'failed_only', 'all'
PUBMED_USE_EFETCH = True  # Mặc định: PubMed đi EFetch GET, ghép theo PMID
PUBMED_EFETCH_BATCH_SIZE = 100
PUBMED_EFETCH_WORKERS = 1
PUBMED_EFETCH_GAP_SECONDS = 0.5
PUBMED_EFETCH_ATTEMPTS = 5
PUBMED_EFETCH_CONNECT_TIMEOUT = 10.0
PUBMED_EFETCH_READ_TIMEOUT = 90.0
PUBMED_EFETCH_BUDGET = 300.0
PUBMED_SELENIUM_FIRST = True  # Chỉ dùng khi PUBMED_USE_EFETCH = False
BROWSER_USER_AGENT = None  # Dùng User-Agent thực của Chrome
BROWSER_LOAD_TIMEOUT = 45.0
BROWSER_WAIT_SECONDS = 15.0
BROWSER_ATTEMPTS = 2
CHROME_BINARY = None      # Selenium Manager tự tìm/cài Chrome; hoặc đường dẫn có sẵn

PUBMED_UPGRADE_EFETCH_SNAPSHOT = True  # Upgrade transport compatible with existing XML; keep saved runner/core/data
PUBMED_MIN_BATCH_SIZE = 1
PUBMED_MAX_SPLIT_REQUESTS = 50
PUBMED_FALLBACK_BUDGET = 900.0
PUBMED_PENDING_RETRY_PASSES = 1
PUBMED_MAX_FAILED_BATCHES = 3
CHECKPOINT_ENABLED = True
CHECKPOINT_UPLOAD_ENABLED = True
CHECKPOINT_EVERY_URLS = 10000
CHECKPOINT_EVERY_SECONDS = 900.0
CHECKPOINT_HF_REPO_ID = 'AncientWings/AncientWings_R2AI_STAGE3_ds'
CHECKPOINT_HF_SECRET_NAME = 'HF_TOKEN'

POSTPROCESS_ONLY = False  # True: chỉ audit/export run cũ, bỏ qua toàn bộ crawl
SOURCE_PARQUET = None     # None: tự tải; hoặc '/kaggle/input/.../links_corpus_vi_en_zh.parquet'
RESUME_RUN_PATH = None    # None: mới/tiếp tục session; hoặc thư mục run trong Kaggle Input
RECOVERY_RUN_PATH = None  # Run cũ để reuse raw thành công, thử lại URL lỗi trong run mới
RUN_TAG = None            # None: ids_<ID_FROM>_<ID_TO>_synced_vi_en_zh; đổi khi muốn run mới

from pathlib import Path
import json
import os
import shutil
import sys

NCBI_EMAIL = os.environ.get('NCBI_EMAIL', '')  # Email liên hệ; có thể điền ở đây
NCBI_API_KEY = os.environ.get('NCBI_API_KEY', '')  # Tùy chọn; không ghi vào settings/log

if sys.version_info < (3, 11):
    raise RuntimeError('Code crawl yêu cầu Python 3.11 trở lên.')
WORKDIR = Path('/kaggle/working')
SOURCE_FILENAME = 'links_corpus_vi_en_zh.parquet'
DATASET_REVISION = '71be0e1239a19dfd7a717e2bc13e3bec21bd0334'
SOURCE_SHA256 = '87e44a792d7a99fb0fde5dff7f09f1a38a0a1fdf02ddd1ac93a005736e00b0a7'
LEGACY_SOURCE = {
    'filename': 'links_corpus.parquet',
    'revision': '0148f6f80ffafed5c005af6d506ccfd9d3fb47a7',
    'sha256': 'dee1153f5b9d9010dea18e1b7724b4915ff847b711d036bec8b933486802a971',
}

if type(ID_FROM) is not int or type(ID_TO) is not int or ID_FROM < 0 or ID_TO < ID_FROM:
    raise ValueError('ID_FROM và ID_TO phải là số nguyên; ID_TO >= ID_FROM >= 0.')
if type(WORKERS) is not int or WORKERS < 1 or type(ATTEMPTS) is not int or ATTEMPTS < 1:
    raise ValueError('WORKERS và ATTEMPTS phải là số nguyên dương.')
if not isinstance(DELAY, (int, float)) or not 0 <= DELAY < float('inf'):
    raise ValueError('DELAY phải hữu hạn và không âm.')
if type(LOG_EVERY) is not int or LOG_EVERY < 1:
    raise ValueError('LOG_EVERY phải là số nguyên dương.')
if MAX_NEW is not None and (type(MAX_NEW) is not int or MAX_NEW < 1):
    raise ValueError('MAX_NEW phải là None hoặc số nguyên dương.')
if type(REQUESTS_TRUST_ENV) is not bool:
    raise ValueError('REQUESTS_TRUST_ENV phải là bool.')
if type(POSTPROCESS_ONLY) is not bool:
    raise ValueError('POSTPROCESS_ONLY phải là bool.')
if POSTPROCESS_ONLY and RECOVERY_RUN_PATH:
    raise ValueError('Hậu xử lý dùng RESUME_RUN_PATH hoặc run hiện tại, không dùng RECOVERY_RUN_PATH.')
if RECOVERY_RUN_PATH and RESUME_RUN_PATH:
    raise ValueError('Chỉ chọn RECOVERY_RUN_PATH hoặc RESUME_RUN_PATH.')
source_tag = f'ids_{ID_FROM}_{ID_TO}_synced_vi_en_zh'
previous_tag = source_tag + ('_efetch' if PUBMED_USE_EFETCH else '')
default_tag = previous_tag + ('_longchau_v2' if LONGCHAU_USE_CURL else '')
if POSTPROCESS_ONLY:
    if RESUME_RUN_PATH:
        default_tag = Path(RESUME_RUN_PATH).name
    elif not (WORKDIR / 'runs' / default_tag).exists():
        default_tag = previous_tag if (WORKDIR / 'runs' / previous_tag).exists() else (source_tag if (WORKDIR / 'runs' / source_tag).exists() else f'ids_{ID_FROM}_{ID_TO}_synced')
tag = RUN_TAG or default_tag + ('_recovery' if RECOVERY_RUN_PATH else '')
if not isinstance(tag, str) or tag in ('.', '..') or any(separator in tag for separator in ('/', chr(92))):
    raise ValueError('RUN_TAG chỉ là tên thư mục, không phải đường dẫn.')
RUN_DIR = WORKDIR / 'runs' / tag
RUN_DIR.parent.mkdir(parents=True, exist_ok=True)
if RESUME_RUN_PATH:
    previous = Path(RESUME_RUN_PATH)
    if not (previous / 'manifest.json').is_file():
        raise ValueError('RESUME_RUN_PATH phải chứa manifest.json, records/ và raw/.')
    previous_manifest = json.loads((previous / 'manifest.json').read_text(encoding='utf-8'))
    if not POSTPROCESS_ONLY and previous_manifest.get('source_sha256') != SOURCE_SHA256:
        raise ValueError('Run resume dùng nguồn khác. Nguồn 1.220.598 URL cần run mới; dùng RECOVERY_RUN_PATH để nâng cấp.')
    if not POSTPROCESS_ONLY and PUBMED_USE_EFETCH and (previous / 'corpus_config.json').exists():
        old_settings = json.loads((previous / 'corpus_config.json').read_text(encoding='utf-8')).get('settings', {})
        if old_settings.get('pubmed_use_efetch') is not True:
            raise ValueError('Run cũ chưa dùng EFetch. Chọn RUN_TAG mới và RECOVERY_RUN_PATH.')
    if not RUN_DIR.exists():
        shutil.copytree(previous, RUN_DIR)
    else:
        print('Đã có output trong session; dùng run hiện tại, không copy đè.')
if (RUN_DIR / 'manifest.json').exists():
    if not POSTPROCESS_ONLY and (RUN_DIR / 'crawl_config.json').exists() and not (RUN_DIR / 'corpus_config.json').exists():
        raise ValueError('Run cũ cần nâng cấp: chọn RUN_TAG mới và dùng RECOVERY_RUN_PATH.')
    saved = json.loads((RUN_DIR / 'manifest.json').read_text(encoding='utf-8'))
    if not POSTPROCESS_ONLY and saved.get('source_sha256') != SOURCE_SHA256:
        raise ValueError('Nguồn của run cũ khác bản 1.220.598 URL. Chọn RUN_TAG mới và dùng RECOVERY_RUN_PATH.')
    if saved.get('id_start_inclusive') != ID_FROM or saved.get('id_stop_exclusive') != ID_TO + 1:
        raise ValueError('Khoảng ID không khớp run đã có. Chọn RUN_TAG mới.')
    if POSTPROCESS_ONLY and saved.get('source_sha256') == LEGACY_SOURCE['sha256']:
        SOURCE_FILENAME = LEGACY_SOURCE['filename']
        DATASET_REVISION = LEGACY_SOURCE['revision']
        SOURCE_SHA256 = LEGACY_SOURCE['sha256']
        print('Hậu xử lý run cũ: dùng đúng snapshot nguồn gốc trong manifest.')
if POSTPROCESS_ONLY and not all((RUN_DIR / name).exists() for name in (
        'manifest.json', 'sample.jsonl', 'records', 'crawler_snapshot.py')):
    raise ValueError('Hậu xử lý cần thư mục run đầy đủ; ZIP diagnostics không chứa toàn bộ checkpoint/raw.')
SOURCE_URL = f'https://huggingface.co/datasets/AIGuruTinix/ViBioMIR/resolve/{DATASET_REVISION}/{SOURCE_FILENAME}'
SOURCE_PATH = WORKDIR / 'data' / SOURCE_FILENAME
print(f'File nguồn: {SOURCE_FILENAME}; revision: {DATASET_REVISION}')
print(f'ID: {ID_FROM} đến {ID_TO}, bao gồm cả hai đầu')
print(f'Output: {RUN_DIR}')


## 2. Cài dependencies

Lượt mới dùng phiên bản tương thích mà pip chọn trong môi trường Kaggle,
Phiên bản thực tế được lưu cùng run.
Khi khôi phục run, notebook cài lại dependencies trực tiếp đã ghi nhận.
Nếu bật Selenium, bước crawl tự kiểm tra thư viện Chrome trên Linux, cài phần thiếu
bằng apt và mở trang thử trước khi tải URL. Kaggle cần Internet để cài package/driver.


In [ ]:
import subprocess
import importlib.metadata

package_names = ['pyarrow', 'requests', 'trafilatura', 'pymupdf', 'tld', 'huggingface_hub', 'curl_cffi']
if BROWSER_TEST_MODE != 'off' and not POSTPROCESS_ONLY:
    package_names.append('selenium')
requirements_path = RUN_DIR / 'notebook_requirements.txt'
if requirements_path.exists():
    pinned = {line.split('==',1)[0].strip().replace('_','-').lower() for line in requirements_path.read_text(encoding='utf-8').splitlines() if line.strip() and not line.startswith('#')}
    install_args = ['-r', str(requirements_path)] + [p for p in package_names if p.replace('_','-').lower() not in pinned]
else:
    install_args = package_names
RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / 'logs').mkdir(exist_ok=True)
installation = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', *install_args],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace',
)
with (RUN_DIR / 'logs' / 'pip_install.log').open('a', encoding='utf-8') as log:
    log.write(installation.stdout or '')
if installation.stdout:
    print(installation.stdout)
if installation.returncode:
    raise subprocess.CalledProcessError(installation.returncode, installation.args, output=installation.stdout)
versions = {name: importlib.metadata.version(name) for name in package_names}
print(json.dumps(versions, indent=2))


## 3. Chuẩn bị code crawl

Ô dưới giải nén snapshot nhúng trong notebook vào output; không tải code từ mạng.
Snapshot crawler/runner/probe đang có được giữ nguyên để resume đúng phiên bản.
Auditor, exporter và diagnostics được cập nhật từ notebook; không thay cấu hình/hash crawl.


In [ ]:
import base64
import importlib.util
import hashlib
import zlib

EMBEDDED_FILES = {'ask120_snapshot.py': 'c-p;vYj+#Rk?41y<Nq+0_v~OK5PCRwPZkcM&=MW<gccP(oXv&c$q*QlvjQ>j%s>=Z<T;WZ#j;m^+`Zl?w&lbx#ZJ7DayL#aOSbi2HUvn0@?W_1=<eyB2T0l3^=?F9rn|bjy1HIn)i`kAz{%OCt-cq<eq?#QjukG!r*qy?*PnQ5;prLc$0w}iFtUywJL1J>t>*@QoCIMHdaU?y?cs+WerRFUx7y)a-)krEH|h^!>#XnhV=L@+H>{uiEb-4LKl@qDvKCe&-|JX$5)InPAo8nLKk_FMFX&E0{#tm>>sr|M;fD|n7`7b_dWj#!(8JrXf}WMYNPRz=m_BK>SN-<cei-x;EAnB~j<w{+LC2>yk+*Ik?ttEb0|y>{XgLbkEZ1Eg;Ha)^1#A5<N?^9VF!69UG0uZpwZr~~eBXgt;P;0g5)W3rc(ofWsqc|5dJUp3Jg*TIWp8*E_q#z-#ZUhXn5r?l=hwQj)oQZV#TO4h^vu-U{PgT8t6|wp>j_?ziF4nvAAV@+pBCmOPb|#NrSIM6Jhv0Bx%B?2+4+Tecy~d(tig)dRm)yqU$1FTwjO@y$BR?*3owv-dUD|@=n?t&rXO_uN@O3F>wCC)Wa1xNhtAXhLEC{vo}8XB8$bMz1^-3%nfT!L-5cAtUmpGH*6`Nn&HuW1{lH%@Uj6IES6h|ir@xm3N!R~=^V$Ep_zpaN<*yfCJJV}jI9fe+<j9uOOrJvI`tH3qw{QR6Ie!1z&HL~FYUiz=5AWXDy?N7?Lp8tt;%fD;7oY$7i?<&?6SocyZ(ki<e;WoJJ^%6O`b)#xH-|6(4&Y?l?c8_=S`07U7~c8JISy|xJpylQ`0DQfI81M5@`<S#q92cCM}FK7dmTS=y<WWTM|Rz^hj0IS=MQh}Uc78qMXNq=kKaq&<*?fg*9U!UHG2L7XtI0nOPGafn1a)rDuBE1-56fJ%wJWXLGNrYT<@7ZUVnAx?jN@Ad^q~^C2R=mIeGHw>G}ETCuXMH`Ps#}6I1To?93F-rDFG2H{zfjcs*o={;C(PdF>!h@LO*H%!MmExw!Du?40}L^wi8r?&S@V)i82<-kOgs#n*lmF2lOJz|Ht{6}B==2eul1dv4%$gXN%2yNMt2ER6p$&4@LycN`9)w(mw^?yC!1KN-Xt8vZ%h<-Inc{^P~Tndv8|PfQX&&n@z_lR+QYvx42LCU*mCJiB=&b}F`Y$fD6>_z>An<#^*vy!m&I?Kq4!N<0Q2nt}AGq%$%pOMv13hcDgQzI}K5?%mDw&*(ql&zG;=fAjs#(Mum~-+l`?bm#S7h%e6Y>fPb(E1TM%4?e*^J8%33kg1v@YRfln$eo+-N$|(FZ~x)=?kl$tIR5$IlPpwbgFOK{$44)Jh*0oP9!fvnXs?FdaK*%K`}W11J0ERs-?>VEZoh<o?tk(t_~RTO-g^NEG5oiWcW=E<lc(=Mubp3BgYVfPy53S4c}W;;5GVR^3@qez_ZqEqKG0$G!NvQ(ze=;>PeA+8%NKc?7jc>?637^f)9%xgb3Xz(joU=*(&6R5Y~TIm&i}i+{lyn7w~$$P-o8D&{HM_`U)=fR-Sj&y!T-I4-+76L*FWC5b8mS0mEBK2XF}k+-CLh+e|c}`jgN<4-Q2$S-sq)IU}o^I5)7l4|9$t?OVsqw@7}+7iGkgJ<F})0*J!ZkKOeq$cl4VZJJ<hp_~A#xx4*)!FoC(Lr)Qs;I_WM<{*ZM=+>U}iiGw)V;LR5-M`GW@9v~I6G)R(=BuLQf4_J7FTF-MUs8%HG6El<Z^HaJgZHFDdAI1UhdF+scgLpuIT~JF);dyr{Veo4!BugUf!Jj7hqky!8!r1F{l7KOk-=(DL1?{u)GqwTYvDO!2692E`FIzpB6$qh!@hfO+ICY_0d@v1qC@tc_QU#UTAp~N<KiJFyabVj{4NRAyUvW}s!tU7*EE7};B8Xk7bWyfe;C~Vl82}w1rL-j~fB-dcjG-#ruoi<=wC0nnvR9L&PjVXm#<oMIoou%n#tG6%P7BnFA5|ReNOpuNy2%>TjfUFtpxa&ZlJ=?u2zqc8y2n7>#5xuBd~JSiyX(cVA6I&?lo=Yueo~?5wH5eeQ%k^AQk$mi=&-;7V|di7YT+z0Flk~i6LVY*M#o2{sw{<_jVdf4n3$l!la0RLfI$T;)^2R@#J(57R3p2+inC3?-~nxMuzBE&-8x=ro>`x09dc|F6i<QF&$L*pH9zr8fJ=KpL=>C#qyO;OcfZpzV8Il+UPP?HnATRJaL}(Dg+*AF$opL|MsZa2vL>c(dvQAmXv?j@egIvH-3USIWDAR{4ND(%yae=L!)nSwq1zx?p7<{IUs*c(-6M}3v0K~$wg(}F>#Au|1qat91zJz}3^aP!33@BgSG|{PneIB-8jw{Th}l<`jy-nt7$bBR|D2z|4^G|WWNj3Mxd9c~S2Y5}^-*YU8GyLtL2uxvPpMT6WKu=;b@Z1T832N3aiQhn$bKZ0qoAFD4zkjRPkOLI8sW~j{eEIqX2S67pg$EwVN|siK@b9{lLU~SIb#toz#?7Z$nSSOw705??UcX(65(#IS_=`vQd=yE{5>`p57W_Sw(dneUYznR>PSA*2`SJ=SE_~%7Lu*EQV~Nq1!HhuNxNsf?!XtMBOpHFUpr0i7n9HOVr;Gt>IJY<(ykTZc#-2NMInhtbfsa3k?RLAG%}Ib>$|-WKpEn(-36)wA*2T?WtiGft3kJeOxXp4t3s`70l3N?KW_WIj@L_Kkel@!ZwGPE1EmP&bA>?E5)X(1^m6uOY#~i?b);Y=5azZ_!RkuIusaZa0qgIrjMb3$8A1$7Y*eICfY8n%%@ueg2R$L#!hoIJbO@_X6!w`lP4!9i)0PE^?V_$0O6FkIl0I02wx0w~RE3xWtTt|gJWDD;D%P`L#T6tuga+kN3u7zi{0Mv|kgJKm23rxNXbfgBkgpSXU~&(9xkHcwodCiiiC~aEp@&hL6e3KZi@R*eYoE1}(1NelIp7OmQnpXl!vS=S!DH$rT@1Uly91b`w5<-KVRd+MS_2AP?yW>#f0fh(4{dNMPd`{5!+!<dqB1`6Jk<aJ-j_;8*Zs=41JkXBL6=JUs9(eRxq?`*KM2V_BACv=`Ifs(vrwqJW~Fumd!KySrsV}%5_cgg%w~d;c_cE_3kobHyB&G|vf=jpb!iNyc305RjIP9sl6XBxRx2d>H*BY6+E+z121q7x4Xi6XA$ZcH3A7x;sOy0}1O~8(w76|A>KM`xRtLGeVM&bnft>a_5Yd;7G#j*ZVYsPW85pu)WyEB;sln=u3V+BWBl?{(B?^Rb{EoZS4cljdFJOKdo3&t}q0wIJjPf?W&NL}p2U-kX*pgO)DV@B%U%n<tVf@z8^%z7-j1?<nz${_~;xZR?lT;Fqf$*pe3O89NY?EtfJLcvkrpa$x$yyY^8wXdqoDT#S2#_L;1Qi1n%sCh4%R?ATseHfkv2yWBgA0pLQI0NBC0O2YwcuRE2<OT5goyhhSVmsY8i2l73RVW-?{gSU#_)}P7U(u4b<YPyN3W%-sZsDAVX}g6XtKu~T1{OvY{<BbT|k5o#)!bOWy@7Y8_{rx62GI9KXv`KZ@&WJ?akrmf4Tq8uQ%a;9AMiVUH|>=%@22PUD_Oeby<9pQATKW1|ye*lT*I(DJp{=o@K1`V1g14a<f?(h=}Hza_8_fFrL(8{2G=Rc(RA2E(nk2)GT&fV--mKYRdt*U+}5IXtxwOz#gbvz%jJJT=;+NEU?@n&3u#?9`_R<hYzqPL!+)P^}}~*PlG&Z+!evYs-r1J4H;efl=dax6+bqQwanq19L){bB}>2G<x%1f%(7&|<@j`k1J(_DEr9*f!+_SogED5Fk@_8Q?kZZay}{$;<L%-_8gg%7Y#|qO$ESHasvzbr;AvaAq0KfQes7plFBl{foNw^d5^%&IdWho+K>P=MI`ASv5P<>fvU$tsHRO`u(P7~cLeYPO>5>-*fT%fJ2BWWlW_Jjj<DS=#S3`)c(a@PSx^ULts4F*&n~>22&&9rjPsokNIm(tE{{n3(xdT7bfuI>^Vq=dD0HyuUV^pnT*A5>(*hG!#f$a>IR1fA>Iba>ADq%PgLk!Zd{D5S%Q-yc<k2G$QE_`p<Ia=D^7;fA@N+9z_dr)T0>1Kl_PsHpU>k$wec?oImEL&+4JB@P1B4U+pH!v|U(tv~Q$w6XAr6H+C(*+|9j~NhWwH{)umXnDNis#KE%0!L_U9=`LGC@a44vI&&VyuQ5l;1hy@fH9uRMDV#MG}gT2fs-J9=LG>5lu&v-o?Xq;lnyQ%1!+|uVrXj5n#;tPTU!#R-6VQ6~`WI#(sCXUiZ-W<d5gPNO6o<ed^6?Z;|PnXv5M=AZ&JEgG(6o?8#sY9_2(2%__q}jK0Z0d6hw<r1M4M2;547HDVF9xBH+BPBrHUnue0TDh#%8Q4Af&lgF^?-Uggt?g)fdLnbZYX#<DZ*;T~1beiEfl7iX`hl$DFk63AfhZcc4#K0$xJglIwRKbCTw$1{h6e=%CZP3R(YiLlDf3ox`)U61;aKGjCtMQ~-s9BlpMc{O8%S%+CX!%cOYv%Y{%{;GQ(J=_S=TJ-2;uMP+IyP}K$5qVc8SNOssQx<q@nT4m5wxyi)CfW`9l||sl%ycYHI#^>mqq=VV{!-jxFs%ZNvvo<)QhRgs35ZxfS@x!>Mnta*Lf*G;wB}Mxllp~nLzU5nup4_%+lM=NGqCA&FlBkXt#yEFnx0B)WY<_Pu;nxXQroqVi#><w`~9Fv?xfyplzD^GhtVp4W_o9aE53?6>Ur+lL21O(NG(yXk^an*w56FNix&+yAn|$7Ys(*8|3M^*(ayw=gHqO=_$FKx@p##^mV1KPAEwY&fYLsQzcxqs0N`XI4GCUlhsURc7W;u#0I2_w<wLTdoWVh(u==|aY$hewB!yS_SbAPkWUk(Fl#nsK(bs@-wo4rD-c|?Jx!ycOuLVTqbvB_UPJhXd_(Zgp4$9xNKqMpVv0&d@2{nr-qS509-Shwg~rfcq{61mtOc^r*_#>3r(-clom{Yh$c;{I=x@fT-+c8mh9SigG6qG-QmQ#-ojRt_eif>iH-~E}`t3dd4K%f{YaZd5GElTJLESU#RER(#K(FU_Z4vf+5VN+?C{t(Ve{AL=7LgcU43P4HJ>DdMI5*;o@06ItFPRV*MD6B$4tzrBW{^aX*9&V5etyh$ETnSzUE+L<_OW7_kU<#MKWXT%<D<`#F$8ebA`5_E!n>v&_kHgy&J^`ki$u26dfL_yz@--9t_I(x`?XRCxS{I;TOlw;G3JBqmE3f)Hjy*No@wazj8i^2d-TqY(Vf?KFJ0aF=#O82@fPJk(v*tOloO8!%ge!ebe+Eb;tF<wzi*3UA2EQ#PhTSDmCI%1kqCZTK5&8aY@0kTld!dAU0`l%LV{VxKX13TF7U8hfaaOr1>&x)1Nnn<9Ewt}q}5W_f~J5oCN0YtE@fzG$F_-Rg&}>B#6=H8bk&w-46=^O^3n&<CdMdenHk#iW}U&(xmv|()gqD3l1rNC50>DdaaBnkN`J4QJ1B=FLg-t;IQ*uh1);?bAExtbC)H%Ns&!!d&K<}x9)9==MQ^luFyI<=K~90B`buxGwgf&bhd}5v4gn!?j1PlRhS{{7BJN-UxCCs|X|4>vn&N~90kN+F^|7)gFky%vIeEEnQb@2V$!ncBbkvRMzXlnKb-7ih%q`O978ywKS#QY*bkXs6<C`3S5^j<lk~{VDh05{oJwoJkD#wn{hYQEHHjn<J{OB|%4%ddCf7$|~A8j4yrub}h{R%#tcuYPNX@&)?p66TLqvjoIU%<uJH^Nb{C60Wjvo(=>a}3`cD|+*ocq4-3kX}Fa!^smrrR>?o)7hGX?;XZu-2UcLwAl&HZNh@|oHI+!XTSIT*1_+mIdqeYC#PrKCnslSo|ruGqbv}}?}vJU(Vt)2{piZ@&5!ZC=8apUtCxT|y+J1k;qUPB3p<~_+r-TO*73~^P&?LA6s}|4ODFXC+&CWeu`Hu@@Hq8qLdz5bl?17(1q>9pzJT+1N3XrMd+(2!(+47~-r5|!{ngIh*APR>ZyvpHdH9Pz5C84z=-;o5etlhQ_rjlsUtJqs`Qzx~tD|>6!x9K8YEUhZ7az5rktrc0DO6%eTorB;&@t$MRL9u#Iuyr13C#d_4w9G{y~Fokd~xT_pK8{;kL=p^5$F=gn~n$hYp^ZPg#iKwDT}!x&kaJ<%>H>ur|t4VzK69ts2N~qVcCsmK|pH7r`(=F?AI_M`1H(A-6v+JXQt*(&rB{%X{q@GFoA+jwWyqgcEYn~Vk+Ob^z|3NsXcO>X6~T&@z2Asl=@AH`&jG24IinsHf0unCXHVXruEHc^LuT~1ar|oiIJ}I_!*PShrfLumI%sAw(tC6boaf{&v6+=<;JbghOfOmy7m_)CzYMFP#oTVW%sw2cE7s5`{{3>S6Vk>rnHMu&xKvAb4m}rMN0ejm+ua5e5rss$Dust{)<;}bWZqw;Y%#Jx$-+q760tU_LtYUKmXV5FW-c$a`?RS!XGfB{;i+ye)Iv1uE6LvqMF>5|D)yv)if|LOn3CsMdm%#l@in6<?M2vF|;4OcO`9b?@cZO0)X3h{{vyZaSM~zX)6}bKF0hB7n6$g=@(8F<_Tx6FhivHwNGK5+>8-dMMEW1Rg1+FXdZw<rPgtt$M)?zVk+?To!0UEjNmL8XGBD0vZ&eZK7VU?^R3M6;N&Uj`KS)ih88gye=Q(05oc)7oK|7?-mj3$?q1yf{0f~nzq<40m!s<!feAn$h*|;KzT3B76?A#$eZUR=1yG_W$V6ks2XjAYyAZ_cLim3j><b?K=l5W&ookn=<DJp9-;J)_7+rcFR5*(<MS;<OzLz%p<L%vh*SUdF86h=vyy{g9UqGg2a^YMJ-XHs4|4<Poi;)jT&miowk)Z2G+Mg5s=qy7eq&iM*)j!_}p!hJ+6Y-T&cDw$H*WPf~ya*Cvjl_KLDHhg4mUAf1;QTS!fP-&L8LV=(BAw@<mta9shu8NhkquN^f?^Od(BjlF(F*t|JCvaDt1t>y0w_*FiAM#EWa+o5(lBGa#feWDXpslf6I7~Uzf=AybYR%EL64ErZCOpp8%kE17BUGo6QNlGd;4tqNXs&i=><-j8ejre2?7#yOHVJH<7{E3CI#;>dCsIGFU{is+9%fo0!;B?<Wwu(SX&CasI8)N!8u>cF%-5}W{yx_(S;+Oj%Tewaom}n@Skg3oRBpwz^bs*{0T?sb?K^9CBZeyiZzoi)CHR#*mXO{Z|VG~)(#69>a@6rtfOO&^K2%4$3^C{>tZ1=@yG{=a4tYQ20+2il5FK*HQX{Rf$^4@V#_pDktA%Ps%n&`btXA8IC8{0(t@%StZ+d=rIPGv)$s{=6aa;*Fz`uS;Q$SMl&zhdyHk}k)6I67U704;W)q!D6JrsbOy*-{d}6b&B{di1F=OpOE|+vEva_WEHFPco31JN@;(@H*Nuw$`fD~efc##b(2mi)C^Gww)NHb)w!5l^}5_(XSWbVp-);%^CJx-}?dNMf<h_&U<y5#J-E`?yyMP!p261Z3fDho}^UCLRH^&)9SiMhGVB^og=A4Ru4Gu?UT8M0g3QgSq;s%<)3g&R-bF5hT#jh91ZWT(mvrmJ0-1Rf)X!sTV~!8uk5VB1~a>(O>~g&-0mj)TFtbkL;h0g&bes&8^B@&TevhCnbE8K1*U@F``mVZj~@hTnyB-r8sKkCDVCmbf(z%L2j5kM<!}_Yic;D4kI!@N$r}Tk|1JU$9APhnx_nd3;@gzXkycvM>lRr`+RrLWg5p6GBl~wCBZr38^&_0}No~wumdxmVcJ(V!?bvfAh*OM!$Z2b9D8^;jJ&giP`<=k|;C=gA#0KN4c2jYv>h=nF1_g4Y50rWo2+oV&IsuKTN45UmCg)Eg-<knL$|30})gbm(&8sTno;pap<!FCJ@SpluV$YQDFd~!ox8Hvkku?#v)9$=5!Pyb1V@pc?5v18`QFDrMn>iL@-GMd4!(gp=dg>2I6jV4GmL}QOnFiZ$+v!<y_>dPO6sJ?AkVOU<^_6)wruD_yUS=QHzo(^59(P(&7%%!Ki|HoyHe$tVbx<D4z<mnWnms7Xpfh^rt2SjqK|box1EVrK8Hj;rb*;uVLb;iw^S0Io3-BzER;vS@DNi`%zkk1lN@`a%K_@D_RgcyE4L=Nhvx^Z#UrsAEii|VZKd9)55U}o~ftMv^m5P3J<?4j{Bo_=0q5t<d1pj)FG2C1P$|r3M%<Wc5(`;m5pL$1zltUAcQ*oD;HXUN+;;0MvO+E<_|E*#|(LA61>mUKvJ&tAKP5_&TX!J$J-27RvZj39e=ci7eiD%J<X{~o<oKlOhBCyMxQ)QutVwOQYz{abSVtG6`dPk&e%)xNs@T&Rop$UTzws%p0>7Hc4L-aP>0|A0W@l<(w~%fv>h>q0W*YVrjms+MvV@>%hx}#uuZMSGVi{4&!lq9R2OChS*|?`@47A}SixnRv%$yB^EJtPnrL`%%qzDHeleEc0mG|<zB(%~K|ww^z;Go{%-+(AO0pUti@4Mh9ZkgNjdGqv9(!>DBtV$@tI)i;U}*FDUTQdFnAxQDh*#}8{#&&{7`ki<!l!KW41e48%!6!u3CytQ;Gt7h4m^kE#QXsuf5-vC%cUSzAsElD(?%?exG-sPW9std@4Lz4h36cCqgbLiTf)T9Ju8*elD|Je7Q{tinD7STjRyNNd+UL^V<I&)Vj~k>%hJ<ThH-?I(UU%FYE=zoh>%TPFd%G0LS$s8JlQ7wMmhHaw#v2FQ|(!VfrITO-h}p121wvLTsV?oq!n9&W1}_3Z_YU9f&`X1c*Yq?&tW-|DNw7bqu6vhXg910=T71xg+6p?nSl2JsGdh%2hCvX#q%?dc|mq2QjW>HHt=+os$$_?T5HWOfrbv3m!<tOgcM4LlmwQ5P&q!}6IXOKq}2kciOR#@Ovpp#S1PFrhNP`{g91IaCOs@WB^U_@@p-R002`v?w!w1r1U*7M$E>dQqG+0R3?;tlz5`lE=pFG@24vWm+=!qeSq0a@6x=p&3R7WoxAHB{&8X9AkO%tJ2OQmWwObZSZfTJH`Qoemf%*@EPUAoa&&QuSdZ5KADq5SA)Ze_q1YpZNy49n$N`~i2j=^ayJ`wi*`*4E0iZZn{-wGV|Gx7_lMp92O;58`8loJbNc-x9~gin!PWuT~sX_44HC1)W7=$$$5mKKmcM#d0QuV6a|Xn|mEhZ;z$4kYy{BL}j9lZ8U63a61#UJ4>BFzyA5v+w+meU?ti_0EjYTq4=EgmR0YiM^YtA?{ag4U(<hH?TzRB)ZCYl><7HvX%r&gknmsSBZ!0!&MEXjEKp>bpsH4OYyFilR>+@|C%k*t{Ua=B?1%IZv0HUfj-e*Zlb!;sn7CTqHUJ4b3PfehstfN5_Md|%BdQJ#ULD*wktx!aI$RrT}C%$I-*_@iHWCViPFa-QYQ!l9y;lZnY$bb(4xdQMiFQ3L&_D94Gnc-h@-mN=W7ITTjZ}5(1YYbFH==0ItHp@-!an{<pF%FAaow|iVjl<cid`v@^uCgt^AQLk14E%%rqD4r<tNM!^=%V@fHomEz=&Q33415@HJW3SlvS)hqAE@bZKOSB}X-J@We$Z4{~Y7-thG|njFb*z<##%m|d0`{ZZ>j5EQd~;82loD>8>VV5vd4*$6uKn?dK<2EA`_tcV&1#ziuUd-13wgBRZOK~!rM*r8IU7luC(TGix-%)yEXA(MyL%C{ot2{*eqD$8ezat>#{=eXd#fLhSY6bisrm<oqqoI*1l-_+$CiapR)c?5w_cjbXGlpDZe)5t19kQuHp-i}+Iqu=$6jLYu|f`g(-sB*C{okK^9NP%P{t&hv7Z4$Z{%6wG5Z;BHB6%Wv*m7FA8XOU?$N_5P%Ock~lQh~Zvzo4+fiwBg1!qc9u(z7f(g<#h;@hqNq_+dI5@j`hCN-CBPAhCiVhxk-IcraVKph-$O1b?&Bf^?%z*=UsWAFl>|nKWM4sszhA@#WL*a+mTFKu_A};n2r|qdD?^8hoPxZyLT;Ag5cZR78P?BMn<V(o^cSE}|{bzHziFDb-LU8}HPm3)j%ru&{iC_@OE)+(>fmqs&E3y(uWu(#r^n8uI2*t(?boU8g-Z^&G^I67VEFNF!q;s?C;@wl;J%?ol16^ZjgT7Ru~IUNIn>6^Ln7U0Yr>)R%=M*`j-fty5a<h!O%C%%<pQa}t287BuSl_B)q!b<m44D~8h|A;&2(Bn(qOV?Hf8nMy!uKIM){3hY3-B62))aS5Irj9O$&nsKU>Sp@?Q>J$;r3abTYh_vq-MJ0{-GYhf?S*@r@ZL2sYmAbL!Q6o<!SJ!VE)Wy0e<-Sx)76)iVMV8G4pCIuTPt8r8nEm0Y>HiJa4L&(LGc)^>#nbNm!rbDCg~hq4LKT`fc-femry?$4LJ9tZXgy33j2|g%Yu1yPmH^u}=!?dfS{*^o8{E+cu^VuI*TP;Fwz?&&ZApX95-uznXOM&4#%uZTATwOX^noB^TVL39%%vGijgX?dRnv_k3NdmbqtQ58rh}n5L&{%xsP7(m6`O85YsP6waFQd7;tn^&sGiMIG^^Pl)Vo9*q?zy@ttPRIL~GJhi@$1kWpPSnNfNh8@k3h4fssyiCPza)1Cl;og!De>_ANPkes<;=ov&wqatf{~e`@;lJ}=SXTC6%%fFC?a8@y;k6ieJO5?sSAMQPtVZLBqKcml$#mzB%K(u~<U(~&Pj$x7_~S@jR@GyD(w0K{=Q+aznD(TZy}Sz7LzU!*cSZ9i0(b>w>Ed}%p1Ro4^bQ8;)oSFlg>tg^bJYJJ^o#2srA0(_Thx%1Kxjg^7ksy=`nnJWX%-2E}$v~+SuVjNJ)tHP9o1-}s@W>C5ze{>3VVk#-{5(YaPf61S_Jg`e-2(LXoB0V1(;xw$p>^d$VJ|J<&D}o+KqAyUfP-^WxlHpNlU$(U1oH7KBSY_sBQF9&T>0Y7Pmy!J6Lh8@XoqlRET}QS3RTU|v6qT2hrtxNeDm;8+`5iO2Fs5IJNxKL!6uYwij5c?QfR^XMcVNmljYbNjV40*8P)=$&SZ)emUGE$SLB496K|X4o^b@=b8K&-gU>n3fW{V^~SWinEw1H~Y6c!+eR7SjzZ+Y1#8$nuYVl4cLtPa$!iQ?0m3h)x(Og1y!qQW)qcvf90aDr_K3`IHcyLj$YD|w+vtPkV(w1dod<1PxT6bWMuSdqD(uL&-Qva*>}=xx#K0cUWho>#GZpx1ymNDpb{`=x$`c8sn0#i~GC>+I-$@&!FkgqW#st??PnGEO6~P1S<APx-ZRAIVs_*}N11KJ&P>mGyQhKbVG~h@Cn(FUbxNHmmB8j|b5H#@jz=YK=-xj|<jRmD(51=Y%hvMX52i&@vt`eANk{>I9av4%XSbzrrO;voe$lV6w@R?nPWbCAO(6iJp``EN^IU7upag4pNpw@OB;lt=hRPB+9IYnh=R2C_DLaA0~Th7ZSG7IZ`t6x~{T#aM|LP?6W;Kdj>cjTAVj_R-<9=w1Sm!>f`5H?pd5%JUxR~+fKP4oFO=L8XOI{NlV;y%dsMJ@gAU{Ag_(m;hvtHKIKlHn*YgEHYP>%;<Kqjq8ZXr+;~weB9(X?d)r98EXJ54nmAi6zSE7um5z1>N;r{Ytq0F??9fU}%>*N<P9%WS4jNRPnQR~cplOv3N!Ji$Mk>)OEqO!CvaBxjJDA;P{z!vJUS~bc7pmD@D@?bG*oP`c(gbc3_(Q9%5))c~dkrYBv^FKb-7*l+^;t=bEn!IY5*J#{K24g|F7Y8(sakefG88eXGN!)qV+jc@WIvMy#@Y#x7iV?pqBzy`GgT}I>vI*>-iN|hnc1voO#bsA#tf&m5EHF7>GwV{0H)vDut|3ZZ{RZvVmsFA01Va5;mwzZmp_KzbHmHHcW>T<#U8$P?f#pu>kg&5x+8OI3i5}zI|a<4L(uFH-{JzlS9zo*vNXqBQAnp`l$hf@(ool$(vy1b(mKcif&?yCTs`W-)>QYCnzsdsl{V9B9t@uBtARALYD&SG&7_JkG1G-WDWG0xTDuB@axfYV-R#c;I=%SB%=G+I_=7RCi>DUc`KhNT;r7H6^BGBu1qTXFDL&d+hH}s}2&m8oP3dvrJzH6yi}u8T^7j=er*t^@X<WGFD;HcBxXQ@)SZrmZGzg-8OgDpglA6}}AVDg`l<alJjBEB5S3qV*GY*+yP>~2;paaT>;`Den17-4iep;DS?A8O#CfnNtj}Q5#BNAbJ+q`^~{`}(@Vg)XDaN_~!?zWD9CIJ(Vz|X)nSIZq+4p<6Sbj`z55;Wh{$2?J#0jiB*rxdHOoV|tlN+I6MMprcDUh7%|Ugl1k>C)wlCtn;C778;nIvZaF_uI+Y3d<&snEWLj@e|VO=O{_|%gao;E<yyG^bDb@Y?1~lM2v_&S4{}wy^K}~9@MoB$wy82;pG5>wW58))rv*iMu*2cWnTctY(yO*)mZKiqA17Q3*sQgaEx0pbyAX=#*q2-&$Y#`l7>`ElKv7o-E}T*^FMe{E~;7u4Z2*w%yQ~Rku7qvbJNYC1!V+90W`a=9-Dx(tz|q3>w8X#DoG=bmBz4A-kR7GI0xU9(u8%AA4NPittv9Un(zeelvvkeu12)esb<^8E?p+izcuFeMZJy;SRX<NnnE;(m8~?Dt9n-LVkIY0)8Cp<GrCpq?l;ihCfGvVmu`#))36yPq_m1c&NZ^Ks;aDBMp^k-3R8!)Qz{d98q%P%5Gtt`ODm!jjuwNA-K7mz902m$GH3+>l6rH6Cc7eH-V_l9K3{2a`~%4|7C<XYa%Gh1%Px|5ji$;lr6XTW2&Wn6RldhD*Fu=+CGcc51pYu|1ix}zmv5T&$IuHf5tb@Hc7vYS3eC0jP6F*$qcJ5<>iX7}`8y-x!pXYuP#F$+R8n4HoN?-(7Um{TEX>YTb%EeP(~`>(b%Ctr2Uf)%y=?hYpROz*0k)K-B(`bEU^<%qy_#3FDP{2oZ_PzY#f$vK5N(+ZX`F|P#VZmf?n#nePFe|^k>**Er;U`6c)DWLwrDA8b{3`*X&R(#_`X1p<c~zh3zC8ue*x5@Q`tS>N{J=}-gE6{nJ{cUxP;1jmgxqo=A2Hv$iA+BtJ<a;ti7f0g&a?=tJacFzzg?Aw#2lkn2o51A$KHTrd&m76@1_1>NK#E3#M;cA38%RG6)MD-inF_MV~e<N&!#;LqrpSm}>S-wiuQp(RoN_m=Kz$et>vZI4iu@FOFTl0gIwKnW`W<<<WzNvZzEtEdZ)~7;uH~P!FR6>Y19MZ~^e9Y2vD9K<Gj^*w`=Lx;XiYGk3s%C~<V-QtKoHFkQP0m47IT@fg~ER8+xPhpn0{T7!DED!!mnH1z@tc515VYd=Xjc1^EBs-@*MR)vhq!)V$~Pz4AbPx2PFTU+{1c))!~dgUQnPgrqL_M6Nh-2kCiEMm8U+CcirK0Xd+wIvjmuB)o1g39e|JICZYbLS|O<L^?wu9h%x8JpyXDsm621wLH0$PcR)xl;KtO1-0MR%%ttg_Dfhi<|qow&X0A5u*qa>qHj^;L>wJGejm~sm13sbeTe_m;bso(D^XMV*3z0O;jJaeUGjT;+Sd*NtN`qamc2&z^#n=BiD@fr?%nbuOHiWN?$FKomv|jB(Jesxr+oCO6t&LKxEW;t6CDo6RI)yv~0jKhhn^bRaA3@YmCkLBkuSA4*<>Tas',
 'audit_snapshot.py': 'c-qBW-H#l{aeo(}|DknCFtgU)97)MSy!W9{Bo#9jC4!^`0ms8=Znt-Le6us7`QY98ct8R_<S9S|2QZAlfMg>O9K*1K1PDSt<lzMWAD6$xzpCz@>6x9`+mqykf^K(vx~saY>#M7(XJcdI!|5~%V=2P8kh3f-PPfJhzQ%=^guyIM@**7PB8VsWUzBka9Lgw?6OqYhp*$A(T*^XBlS~9+oCU{`$jaE;*w}dWm2i<JSs{XKmIhfaUwvhoB?~c5FnnBuNt_Ga<4#fnV^)or{Ul7QAM+reN8w@pyI4db$obQk`8Y^r^-O)Bw^tTnR5kxNPhyQd4GMy0Fh9i4s#}IdCX4!K4L3i{tM6qTj+2R;1VvD_oh%|x7MYaZy+|%VyC?Wglf3st<DHiAxJZ(yT62_@{x}%V4Oz-8qIol=Odqu6X%rT(zM?xtKqM?WR1-)~gDgvqY4~R)_5-NX&uG}IuT10=O9>_}EqyS0^%a4CnJmgIX2N(;5=?S_>QO(xkSB#JV=OX^XS+^WOgI0^84PM@<7ANr<HDtmEr5=2ydNiF>{?Xv;v`!HQTP+-Iv>1u#~BJ9=<(8A8*n}kTpG~=syV>J<6GZ<TlMhf@?;XuWDW?{I)h`M`GWSsJxnI2Lt$28fKDV&OzaAJ?x~;EdWXWvWEury=?ql=a4O=YFvGmO$imb$z^qw*ki-(B<svOkeVWK58JENhj_O)ur^|zAvL0<MfsJRb#xfW(IMrRAjAdE~m-{L%dq`Z8-NTk=L-BDCm6E^r&#H`>v4X=$w&zCw{dL8S?jktJu~oZo4Qi4?(kCF0Y_Jef4@4wcWg;%pOy;>u5~cbnZaO?IWX`*6%}VVIZ8v=zTseufJY2|$5AOFfpvu6nJkan-f(_b*u-T;4ko6W)7q19r;<a7jJY;aUGibI*K9+@J(=7-gfHh5B>iq!xVu&EZoEhTYd$JhM#U~Hm_nc~VJU2|`>|C{wUR15nuRIA0Rg{ba(u3&l00U{^J7K|14$qkf@~n5;vv!1xFi+>*Bssd<WHkDhE<)mP4PhF_6PD@<;~T=+didb}T?nINW^b*)%Vhz&2W3>a1ZpVu4+dV4rZS$ms!wx6^|)FRgK`oUe!Xvc)KJ*_1}pKg%);rZ$dfVyX^XIkz^?L{%uYpch@FI<5xFuKY=hYRK#9f^Nt3~<qf?PgMKPDk{K!l&t7pmvS;R2fR8M8!L4$*3vF1}^?QB_6uZcI4h3n{r5F}?nJVf7Cam{o?vkg&s>!U~Si$$0(0&vVooS6v#*unXR#>fvPF;5@2^Hi$sST)Ze4gDOPROXm1@eYvZ5DK*bYu*@c@K&6vAQ8V{aI5vR5{ex9tpPk3lY){Bz^jR(r^Cdl1?wsb*>(nuduH#7uR%H^@gytd`L(DsDZv+z49Sl=XcWoV)pQz&9dYZ=M3!nE-w|Qzb*VMANYhvqc=~30SvGy2k)Vc>*k3C^0=BjY@CY~s@J=AG23X&Mr1GL53wnP0HxAU3u~v|Hq&u;8bC(q8fL3U_tWu#(Fb#dGXA`LYKxwe+&IZuhxXgI%5s<d?V;FpZ)<~L=pHL4o4en01Lc3V?a4&@S%nE+MQ?UT+y&^Y2TMTU_;>6G4#Sy?LSR76Q_R2=se=jc&-OPEKzrogFD6aP9Y*$HvRkLHZs+t<=g;&PbXRsC%@qjH@+bg9ZlT(?I+zgH7Af>v`01ilX>Rr|5g&(>xrgB@iU_Z%j$7HhYZ3&{twK}!oc%TH?M~3ww3{utwu)#tMvDI}@dk!#U;HV+QGMt5KItH0VGxSo6IH=ng9q)gvp)9Z932XS4a-!|Dc&yD`&CYC$hk`WBEXl|l<o>+D@6$82!NzAET8}@77I*ZpW~U~GNeGi2ocfH3S(QGqIJZIyrmMb_bNA5)w;w;g|2H4pyK8(Trv=n-g>h)kl(8G};xQRUf}eVMP8=<Kg96Rw>#zxR>7S#v4I|VSh8)kzU?$O~cSX}mH!~&hr#Ma9eV$MgKR<6RhXfuI?>Lz>IK-^hZP3)qwC6Jdlfo+;33ff3bT?!~2Nw5+ZDio1r73BiB~935@Q(QzvXJl7OP1wfm7trYDNL1+IGC?Hb%c9n5S$Nqbrn8qVMKtIovy%Z!RD3fPI-}J(sHqR!UnN#ILv8!YGiQLKvs<WQ}<*~D~M0s<{T>B38?TfEZ0n_&W3)KCn=yw=tJ&t&{~g1U_m;PrwpMQh=v2!9r!}3!_~6T7J^~L+%{7i#%-$!4h}KaVTFpI<K(ExQETcotwH;2kgsJK)G5O^I__!wG}d}9Rr;0>X=dA*xs6nHSU3yX3OiTt)%6xe1J_y<q9cAaYL~Y-HymJQ&#R+TQ%&baaH9c(j#=Nxxm~d;-CG=o#XKpCYMK2uoV{6##Fkr22cyazRZcA^Y@GWg85}hQZI#+KKx}$oy#4dMKI2m0%#V_IHl7Eii)EQDrV}N$f<qzHEVdvex}TrNL7LAIjYv<O#wPGNRGx%+kvFus4bAtbAzVA3POYRUZOApi95RLx&1k4ctv&@v6bus!fZF1-x&Z@j<?xL@l_rytz0MXe)vmx1V3+<3YV;gc#35Xs0IBNis*mOltF{(RgZ${`ts7TSRQ3Aq>8ap;OKK`0|G%_s?rS3gGt@y6D-!caZbHmh0~q0=;2(=DscoDj!$OS9jOetPlKDJD+<p>dq0EOAfQeOPIFpEWN0D+$svu13C|)TOa@jQAmNu~uw3YPCiz2P!k$w_Krz=nV)i||WQX9EC9lo|};zS?cd-VAJhaa>Lzj{(hhR(!JWDc9EqlT_e2?(n${AV}WD4Iv`q-2nHJ}Tq9Oo<cdsIDFlnsJBkGRE+}oCV`k@jkGNJ3ue=0zr<p7Ty*mMh7BBeh1l{$|7)Bu1N5`Vij<R*BgeYRvIyGzsA_yG(@9CwH>;Xj^5i8zzHUZe2UsxaVZnVtF|A&jQOKY?v81xPJ9Mk?xhahs|{OC#){XsNLu==^|nnkI>;&F8lrN}%sTl&`l5;Qz_blKgoZit$CN1}COX2lx7&2jIl}G~>udIHwRRc7DoV(3a|@(u@Kpk)!Nc0D`fwQY9;j7cl`+IekSpJ~esq{XiDo!*AXb%!Gwd@z4@$TTnZrpL8Y?NE;3#7sg-AWZR`3$b6wwLwt0h9%Y2H?*=%BUNs&Nh3sdk(9SQY^VL#oZj34<Y>5BmI>YsoUp8*^wM=?>Oq&dTwc$UMb4UJIE{-d^E-S(4O{W=aVm3snj8k~3CMt{w!p3Zclzu#z>z>`)g^-x#a}rkQ#LxVA9#0c>)<1!YMlnh>;rAbFl8K0sy0->d-Cq7tNuJ_^_n)KE&E$?Ta~!G3GlfjXI}z0%6fV(wP&fjZc7?r(Xn1!n;jL9B&#Jf+22{_Y~cNvX=YS$RQ^vvxi#V33&z&T<l?vzKDW-Kt2yyPeHA_^MKzNQmZR9ASvfc6~})!dJ70dcg!zugAE~<|e#Dhs?%jWr&02F7e~)0ZJ4$Dc?xpOdMc$Lkha-{MYWy8#h)1P=v$Sxs1|X=VRkM>73R>os!Qa`-phJseBYuBoVwioXB$Jl9*ugEU%3re-O6ZP>fDb+IgjBkGx@%K#IdD65<?`K)#=P!ksnE9wqy&^cH1bO`kkiqzH0S4;a^G;+9mVfw8Z98cVQJ4>^yyFQ@M-RcR)`US$vZ1i_r;0tBjQz!6La3k#_lNx@I6l!5X9>d31LTl6>%!Jx&QiVqrrnX^o3L6x)RQ4IXNj@9t*m5zaPw@TsYsolSun>AOl59Eh6GQ``p1G9v-5wft)EWGWSxiH4s8^ws0FI4TlE#Z3a>~*M03Lt2eT3C)9Z<sY3wM$NR*t0XnlVQC{%c3Z<t5OXk-JKcDy?Nu-TjG0yJ`npnbeygbrkbIr?7YiBn_)=Z!72-BF>Yh=2UppA=W$bdJDFH-0;KgbT{=U`)3vBHgeMna;rZ^qOB+5UeUHCzUWzJSHy9ubUBCvMa&s$Q$iONdtj5sRujC>gcW#IG))Cf8B2~snIW|olwp{11uPTcMb@(jj^Gu~#bXQ3AEJ_Yt=X+WWRoN4c3tk}+8c2P_)1$1l5jluHy6ZB%RxbWgN!b-Ty+*QNsiGP<n7bUwuL=c3d0QL@p)PlW+Yfs-K1vZlEGi)%<FIFEV>C06jj;<uS_e}+sSAE|V58&7I6ev}i@*G&O)G1;G$X?7a0*zHnSACfa%yU$q2-n0S+}=}uG1)(QIGwD^M({E--PGEUMv}ztcpgmPM6xI=;pO0a=JkYtEM21Dz<6l!pYlig&B4mVFsQXak=hB&J2mGv&LEAfv#PklYnu6uAE0N0)5KGW6*&tw_R=pEve-Q&?^uJgXIK}0ifure|3?yXHk!^m~|-IT$PNG0+5EfQ>usRO7GYru&<#Wkb?yYi%2Mzi-mNZ&@wDyW@803fL1RVVS;X5>{HeG%u&i32MZQ&N;pQzF)Y$Row6Zt5IYYlfOPcn@v&#M48+s!ps@;!coYI)sv&K@1x1;YySlZ1WAg_GZ#?x-w?|3z<q6r<bS(lmRREIsB<Z7`Nwm~OJA~SaAX^S{fp;39ynCNKd35{ElMf&DAg;k&D>k)WoJx_u6{K1tKILr;y3uuVA*F%bCPt+|w_8Cj6|-r?YX(j?qMEPIOpN-9_*avC^S+I?1g5c1EIw{Ok+qKL4x&}+KdZXv`fpz)^R&2JkEj}wo+s;h5V0?j`)3mB?kPL*9CBAC<0@|g5%-?OM(+C__eg5nkgt~2WwXDWMEb@Gw2t(fAb$(p{|)I$_LZZ;%99@VsLyLah3_!Q=?vmJo&~&TQ&wC5es$LLuU2nm9T706Gwt$fAj^5F9;7N)RRO9~RxR+YgcT9PCrY`7+CVhu$MlWTSPHT;w5PHHN83NWMxv{4%3aB?2AaWD#;IlF8iTL2ajT7#g7qC5DO((y6IDunT~&TXX;R-pd-Xr(H|PSwQgO$cmViybp~Nwzx148%<s-7}8^P*G0TLpzryZJ$q>C?lKpCKXhgHa-^Oq0rzPH(_Jpi<xHSRS;$Y)M`6OGd3P$8-1!E`x8%Zg7&B<;|5TuJHn%Ic9T387A32d3<)9(8Tkhy4ER^5xIt+1Ang;y=O5>H^Z&pa1a>moI-k7I!~<uq{sH!aC}TXBS_vOEkND`T4lh4O+34pg^b&$dV7O9G~M$EgT?nCgC$xPixq46R~~>?`8_az$Z&rmk?rW^1Sj-%ypuJ9(6!b_Uoa=wVN#c7APsuO?ArVx>UAQ5wv2e*0}0UmGoGO^aOQP4=+TmZZKHIG#!}=;sFYhaNEkk?IZ<_RP&^gR$eG*r&_AqL$6|1{YW9IZ#uosa8~HyPUhEB0|=jvVW-w4`z?cF!WF3Ml5f?usz1MyzIOh^(AN|ZpOE>41GBZoy?QxddD9jsx405$xR|X3iP-lMi$I__agI_qU78%&wa!W+0woWPzE=_<C<-KbC^JHKNpaYI$tX%fp{BsuP~0&JGCndt_1ttaeU9YL$ve+aqWt8Yf&0$rFP`te^ZLPnj?R?N@$Rp}u7jn_L3Bgo=J#(ffAv#wwAw`W@ESu<#~GDj%Kc=V>GbR>@9TAOzrM1hgW`3iA!me0r9$1JN}BFJiJ6RLR_BZOf)fiG*VXi6Vx<t-`&n~{%B4N0Z^daO=}S1x;6u_kT=nJ0oo2DA-MvYcq{^bLa^L#QPE~eUGzt`pHWR9%b1rah)nQ)4;H)-)KT=k2eDNPh#L>NE%;Vk57hfXw{U8Xp#Zh?q@~5DF7Stw>_977`9@j8{y0mmj)wim8YTkbkj-k1eX>p%!K*zGUhe|6rN!jy7o@}aJqHl%zy7Su8eeh5?eZIpDd(U?eFWb}iFtnVs2+_^T$#fc$`;_^MAUi5k#3+Izr!#oD6y14kMRcc$O;=900I8>38$D>osjHD7qRu4XDx%wX)E-k+w$l$2g(o2=U8k>>yHn<{F`F5V&~udy|EvM)h<&*LNlrO_qj-X|@>T|LNM3@<6__U`?@-swEHEo79Oy5OFaZhTBq%GTS11ZA^N#w~`Zm6AqVFhz0Bv}w9c8VKx#XMsd{$&biBVXet8@Nsvp#6t7iv;$T?~y<hO2PV+OChsHnn*ZOHUPp*b_I_z@twXs=$Re)ZPdSO`~I1CCpH~hZ-iGhF*<yxl*DBX;=C4_3<nfNUz4$b*JIRI_O&ELaP?GMrF}8*11N>Q5WvM-q&63s@kMgc$fRHiCG6(uO@S&<F_QThD9Yo&|NuVbXQ>Q?s>f>PVwrH<hV?)K-JOhdU%p7od<CZrxVLkr|aP;<D)n^uE+aXvZTybcU==-K2G>DnRB}=k_9`I;D9Z7n)VA`wpr~{s+-=wg32T-s3o-B_BskYoq35G3sawHTg4kXxnU{+R6tQ1S6v0*9pi;F%At0-fM-%J(%jXZC{U7@8S0<%aTxA0Zw~qDe_ZU|8q}}$sA`0|Fe(?p)D~F?%^gPzan^;BZ->EUt#j;})_wS^qRYyMakHfa{o2_x;6ZWiuD+z3RT=!M7DH8rCR|>iUSZ=s;t%(&TZmDx%i<bvLu7bkRYIgkQsLWBohWsmf$^}^yM3y_wjO|jI8DyRbqns?b>9L6U+}T>>Iwu^dR-MB=t}INsFUBL<&1iF)iH<+N+YYRu3PH&pcn>?QFP}cF_rkYtcP^Gq5`p{XWiA(T5-qa;*gY@?e}Pix@tMATR<t6a>qHhS)$rQ>x`=i`kgQ=Z$Ur9rt0qHzO(ZhJS__Q@{4KqcKBUObiZ$R<5{?|w?mcBf@4Rr*xg_lXyt9%yG_yt$2&{Qd}m9wwH1b=Pc0M%J)0tqHM;qBdP1*P!UQ+Va2sNel7i8#x6+es+KJ7%3}<t6^WHw!>8Z0rh>y%hH)&uJ;ng0mkH_Qf6lIxM+34mi4Ao%hJM<6|?O}GNue6Tvj;qNw{n-Rgt~lSsE2+vwoR4l!GlBoNXF<wSILDTr9nV3%O}^PT!tc%FEJ(L0ia&)i=3gV+xjV+iQ@O2{^p?G(bC10BGoVPa(d$2W`_1zmb@=We&iv-y!+Ci5;(to}+*w?F5h8GM`SKs?dhVSq^xaAK9$dcoZ7JfJ`k03o|3+8*pnBfA<%v6&FMolHgxSTH7>}53F%Kw?s}K~IFMmshU_>#8t-pPY3ou`Qe(?fl;X}eVPcC2l*O=aJBh($fZ82tW>HPA=uN6GC*Pgjja`CHDOfLSLPt)}Gvl5p@E?@k9EDm{IF~)~@#|0ieyL|aCq0T8mz3Irag3QL}>(9}7ETYR7ze_#w^*>$w3dxrFAO8@TUPo9&0TlRw2bVAYbwTs{3h9;^MXrAt(rUhp=VBIKeCh3MrF+i7Ql(TSb(Xe5y)o56)-M<H%sa;Fq~EN%aXj7F(QLW5qpj9nl>rj}`*;5=&M@4~mS1{6Y@i2VA5|}t1U;wu?jR&BIN-eN5Cc2Fpj>ds<(qqCzj5#O^5xGEo-RQAm^W^y$%H_~&lW<*um@n5H-RRT3XvTS;hEJc`4hRQ108CbGz^n9xa~>YVGcleT;ZYn;nhsqNSDLk%doewX8Nm1x?$<3DsZvGd8oa%vA2wq%2PEoXMNEZp!yZzV!~?hb@8KG{1W|sJjLz$l^D~FO`+fGb2JAtr)DpZ0JOsa=UZo1+~C~XXvn;?DJTI$`4kQuRaqBH`XB%B_9hCsTD}+Izo&^lL+9VK#C(O?;gYuQ7pY*irL>kf)10<{)q(v0^Ih4OxSI6?1<drWw-!ZHd)IV1gnUA@F%o}&_ypRdrc)oH=IQ?6yaT?`lI-DCB(KDIaNemjx3z50#40tRlm~FozBV@J&c&BwsGE!5r1pNq7hjx;6M7V5<nQo$jSR*h4k(OE8;Jp5qDql4<3=W{%p9@=Rrd1d6i?LgMlj89@Cj#ZPFwflt5dy$mX-wZ{v98z5L4&@ux{=-%M*Q0n*NvKxp+*7?3F*K-p|G6=BD7kBeWbCRyV=9DSDj2#XLu!GY*;Hn`2%BFjTQLtF%`IoILg^+{#z1;q8-vup;?%$pinCSNT(><o&zb;?tJcn&<TQAIFt~dsbN+X!#eU)={NXZlfF4`S-D{>#1+jfU6KI!X31<#_f;p-hbjhy7%zIM^C)Pq)%%sCM%pLylrlc(+gij(Qmqg$Lw?IhVQel@2k_XZ(^x%9uV)IV2&Osl6V@<0LblAT6`_%)mQ!m^VwG7',
 'checkpoint_snapshot.py': 'c-qxj>vP<;5&y2g0#!edlqjBLx0&P;)}zL9?P<pG#BMtiiRIu(+#Mpm3Xpn^{`c<UL4rJ<Wv8EdA0!f3U>A$s-#&PFc=&+{Ugta`>#itBwq@CMQ}e2o-;<_W6<lt~ukSt(TIGbdQWMCoB)lSZ#mL)kv*Fc-yrCH*SzVUAjSdeF4|v(sq9t1@w*_C>*WYAa+2@kBTl-wggSDtj(!dyy6Epn5ArZ_zwY=<~A9%Co1v@x+eeve}A3uHiNMiEnv5Kw=mU~MOtB@=tUe^>gbq54%>XhS%cV1<jRqemv7;@phq`$m4f0uszbpE%CmuW7&@PNS2V>e9Pr=2JytUf!1UuXv<%jzn}@$XJgqn>X{+m@A0ixV$lbWM5D36=`hjuxH0fJK@~TVJy(Wz`Kz`1Vcu>F2*+y!T)&71@^Gur%iarryIHJRKYy<ZMlDXu)&ZvJ|#k^Np|n1xvM|enIUV9a&w4<mgC`KPj4ZEyy*y$E)>w9S+JfJK>M7NqgTg!{QPaB>~}*RCP;MbzLlppj<NYPg->BLWo-UYwyS2w#Xx_cp_ydft;xokr&L!Flc&4mtQCk@`FvCDe?n3o0afgrF7_cftK_$!r9B)d#ht9l|YhLvZYlvk~bti@C(;)#x@90Vx&bUiMU+zino*FP0tjQ(}5NKe#Zfggn~tg2EaVq`og<BKlzoOJe;0<mqsVa@nZ50XE%cMUX?hf-Vwt!1hb$wh5EK)VgXi6C2OCTDoyY&3X&W-Vhqj$g3JhD)u8AI&~Z^$CAdAQXvrM5c0<?c5^GrKd&@l}F_EqCXXo_hU?iQW%xdGna!XT0Gaq5lXz-S|TZMyqFsd6?`JPyL0ihC^Fj_9{ZU${5+yt0ym~4Gb0_F+3zxeYvUJz~BotX<PW4dN3Q@UYyS*Z151Zde^YmNhRK?FBe0M`482DZHvET>t^@&rUeKh!z}l<n8J`@kj2ZUxtOwc4)J!IF~4Z^Ngnl{eloRLkZ!@L%!ZuZ#*NuH^C6M>IoEzmqTBVcWKie6?IGF8_TcUnR%E)hm2bCJhpvz1p*pQ8x};E&;Sp6HsJUP$`KGr-O}aHKLl+6x}#YeaVV7SgWA71rs5uyVlyS<2OhY^)%{D5$bUZ=RJY45g-V{eSu)SGawL9pi$xNn%&8Qcmp=125}W>gt6P3*<%nQQq)N2Pgp3+n2bg2j>}g1fi{-Mr}5tPN-}Z7a@gK4z~&(nX?@=cx&WxjYJvC~ZHjv3dq-NpAQ*UA_p_la(VR{vnt*$2Q%N<i5$_o~Og?}N|3zgDFi=-K^XGOnaSDhPru*Iruhsx9y*Xrc2Wgf%l5_1Pt@s+8)5@|(w@L?=)*Rgxf}z#I9+u?Nvx8nT(9yCf^lik$u6k7MS$J;n>hZ~07yKc1Sm<$sG|BSw*srM-Xpxdjh3k=X3I%-*D~)}u%D}&>P3(2;`sCXY4vxYfyKB^W<41AnVN6=Wf5;`k3^0xh6~*z8O0^OUtYU$h!Y7>7yW+`l1WZ@(#t(ugS0`wimJ{g0D*4Nft}I~o?2;k4(kx>Q9sz=(AB3m<G)k#l7tip#PfYTHc#mPJ`y7$N0gdvMK0qi;JM4L*4Txpt;p{@aD-};TKAQ%Br?wQbB9a5-@C+bg(N&fWdG}hnx-o)z2)`c6B~*1{gP*oOluHpLk+K*D7w1O4UR)%y7W=hgtAMiCe(gO*G|ziAjb`or9$R7l(*bn1_X=@4xW_p5EY0AFCi*-M9=)_rp$8=xSe6<@(?tG3=z-&)&kTeo&ScMFX`h}3kXm)T$fp?P<TC&<D)Ni~cbxC+;PE(B(y4;6;i!QCUfF+h45RWI{`#6g#-Bt7<(vv7X??9;1J^&@m4eB-xB*Lmj2g1^g0?u**1ZP}6u~p8v}*-e9{2fc*Gir<(ry_MT}6N`=(`s8u3;FNKy|Jn7lS89n1&vE=@5z|D0H12FtXHwiu+-Y2;nl7-FnUMd=KfPwshC%aMOns9ejrAM${WPzElPDHLp}@tT4h619L2EVNNk^$c72M`(<x;NgSVsBaLF^i8N|81Bb`L4pxq!scRv$VzJ&;DXPTITNRDB^km(MjHNanS+8C?foosr#Qb;^yk-}TB4hA0o2o^(0!p=Mq;`@jt7y$Rw6D(Piu<wksX%Bt9nr76d4p2*CJ=hJUWmHKU3gu^J>|O(DU|v@oqxJ`9nzwxZ(;2>f6_jJQS)IJ?%cp4p~1p($2m+;@q!tgH15{udNq~zCB)2YKX@h(qL1|8Pl;JJMbSwCZJDUsQH8J?(YCLPb7lLX2f;17_7sVirp<ju;UdyhMGdtEV5F7vr!ifVmRB8{C3w(Rt-@wVCceAyjunGBm<b{()25c-s6p@*H@)xDKrIhZ3&#9_`N_CEeHOX;hMs9(Om*LL?<+q8I0G4UC&0WWMmwzQDrE`~BBAlI*+psafO-wE165P&TXzV1Y+T*L(PE2o7#RV;(YyHr8>qC~#=X;8A;bB*waZlMyLQ}DE*?iW`LuxhL!1}`@IH^92V;XV>Y%|ij+-T+*255ab@63t#W9iNf4ba`p@SNkK8DI9M)B5ZhlHLNO#4x4+FHsl0LoC?7+ZN?6We=K(2#cmODU&+^gRI=qlGdhFg$t#R)i$38L8IpRyT^*U_t4&>s!2U*XIon$&adY{By$ucuQTK=R@s98xH4Fbh_yf>-NMb8Meh1W`nfaF#jwZsbJbWOpebcAhv_L`1>R2MS`;li0KRyqLwfKlAh^`Wwetz=)Z-dl^qtj65mU5y9IMJmkH;O4t6mHkB))7;gODosUGu9cPt(qrSGwY>IwUvv1dAF-n_Zj{UPGZT}>+hgVscf`^Cz@h8zv&fR!Sth4QU>^;?Xs9qVHId<=fiG5aFV({XYykD1?8T2<xcUD-%q9WP{+#7-@h8RxOeVuGPeNalIzJ4s6bC%R$rxSu_id`TX)8V~z){4jZ1*g>0w!<ZLVU75C9xb|GeXS-p~fCTt7SQ~Z}QHap1!2m+p4@|-cG~FPNCl~4;=zTV180(GW&pV0%hMgT$2SgT(HGu4u|KjM#tO;M922Piwr1vYP+A9XijzUtg+df;&Au~i5&X~D~oqqXzFQ;#MVw~I4Iy#QWta`GlzWrPiC-2y_6?8If75yci2sV7HNObr9_GHpFnqOi+6eMn7z^&9xVEb$4f`A9syhBUP)ZRnG&XkWB)!pK8ClJCeV^J-tO{JaxypimL6JK2>QPD#EbB`0uUz?UPUD52iYpm*>bEz0Jb1dBc>vdoR)pDW&#_ad2<cBZP$Y(5b(ij9T`?LOW$hS(-_4u^OFfz?>w_RyKeP|9O;a=n4>F(OR3~-aU7ZH1k9n^#)l|wM%(tbPC^yVnzs4?y5%v$*_a?Qr+vCL7|VEfyjh^1dqECc0`ENbvGsg~8a&arl}cnYhSj>FjR6vtR)sJ^?zlTR>Tk<JA(SGTVg9lGz~@IGGO2ncAt6EPnVr=av-LN%|yZHyWMZpUfH%!byJne1BvyHVB~Sqn6)9CHaLe!U#zzno0X?W{EGF49<`hl8iNgaAIpc5*wZRh*DK>~-)}2C{1yW2gH6vkRxWp%ohL>ig0RCI6$YF(*GK2B|SLbHH!|#3Zkou6+T4GAeTYVXAL!MF;-{2G21<',
 'cnkang_network_snapshot.py': 'c-p;K-*4MC5PtVxL8K3oIz>ydq8qY#K(~5Wu_DQmY<chufwpLyjYMfAmBa}AzweHuWc{#Gr`@IpTcXI1$K!Y3op^L~bX68i-f_hUCAU0H={#jbFnyoPTT;k;!Ib)*ygR!hR3t=c$!NwBGS3sv6j5s`aU}dDB;^^`(b3V-XeslI#PPD!rDSnTcvj?61A@pk)jSt!G;(cos@R*?)ze3n3p=`?dY$rlHS_`=t4@EHtYp=LD(BA1x@A+}>NS{~aIvZzc*g3fB?Ba-!Isg@`Q>ku%lP8$^{-@XCiB63o`Uf~bkFrV7O-sJTZmgKR<Royqm!77m7s-M=Q=7jUNGWI0?Q-~$cQ~~n8y!JM+B}fn#|5I8ZA<)6gjgS^$X=Gc-pU46kXbz$-{yb=x;Z3!j`ZtTyP!7zGCSzgeDDxmFs{PwnwfR>~nX!*GL#Yh-w_yVd^m{bLq{{I-Dp0*QZ=Cm}rrUq?$r9AcY`4Zbg*k;Hq5kg&&Zwi0c@dZHn-x*<9gDmQk2mtb9YbHf+X)ZJwUY$T8_o5?Yc@U)3+|EIp6j7-tW7t?MjDKya3MM9wi;J!SBQt{A>O%LTBnf>fGH-Ngg)v>>@MWWtd)02?o(B2QEB5cI+KJCL0}+IB9|T)|=ojjux~y2uMA{1)I8;tGh&7Ntgw2#I%GdLb%;-yn1_!zKv1aTD$R#UyNjL6^MLy*}6uENZsDoBVn9$A=H?;R|pv(t4TFl^U;rfEvu$jRNC-zMQ<hp2TP8@y+|Qcjq@?>PgU9dSh#hrbNHvZ_`0W?n2V&*xCTQjIfP1s)t*)(!>wid!yy1-GJ9JmjuIwaN*|d_ra^0CMrtpqh3a7P7~$h*v10!VkZ1atpbE9Z0qB;Fdsd3$AtDyiC6G~#Wt+LrS|3&mQ+X|aDr`nx+NDhO*e<bvwZ1oJ9|<GPp#e2wYl>lTwtficpnl}$gxw%W2=nEXf4>8PzDh{zLq81iyhg(3wDuc<DwIZHI>@oTsJomzd#Kdala?z+pgU8c{1`F#z&e-P(-Smq<)fx&B6DSYa`Pbs;~M0j$Vymy|$ev4+WPjc^VyyjXhQ!Y5<0d4HFC)zY&pKgrpsyLEjS8SvPvf-JkT+P9b{Ay2nlDR}+k%VW<Mr4|ze}g1k|+h9b-Inq+iCXo@`AK%sTVEMF7CHH0(dgnS9X!U)noY7)vZZ*YMw96sidGY7m}qeNWAH05{9-{VY4v3|TP1)`W$R>jM%k_6)_gA$$?i<Hg_uMvh2-L5DpWf~Gl`b)lY89O8|A><a2md2i|a=d05)(})7nX?s;VNATLFvCdq!d@A$3vw`}nU^b5m}PKxv98GIR9%l&8RJ_n5|A}xC6p~*Tvb4|c*e#~>|^X5q(bAWRd?w<8su$VSQ%!q1elPN5`>aG8rZFO7;sfrg0lhMxV-WkxCNTv_b{Z&ptv#!2BZAe%U;k@F$JyD**4gDx))izU)_-(#^hv(a`iT~iwhLmO5|G+W_G(>Lh<B3uwapA8CGj!^5`|)YbgznkGE`d>KHzS`<c-*xB)R6a=GuhAVoT5^dX*aG&mI=v@BsCnYhCT@U)D929RQd?Y8fjmQqIdl0$N^bwpBTh4P*4Sa^GX2``A4SmX(Oj+A<N{GAsBpMqSbrCN6!a=(Vpo4#j%+~?Rqy>+>>n@S_pjY_}3FaAjsn(u(_bRmge1;Jn*vnAjpEA+tpuw9yqlJ#D`AeZbTTi_R^DpOEpDkZ<eh@?x%z?LMN5cYPFmdURD{0fq%f=WC28b@Xg`F=hh@tT9e-tNP2zM8t)YevQ(l1_|9`}3OhbUn~^p7ouV4#7L&+6!-kr>mncUpk=hFm>7n&oIaC&5%7z6>utnD{g#4+qQwUUJXH|(WQzpNWxt!Sln?Rbn<55t_fE*dm4Ie5~hmEBEcH29_rfhTl>lJF;W&>0G8NumT3Owb;3+M*I;mzvq8h|Aiv`@oyz(1**MI|mXTG7fMVIpKR}}t4p&$byHfz~^pKQfXUI#D^v@ZNZJ7{~Z-VWBWhTp42JYdQV$Y#<yFBpRdaJ{u_UF}SR0!Qd!ELOOMXyG8MpZQ>oJeWN)4{W5IH+RSpYspNoKA<^?mO+Ln|;N$!5r-`x%YndK8^H0?_smZ)0={suWLAA!>`;p-a@?q4O?$|`t8Zd>>$8@tGMfJ=A(xO!d>4^EK4l+%-Zc4!QnRVpIqbs20)Ry|4f{FC{CS+tHUCI3>z;(^AQkifvd!vKg0Vjoqhrdt<O@P#mj#Hk{qHe',
 'cnkang_request_worker_snapshot.py': 'c-oa$OK;;g5Wf3Y5IIy**43uiBn1q(z;>}kd)XG-w3pNn7>TyoNR&!avE3m5z2A`3!>?>t2U{A=4CnF9Ll+kpznHq@Ruog#2yLXXLRt1%{L>0!4Ku<RS*w)&s>&U+Yr#wlz9(sF%{E$EVOXyDresT53X@)3Tuhd_u2_~WTia@pWlUC0tu5n9)s|amGm}Z6J#Zs#ue+zeO|80nQ|FJu_D{P(MhDxv=T?`ce0Z1UwNfG{{gcV05KETx7Usudn<ujg!_^2?NVS^6E1{-PWZKfQIlBkTf~Yo5rtE&<m^W)U)QNa`y5MmIkw8zIx{0rn;q8*aNhXa{#&VU5nE0pcNBH@1n~R3bOBiA&3->BvQaN*m6X=33R;pI?L}E#K0ihr_oax<2c|$hE&yn3VBAPM!^Q>u>DPAEHQx;XoCtuNfW3_*MKl}dbYBAkAe{@AcYgI#FC6K)!H0{V!_Ni7PIWN7D3Y@h*R?xo*I$j2sy6B3nC7<s1CCQl}bM9ESM2Lxx!W&0f=CF`uIEx+-U8wetT5!P5y$H9-;;bcHubgW7hi0-T#l4}#kCG13Va~luEl?9Gq^yH!$x&m=9RU>}v`A0DYrf_<WIhJ6qmyF>^k+K%-Y0zF>~iRYjFWtsbdYrX->~Os#bp=s=o6@)1**R#13x<Q%K*bisS2DC70P5@#=R2dp282l=<Z}C>eiCleacz2p;|K=Bu^>-EO;@}JbFB<7AY)V)%l_C67XGgil`#F+QoZ#^3nA;UpOG$Tu<3|SMKNP=Gxb>W9qTIVDq_udz_EtY!qM(zz1_iCXe9cfYG(Rbnw6#|GL)0%pGd|lb5YHrw1lA=++{$b)A&}wdgb%af;g+I6gAKy^y($9XTWBIsHtTCpf&%-ESbxm(jCpUWE6)cfHKmGYBuyxpw8%`BIttc!vTrm{(|ps%b5b147V4L?vzTIE*;B(1-YUPl|oHIpfH~O|2h=Gg%{3?j7lG)~NKD%pAo~)vBy>UV7=Fj|W}E-4n?eG;Z;P(ThGAIVM89znM91v_~gUYq^q&ml+BZiu{~RsY4cpmTFn=b9Ffc>+zsWf4;l>ZNL+0>G%itH1!As@RAzclwDpvZs>00s}BHmifHJZT$zo~ICgCCk&;6l!M12T-W-O!y1}ZEsqx?tG=IrT*CE#U!M1aJ%u(E4Rqo9>PJzRci<)Ejs=z~boCcD&dN;U1ZKXuJar0+{>B9x0s<^95TB_2r=7ot#EIHgWmU2ZS#`E7to)Rtg*t#vy>Xr&zJvj+t#PsQCXJkUC)3zzFW&-AapdP<{zw%h`w4iEr#vSqeDzd*>yw7R2QoYbHf4jC-ImHrG%G@oPZ?{Bwv#rYLW#8$&aDEYg-L;W+-8+R0#eK+-UKq{7)S271DIIKlc)~1NB>%%LUY7L+&?}^d5oZ1qN2jHofD$~v)2&DtH>?B4?4-sq&L8Sx7oFackPVJBKJ2XETJv3envLL=uUqv<c<T;MG1s(Tqm+$rZ*Jb*CbP5d(MjC`<y4(t+c2R@p<<Fq*bhBAD!$FgP!la?uSQXGi3%EqT8yP3lof`D5QPak@nuplBmWhsC3N32F*<C_es2q%*|r9>J|v!kpcLY@Oh;YonfbBU%Lh6>Kpdb(n(_FmMwjGS_?Ai^Zm)$RD|{5wLb!*Bn_S8$Ij9BU7F5>nsRS?3j-z}Sif)a(ygaqPhEtZ*O3P8Tb@h{%H847ua<YYv2S?BOwY~JHp<^ik%DT0J)-`+)R9F}AIWX(Ti*xsha5MrozCw}a0_B{#Pv6+})}pN%<9V@HC_^8;F4&=9->~Q}H3Ha3%d%|Ey4y0`mJ(h{6;_nVe`JoklK',
 'cnkang_snapshot.py': 'c-p;vdvhGeb?^W96l3TnvqLTxiw7xcMUaL_NP<iP3;@b@7OSb*-C1CW#q4@!79h+;m25?c6j_#&>ucY{a>YqmiBgK=l;c~M678fa`X&wm$bb1O<abU#rsuIB75QG4jh*S~K7IQ1>C@-ct?B9Mm4l$`haG=r82Pau9R%Ty)d}}J{0zMaCDw2dC4SfHMT31S+4Ze>G#vH=lpI{P!a?G<2ZO!Yc<6V6UeNK9U=Y@(rq-iiCkVa1)fsesD~K&G>g)ywXf+DE0NNdN;@QaW`4M#<&x$Ue=MLuP>sI6)qW>K~whmE+`qBN?rZxwo2pE53<n@E($lCQsQ4l9VhkEg?$Ug}DLn{t;(0^RDLjM3A?)zO}*SGGCx;y^9A0}~aYI=HliihR8y-_lXeAl&teHyRjh3Jw86Hkdcok73vceq4Vx-l4qi62eL*Un&gq@H)Zc()(4^=q=<x4f8t-8kv0^2is!M^PWOYfNzk9c3{FoK)%Q-XI7iBK`Y-Wx%akeiHfq)YPjh8=I@^Yu1uw3v$jJT%MtcnOU%>R=&2iv3z4|eIxzs9$a?mv)di)+fy&DZ`@wq+Cn{d<L<2$14?fY?R!b$-%n=d?5St0SN#Z6jj4=z9vaqcuuoGpdv9;puiXR5dw$Z{MTPnL{N<Us`pkU2W^I60!M@Mz6l0!)7@)d7=(it)JbhL)>if@IL4qj@zzD=8hhcx<bz?9<Wcl~~&M0A)Xvaa~Td#Nr-exBXhDpst|CLR5^X29FOP6U3moE8N7N33YIltTSd)>>=_Pn{?b631>w==)!&-s4$a{Eesp}V+vWqz@KX|e5f=R9w|zF2QR+rIqlV$YtMn(F#Ju&}#3h?5HD!f8xd_(!kkp`zxAKh>x-^e|2!Y}-z)KL7()9BvW$V6sp))by=&Kz^!D<uyO-#)lYv#jY8j9f<{=hh8VC9C-baFY%C2Bfl1p+Lg%OjxX9(^j)>MmPAAXh<ClR#*$rY_zb#<8dgA~4#GI`utY^!)oLGM1{}-pgHgE%7}FZB*rTL3bH#Q*0{?zD06!!Z)oa(MDlWIJv<KazDn^2(6DDqQH1wB%pFjsIa>EPvtT>5+P9odI)+i>iv_I-6!LaY6Z4`}$v=q}-ZiufD(2Wu=2$Q24EpY-2NPRC73mTIKV)cTs>-GB}SZ9}(U9uftSi=9C)7y6IdZoF2IMcf5*jY@g)N<7+g1q?oU=j_Qjk(JUR~B0}G0KYV#hoCasf+w!-|P6cQ|u0lYAMrmv))oXMiZ7niYIAp5c&)N!qTI$Kd^V&^9ytHcEiHc9Js$uo$j>vawVA1D^w&V76tcba+QuVn7JVdiaxsFu4+4eQc>Tkx|SGCXLk@l?xSuWYrg{7oYY9V?fFM>rRiHeTJKg6T9q91s)0`qd;GM6ol7<o!f!dUCok*<T@RdBmHb2`Y__V_1A#twMLbNrgB~Snf+6W&5*?*4TmVrB7&M7$@)Q8zAb8hybQAx6#~&tE<!%^I-IXXBL{;k+w$0IyUmV&RP%GO@ig?W%4j~sR4{X&9CZW)yi{%(}@#RG<2ym+H6KuTWa|O^HVz_$i*va++?he9{uM3~CZu?%0&Coi)>ThGY(Qu!)mN###u3GzE1gx<XA;Nf5KJ@m7LM`BxSkrJ+>!nmOU2VR0t##p=luiUT#U&2DNp<(V`<1z>)%U}S1mL7~I`{TaB_8cp?CsIQV!du#SC^~~{XxNVy<X4e+3e6Xs^-=#!`48CBQFKw*K<I6SDp%pT%n{#0GO-=NsT>@zyg!5U1+pFxEHj&JsKvBFC#B1*BhU{Q5%2yZT$c5e)H{r8h`rJ|9Ck5^zVQ3`u`k%`oVwvZ2a5r|IO(SetP=O|8*YBSC2`!7swMsE)wuX>jIN%##*T3kp?_t&DH7I#Cyh?M~Om8L!q^!Rs(HPVzo6SbjZ1dT76RG+*&mn&DAcYjS_#K<<L`c^HxOLj?Iwp8iB^$3cppY%T6`J%=FL3&|Q{*x;9F<u9mDc{<Kb7Rss#w(>3u77>Z(UGzw#ss`ulm$t#zJoKBP!Lrpd47N|k1fh+L1i9=cKu2ecHH>Plj@SSFsR4oh;;&Z9*?YFz0)oE}ICR>X(5@^A(zeMB|m;eN=U~b$nmjB9qXfqwsJ9~X^XNOdO@MzoX>@g-#2!=urwgzGU=y~WDQq||}9Ubjs>t&JPOWUGmtQ%H0Ov#ywb7R|kkeLWO_DS^yUl|-D+RYfVH>2<*V44+;CVDf<LRU51l)k8J?i*<pty@UW6qDZeSruebVngOKU6Xf;LBB$knk5|AA`MM^%(9m8A?3oBb<L{P>+^aJSyWQT4!+F6aOui6a_X_p1zvNlwCt)?o4?dBm$BkW(N+_!)v~^{q(7#Bj0)LZ%N#Zl$kZ|?_F+zV>15|yJoN<sw<lLWA4?q;DzXdcaa;*82$+>n^0P`btNo<1PzBb0Y;J&^KpF$WPSDCceRQdETp<gt3?`M=?hidO2xez5G-<mAQgH<jFt#hx)^t?}TkJG2_P|<|FEe1Lif{DCwk`_PF5+?8X{E+Q-}gf9pD~sl(<A1Nj9DyEe@GAFyVjOq$@jZ7QXZ(pt~NUxvn@8Jr{t*b(?d8&3xi$+G`Q3|P#V8b1_7_ujRr%P#w)F*XRLL|7wAPWcv!zb2nM4VCGZ`LX#GGkk&D3UflvXN@3kU^tX(pi5Wx{B-GMmvVEtDdC&T|0fdtQ5&`qaI>DIIrl6@;IHO&j$3OwXRFkt(>3!@_%m}8URHFTKxK%JEJn8h!t=Z(U)Q%(_9Ci@y%;)xx4X!EMa_+h4ZcA-tY2AkvTM-NZl{MXYT{Mp$@KWYB%SAX4Tku4({Vy|(fS(Bl|?-N<IC_sD3Z}jL=-zK}kKzpG-z}^!3Lo!?jQLJn^-@3Ve8{~$+2ws7(AA~;U6V@T?-!My$;PC0*1IverHuMzx=6E~R^!q~lQQS*xQOUA`tfJ}Lh7z3_bXp-GYscOWX^mAf#nS4|Oq&Uq<OF7p1X*xEKe5<U?Ub45vCjr!64Bq?IsQA(e;4@gA{#wKzi*7-ipw*e?ob#k1w^Shhv@Hhd`XwF-<^n;>L59{1BpJ*SC)5cayy069c?r1@YgK8@3DDk*%_v!S~YU1RkUr=w6>GhMNzj(Z018;L%kY;`-yLu`$(@r|3Z#aNi3{xdQ`QQbR|v|d$!5$5&<$lXyaD?9525nMW<etO>GbE6<8I9k)#0R)|TU>t`{MI2T=(cYNLt<gG4M{vFahu#i|i7Ak{)Ya>?0P0z~;VYv{=viuGo&R@GwGTw<%#p-o7VUULKis3ilUU$~<rs@jy7?2#~=ZKt^RC<GxdFhJi;J(Y4p6}~1Y-{6PDXHRM>cb4H#K4K-7wbcXV)>Kd;I~EdX=nIcZ)gsT_U?d$=Wb}AD*p0RDfrGhEK8>cJIU_6|gZy4D%9{|Yl`qFphxcVGCy0qVee#@>Td$pt4&r^g0?G;)<~kO-XE@q#`@&b$3>k}1BOnA`&bb%Tnyr$jz$u7#xwrI2{l3~??zQ)}yR=xs!iL$BtkL%mJT{I^1X%C?|AN2}EZt)x6CEUEING4J>rn`y<F$owC4V#dOR}EtoDpc>gO^zl+7<56Y!mHmJ~pi?!3l&py(w}u@`ezjS!Hijvo$5XX+4W2I*h*vo|r-C6q-G_a6z_Ahc^lp0}ES<TC#OZ566IS8%W<>L$F}=#nqLY?#9Zit1Dl%4Gb^@wPHy5hko284r9%SWe$zap-*!K%oOG}tVq`4vA+CtMKN876Fh4#gmWLuyt_r*OPnDW0MD`X(Hc_@o`;j<>X&9_@Z?-K7DI(TfgK!L7WC)&>!R|^%r&lS7wvk(%Z9;_6E(jX%83-%u-%(f_@%Z3ufni0=di&A2U4BwGz@?nu$?Yp5~~w?lXQwJ&SYxc4U5tTnlC9rF$*4cEm<s1AdV!8qHw7UHU2K*Gc1r~2#I18SY;45LN-O6NrGvXgpyiKVy?(*s`V%Vn_A0W&heK*u}cUG9Vrn_vMXKrJt|zg79)Tp%_(~~Nmx^&-`IA_CzIBiY&#(aBY+uE6@WZEJj}*fCc+k=iAbyz=3-v9UhAUXH%5MRlq-YzvqSpL<UZ26BPhT;glzE;Hpd|}ViO~Z1JcMyIj>q*>Q!rTalt9LpI@b54ZE8grF7^0*;Ipbjv>tl%BN*cV&X-R!!*ZvgZ0ZncWIi4G~LQBEWuM!q{#M95-PWYm?J@YvB=F;G9!RNSpexR>QEwOn-~&;qATU55hji<H?1CpZN*&MvRZCs^Ek-V6rcHBsb;c8*4<72Xje2g2Sp3@&cvVswVONV2oRe5{UL(ka-1ZQ=v!<-U5Vj_6Fz$BS~10K`^e?_H=-eILj_^UwgrE+e_u+R)XPD;Z*!s1a!mFSRcX`b4p@Rx$+c9LDF}HpQ@m$cFF)X$Bi_kLsly0o2E7@~#}4xV)N&dlbOce|Sk;V6lM!i2i$aWAE=`V)@#jWS>y_?Le5`5`F+k#meQ?3Cl*;>U*aJjUrRr+%s%s5N2vGL)>Pn9Cf80P2FNsD7gtq-2Inwt>l#_r3z>_JMo)%mK^1y=IH`p1$MZ!v}Fpoj#!H7cf!ZnhS%-I+QE8z+SmqKw;AGfc<K(%NUtE>UJ_*`Dq*T{0yLxy&Lbr$unNt&zB7iTEc?r=hhjDMx_=p44b!Fd=c?r2h^`y=LkI8y7I^`Y8mNKU;9D@9ATGz+@6+&`E2uTq;`xp6JABq~u6#8PremMs}(lw9J7CJy#7`SB7jd}ag`vbzGrp$~Fm<)>l?ifpbhp;)<Gq6oeE;3%tA&y+Hml~BQ(A;z@fD?V4vmd;Zw?`N!6Hm%)&Lhi(2LV|EoP7qntSIm5+174PP`q0dgUsBXc9saz;C%A@8<@bT@3HL|CQqz_)TB_?|CZ+0o+)~D=cD*6ikRTt(NYN-L9`vcW5=h3gcsFu9NRVVS5h?!*+DKZF_x!>v>(6>i*Cl#4PfKVsS3Tu6^ip}Kqx<LV^7*P06zodv;&lgFvs-D-%{<rI?p|=5>)SQrO$;|xe~Rky<gVm<PA^jm8Q67yaNtAB?FIKMtcy|bC$GvX8=vZ?OhK;SNM(~`?D`F^DGd8Ao=I|e%3B<%5cVOk5&lHU)Eu<pOf}{wZPW|Kj72ysmbn8`6IMKj(sDj5EIJ`9v>j}>ZF6S(Ufb`p_dRv%<XuArw$0Du^#=a;_~D;Fe)#%t|MvfkA3uCYJUQd-Cd{PmmP3{^cjc6^tAwdN4<(><`y5vmvIzuzmvVl1;mMlH3fiKb;tBO)7yO~k`_wPSpj&D5+v~vtCT56Y>S5}J^k}*EfExQ*hh8Ybgc<i*HaJie4r;0;g~Z6@OWHYM7annoEc#izO9@X6i)^G+N@#-`2LdKFJ*|_*`N?2@Roc3JXZfX-jS|EE3F#n``_mK@Q%;wxNW-t9@&~ybse_ZPaGT@`$M3Sv%Bt#w5&=rq0z1e-O?;I!FtS_D+j{hxmRmGZwJJ<5EpCtn1o`07N@ZW7DPBZmjy6a#49FByk?!Y7PJ;Weh;@jOY6WRvT%@Ox3rb5E?zw7)_cKC9S>MHYA%dW^-@;CZ=4Du3Y50>&Fi5{JCT~s!##nWF`Gp%dS6+PS<<(cda_jcm`kg=8*xb7N>Q}$^^*^4jPJ8VRM!&Ng+}rE#hlAlaqBt2H9Ns_r=CrdkEj>Znlv9|M<Z$sPxr{VTvfe<^T5cr7`(9d_dcX6gmUKzT$hQK`z8E8-nCjrH*sh4kLPdvGQZ_M#vx4^IL@dFuCH%^Xe|F<s-~2i5D3!wko`)`uRWw(VOs=v}L`q9Ox=_936cW1x2^7vH0bM7*Z>u&m4yBv&eAv}Y$h>JU-+p2BrMv5QH{BQ3Z+_iyA}SIp`<f+{_P;apvQXAtS2Pn>&*gA~Y8CC0tQPzY|4*1b(>J6KE#z`^)koDN8(Abq=ju<`H6?R%)f{IR1$+5a7;}r@j<j{P*CWT)7^cbA898h)^`wZ?lUUkdXC#8va|~@2IU0R}pR;>(>^v=bp0Vzz05W--BY#IE?T2YRnX;G?--G6jl;{A!64Dj1R}49*lS(YnxR!xW3d&Q7mb`coQ!m>;)w(p+rf6jbv&<DKw-B0ZlS~;uzT(YNLxNg<9^eU%NAzVTW057Os$-0Uk81ReI89J(Ypk@5+7tjv0)}OL$k?d>R@Bf6SqnlxfX=5awaJlnrzDFZzX*-2)N`eIZYc{+rvyG~tAn*nuiV5Ev-9qmmF3w`xzsE;mNi*TQY(}?Jx9+n7NSgzlm)cl%GV{o@p^p1rl>G=@)N^^xQ4vcx(Ma*u#thbz^|HOsF6uZE8!8<Rgp}^ZZdG6ozHb$>?UQ=@?IcSlu)2lXFlu{CR(~2dQl|a`ScO8Em^e@t`qPM%uwZP7Ck{FteHzyDnP;m+Gcb;#meo1znRng)MqdMIX^gdH&%s0<-?s!@oad>Qo;)Jq|M&Qw6qyOt(=2j_ls;lUqHryD(S3banPP@q>RH{Osi#bV8e!0IcLwZX6U0ZO$%zKQ!a*8$Dd~$S%TeIU%S1$x+chVcWra!&hp0cR+&Ne-ymDItGjDoSzG_=ntN;c>+5&7vK>gZfb}QJLx(FO4GK{1!GJwtBInDjJyk#rvF9=^(Zp?P>L$i0oK7?w*mK!carH~vU2;0jJvW}OJKNm{^T*AZ+1km+zk~zo!CY0mp1%K<dOiK&2iH*_hn!?VgBzhpJUG`8?^Q<JK9olw<4wp0k;#<4@<6j27WOrtT2Q|Gtb)m3XPJc@I_o5penfL96wHBapWX*y(Esj01OmjF2TnrhkzBP4p>~QYDs8u6<cdp=Z%BPs4D(52Ul0yKCR<f2(WI2}bUN9@U+!RHhMyc&%;%1%s*&)yJ4e}^$VhX_nn<NnEI0)Q8mK@}f#pL=wp~z&)08za59ZoUi6BubXa|#M(6)2SI-Md>!Ig-)U}RUdnbIk4nb|5TR0grq={c8PGRqY{(2mbUQ^X?)ulQFTVinE->=cK)bShS+*(-;oIjahhTp(We0%dELsklo}G}_cs??KABVTM0<^UT(I;>_z#)zqh0<mY}E_%dH1s|nI>nxsKDUnephdi}kMrgT+QjF&2u78AidbjsC48X}DfiRH+T*_)_I3V*+mb~)4@M9OQ#aaP+Ykf_|DSX4N~V706^*qmDn8RMjBh374ftnX#WL>sb4Lr|2c?%)8}0<u+7(x$vGIoZY3u*=I;oHvQ}l>uJL`lUGlyK^i%r_DS!X%d$+s{xSx<r1Lgo<?JL4rdVza9>F7P`Gtp#DF=uA{umjk!=vj&?D=4o;;1N2&8gZ<C^ti(5KVjDL@&uO}sr7vpvK{JHXlW0lgj3Zpq_U@p<SdP-J+S&x+_|x~L_v2?=!+kE3CwtIas+W99P-%M<Tf&BkJD>hsHBd9(<DPDar&?Q^YdLF=YmDW`<_BrmxtmpjLN$|G9hWF(zJAEJ{8JDKDsdHhg?GN2}${`iB_Prf%k`|#iKL)l^?Ii51cny+n#;}*0EV0x^N{;ISMm3wEWY66U;^o4|k+&lC~r4iV<5<Wzbw#=SKq5)@*tEiJI;)_6vPO&IG!&Ve?1y$)(8b9P93!1nTgCko}aF`0T>2znZr3CTx0pc8-;RxIoBdx&o5``WXn3{Ng=OYtyDTrg8kla<N8{)J?MT~(dkvR#@8>8_5!*t<zQy&^X)*>B!q3oX{IPGG{a!SiZ6`dADVv52kaN<YC$pCpw2D<xjfC~iVS#ZPCF6|^Trm+;+O=Q?h%nAj6o~~R&auT1v$;tY(U+wZyO|r{VBc)FbIyuA*wKOdhl~m-=<o45_JUsdLH%{OC-q}w-5nO(qQf75a-Va)tcrJq}#B;?ANh0yqO6!rT(tQPKrj$;3qPcWr?wTGwHx!o}WGhw$OoMKskBeE)i>NG9A4f@Uu)ld7hh_hMyj^>+z~^I)VKTHvcAx-o{xC5xa^eU?sJby!&3H@2Zp;?^T**c(R9ljW^Y!y_c*a_`M0A`Sl^&)C*}MswrCb+_v)E8$z%w40zDImZn~7W}bWoE*-e6D>&gNA?Lz9R!8)QK*)T>$a9JM4yObBBh*;*7}C{Mqu=_=}8wdzm7KrXnQThjyJYvykXzEY$!)@qZc1SgqW<du*jnS+D!_-s=HZb-PTOcDauY3ND>hKponFo+U?Kvj?6yohHi$rRb_Ovq9`8Pd}hbh8_lBB0H}$`!Bfpw~kz(v>WaYD`m|>>|8NVpEl3lPz0yS|ZJRiAfz6V!UcP5rwqVb3s7KPOs3OfCgazHcCVoXD9N8yGtSqg*S;MTL3oQrxkfi5T>8&h!bg|nh^dnPgHx==#1l3-o2Sd)))!~&c7752X++wW&CW77ahk|&!m$J#A#O(PtC@qy1wi{E<G(sSWV#dXRs=h!QOm%ePhcNhXI7U#Js7%IPcypJ2%?OgIig<v3_%P?Ik#RZmeu<k>4i+))c`a&oj>6I8wetsnbZ=%_(y(scOv`)AYG+J>b(%hAfT-tW@ip4OAVG_l~2@ITh>Nn+zP1&*Sauv(4q1KlWz6*}8a59k|M7UC4gXM}@G58+TqE1xXLZm|(`uPv7|T+bs>P=(Qy0+s4M-JNXHw`c*5)9cM$YWgza{U3TSB3Yyr-K*Rx_;u(h1x8FH^|C7_d`QYsRcgCl0e{}lZdyhZ)&e@}1p8e!o<Fk+6IeGVAPyX`5&wl=Q<C8bv{p^<?V*~%~5C3^A(s0gx{%0}`hn~)U@oy);{>Rhre)!qD-y5I4|N7|<{zaVlJpJxpe)hNDLD5_4>GenAiBw)iqO##_tdMt-S$5T3TVLH=`CLQfthnoq+1c&Qi{;&9xZ8}Vrz<-kjTxW$Tuaoqk8?AVJr@O%dTq+_>Lg>B1(Z=2kJG!wob9$oov+jRLg@J64ynE_?&_gj97F-m$F5tWDc|22@9m8D`{O;F4H@su-Wnqy&>wfUYunxNaJxoK+;V=8XtG=gIxe=99Y6F<9G}6}Gl$lVSH6PiC>_q}0_ipj5%xn3x+C%!goE^E4`K`>rIW>!(7wzA#~5bwwcq{fEgZ#ZJ*Za~U_v$NwN+_?TfyOvAHIF^qaQu~<ohRY{_OF?H%{LC=<&xtK6&?BxDp8!-*#H#*MvESHrFfc-dzBU|9t&hwcq{fyIezYR7MGBd~(#<B_Sw+X6PJg6jd7-*o}GoK78A5EL3gaVmB74{QGv}65ikZnccWd-`}ts&(ixxcH;`(KmM`Zcn<IHehYBw$NAF}DjlBDkk-_^<D{tRGO-^@+&DQbQ(?|2#Iy*!>6uk~gkk`?`Jt*d1Sicl?E5{+{tb3^*(LQ&C^dR_kq^ymV2AjYs*@RamD}T~Og`LP*}So`#(Ub@-P<o9n=B&)so^}SR`S%a06O22*uY_<bDDjRbN=KldU|&rI;s1r{IspHl@I9@7IubuI(9=X5GM`TMZvwPME%ULMJ382&=T;@3I}8;F!R#J@|~B{@oC^`oFu#V!Tz|6{SBrF-P1$D_J1f1BCi|VS>N1Tec{&E-K`C12WALJ>IO5=4V2l!-WJw@hMfH#=Cf$-Y^>j2-@^TW%eUMcx0Y9Lr@+V({P^@2kH#n8d6RxlfB2J=w}1Zl*YBTw|HJXwU;W$3TYvia<9F~dhU4Q8<R<ltN2h=C7B-N#Pyg3j2z8hlFn4g#AosKKH9W3D$5}Bud;G~SKKtaOvq$fpeD_bsr*Hkm*<b&Odimg2j~_l7pZ)qTPTqURE~dyBnaPr-a<KW^qCj_%d2A}LOfw3%*HsR&G$Lj$*PVtaVH@ruhS}0Ot<X=i2gHrEK%vW0=)@?LI<1qc);X<B>g&LWwxu*nIe+mT;XKPJ){+eJj+Cl#uhNsQ&r`=mZmL7c2z)@c!z6@LS_ew`LLLdTW^Iz8lb(I<OSjb5Zr_r?jEm$@gG^7lb~5ts0UEMgLrdOf1~$YYjOP(`^@qCb<*nPd*a!hk2~W5@)60TP_|gpuly)dV+XK=VA7~JSbk(1)Zvw2P){V>knzxH>FIS+S`n?{V{&cQxUkAe5ar3q7tqa@NuQ6wClPh0SSeyx&lYz>m^r}5R(4ddR=#wx;zmC3Z;t7FS9h;ztP(G#9@Ke$qoUU=tfstXNW;cjAr9mbhOLs6^n9{q9L<7D#BNP{xrZ=|A0FjWUl7uu1Tx*FAUr2|daxQsg5eV{cQYv#Jo7j9@;$qg>o9T&j%J`nB5$n8iQf=wyxBvUi&wl>s<joJE&5q9={lnR#pNS%=y_O=X{48OQvTAa%-@;t!*zc6czV4F2%s;N2JLU3Fbeed}KpwVE{O8PPS0@{bS9OeF!hr?>jE|t?6S!2EUFZ=NVHP@R+@Bm~#t5+c8a-xQiDF2op22V`6K5!6Lwyt2QhLTC9~x3v8k?-RtQMlM7ogfG^3t6!1!#P(DY>cFD(JDq;u9jFKO)2^gF0T4cO9zIjtt?*+a}EYLiN;0N2rSQB_map7|udB@23{+s?1<b>Cu<uPn+c^Y?-8;<dy;NNK5~T{+;Y(3xK9uQ{a}3L$n&Xw2-aUty?SZ%PY&Q#ob<o8j|X88IOqomk&s^oK#&ZO9Y~@Oewg61EzMFt{^O4XCfm>PEKb^JG-^K@zM(PoR#IZ0`A@@Ay=BnHO@yPNi3L2!mK)cku+h^+jML8%W_Z79&<L4G5p&i^rA_3cMEPO_=uKYU0%Jl{KBo3VlvP~Pnl9+s?t?<4zzQ!bW;x+&Jxqr1AGUdM4yGJMG<j!M?SBLAqAtLJz`cD#~bL#sgh_lWpzGdy(}1VmgD4o|2|*EO}km&8<K&;iU`iIszJ`svh=$un@->GEswF{&S;uh(!?^iXg0;Qpfln!&=zUnW91)uu*hM7(gor;sU#!N<Yp@2S2i^2`ba)y>?O)d5sB{N4IrF}b1fDhCx0#9T77Bl_R8AU7n$wz;62rRr*23Ur*wG;c0($>VnE0096iZh&L7EK+mlSbvL|=hmdzmSG%AuRBaPyjrPZ}%GHS9EQd?-r=^dfO$G9q5Trny(685<3E6}U@26Q1nX%CKti(lTMPRC5sD@iY0Ag;;9K&8qg1F#&x7E;SrJB%vUx5m)XUS_2nC{Er@TbBt!mE!Aid?!~)uxB19-GIi4OkxKjy-g`1(m^j21G}M=RTUpr#bpLhX+T*iuR5r#OR9xBS#sn)=HaKRmaA3-8cJo&7Rn3jQP}P|B4FxUIZzms2_)To?TqC{b4tH7d(7*6xQrgEWtWmNXXO*JbrR8pJM?7oi9TZ^lZfpCiwU)g*-R;yoTfSPHp^U`UazcdgS4D-;#rmxdl*}ljOJ&y(n;9>^1s92(y9AVEaGJM*yyy?E-3ssGQU6!UWj(CA(pc!_BwQqna=E&campWS`{u^a&l2H8|k~4Qqf@jA;dv>LTq(H5_9!@P3hPSbCSe?rZf;~RC0N#`)FGwm!XO#(of)xx<STEKsO#l-6wks1|5KYf|nt6Ai+B>FXj`6r!r!zT6Xnn)!_50!I9Mx0aJ2&a$a*x7gY%(=LJyN-Oi<AyC`Jr$#SOzYMGQNikF<TlqZM{VNjJ%=VCJSd+v;l7l60Zv-#>4BL2^uME;DNKn~RhvGi<__mn`X%Rr(*9M7n`Pc3bn$kju6Y51>N(A_T3q;_^7Qse^X^#?BiZn5K|xYBB<=_Qg%d1Fx7ErJbA2f_CT_RYI@Zmp8H(!IHU<F2V~94jg`vyBZm@<c+&D6$xfqJf=DypppIqG482nrRbS-^u)6>?OZ;',
 'cnkang_transport_snapshot.py': 'c-qZ8?~mNZb>Hu=pcND(WgfM6k`iaFRp82UE5x>hlWZ5bZU`QidrKXc+>l(|olo~69|9ymix%jIv_J~91quWK>hw#|eoFtDDz*QMz8^E>3_0ZPN>{Xky|+uwym|BH&HFub>(;I3@lV%rRwZHTSw()5SLJ?q9u~26gfGgfw5lS^%2i%eR=I!|GwbBiiWYHnW`)_zit=K$F0GgON$KCZb!%^4<V!0E=Id%*#6e&sOX?bCSzd)zl4s=}x0!`iTqVm`wvfL)3qQZkvzXh(%P>j(b(N&0Y`O}IGRF2nl?Srlo_Klsb+VG*`A?dhP=ERKWtnI4`!cK+JkAQfK}$KtGkhVw6L@O0E>eU>2*{T3tX!o@<>9w=)_i}NCz;lb=u3SI+4%?`eUhZ{GkUxyhb#p3_xAQ?@!U%Da29Y+#~trk@DK5|CWNO0^ac<?;QHrDwFt6s89R2AorT$HP-fw(T;!F%y0l&DnanL%3<BWC7YU34Ft>%*fGJmT1RZo5`}j1#Ni&Evk7(5$I}rHrECSnl-zuxZ!C_sGMrW|ge6~*GE?|Dnpm7ifKN?JYBvD+zgkXGb>+m0LQ^dfKnSc&defH7O^T$Upj7HkMAd9Q>yf_QCC9iC+lUa~hgQn2>ZAW3?&6sbu0<ybY0u38j3;p-r=Z{}}_4Me+TdatNSPM_n`UuD{1ULKyb~;E67)iYG^%r{|KmO#SFFt+o99mr2r{T&TTL=Dq&$4lc=IaW6A0Lc-&$>JEr|>wes(87oN_c+g5gW*V_8OTmJO$AQFYParKTh(`o*eBzA`(49@pk9HAK3`ISeI20XRn~)Ct+H~ElA7oB1p0{$>QKVOc2~%M1h{4TvjoTelW6b!^h!0@uvks?Dk+5hcg5OJ?`B{7$7**hX;oMxjT9UQZ<fXO<u({&EulT3mp3HwU8lmCL+MA;9Kd$7m)|+4>XwAqL(dX3$SleCK<fUqL>@%D<Rq`8*7$C6}I#Rc()JG)8~&7HJ7vpZjyEbDf)b&fgWE(@d`P(JLr{7(+tj%Y=+B7YH0;xx8nKZ@^K5Ar{2WYDM{4iUSn%+UsabYLdf+4+KWMOJ+`jk!L@xn{=sN8z257`#DOIRu_D>hM2C8sFSwCw#lTyDV4h`hg!2`1a9b;`Pl*0{yJ2nw7=^GUkYSFmY=i*cx`2?a<Fb@P7xY$;V=5VKT}4t!N#+nb+RyTHN2+%aA^<rl^Lbu?8V2FZ-Oq~zG&?G3QVyQ=Dooe0LF+X-gTyPyaM;cM@@$r%TEQRX<VCRt>X76O@-zC$T~ILOMOa({H0n-5w_MNX$pu07`8&u++pm@@yVcESG^_Zcau8R4wqCAEhj4`fvl6#jSVl=QVWD9GX8=l*Ll<91`7Fs!C-%CU?|;yNUBs(21V-T*GFigaD$ZsGk~Fl~kpwtnwRa8!ICf|o@Uy}ampcxZ%^u|6Ooy#V;36&mJQK53K+My1xuErr(y%N6Ct$^ybF|KoS6E!g<uA*+*$Ziy@;f9_0FpSVg1{-`bnaQOtvs+$vw3nl`5ahIks#Q>hxd}1a0hl?ZcE9!508cF>Go}@tUT-XZAZMw3+w9Iy{<a}>sx2>CGZA_3{+@dSDz?IE7lttuCRILIN#V&20sRPQx`{SKeQfLqj5*(6=4Em=Sy5PqO@Z_$zTZ+FzrbfT4*Lf_hzPmfeM=NM&O`|b?{x#0;G_A7WA~1pk1w#*a|BvjYIejU;sV<pOv;YD2XF&<XXSOgp-oBNmjMzt~5g|H{sZte%)EaC55e`jUh!nVDjF?Iz$}8?9%Bj)dj81h4o-!jcENZaP@m&?TD-F=$aX<V`j81j2mRN$<6SZhJRyGc^eI%?o>ML!`S!pjo32>H!%-_i}-=-*D-IuqUTPlb%0BgCd&k66iP4sbv@r#D4U?k@w6k0dWZv!M|xZK{5}4$%F`rDVr+JG?ZN+gTd+7l@d-s-i_trl^i}G3K|n2iR1LNP1oS4YfsR4xv_Si8KcRN*C?mI)KuUY*T(K3^#W@G{IfEfd6Lnt(FdiTyDP1d+llAO0u7o-vTT5XqiH5RLf}0Qs8Mtz#3k+jqFbk*E!!>LbW-s01cG|YCwsAePS=eFR?!Ve_w7)p|)RF)}$whHI1B%VoMIsn_dzcbTAi)59R-PSWb9(JI)JA<LXbmf0gp-j%?OB|LXsyA<SGdK;<*|Y0!AT6l0!cA)t$h-OK7Kf^8Ra$zFe$1)NJs|qzDdpLkQq%)GccMD^}cPLJ~b`4FnjDrd7jSl^UMk(+`J!xx-8caC=2HmNDo+v{w1glZFI`<3rujiauk$dR)k>_X6xADEjc8%fL$$$ysFaJc6ZqTf*g}AcEXKsf(GT?cEXWF(ZW5!^S2@jVS+XJ3KjwRB*9oHt>%sIiLC%i9EhU=X)%qN;@3u=9F3WK%QTKx4n5tx*!Av=1Yb&N$RoCGYT8oA>W-Xlw`n~~XHyBk2Cq1VE$QzmLn{b3k{kF$!IR_1fRzm0I{02Cn8Qu{62b+r^VmQD3uIAMWTWF>urbhLz;_%>ggq22^0Ea@{)46hD2ATUomL*9UjuwTQy*a%SPqQa$+JEfdDgvqcioL<;TKsM&SS<d+ZQ$o@bZ}CxP~4ZCMb7Z<2q_$5m!=Kq@7o$@%zG}lCYLJ%reiCNC8Mj4Mq6?S>S2-q;MX7#puHIkS*j3SDAh_FstHHr<=y`(l#d4Y<>A^BTR@exs`PfpkrN6btq*74mLoM4ehppdK0#Up^-_r*0b_zv|6y(HPCS+f*v=LOv)U&!7=Pv<ZB>(9~Kaqb#<zMZd;+DDluRinFItax<X=4LwQ1<ZWlp+y+SqUNXU{qJPUU%8KvuTQorTj;7ZKmHf9vNk_{W)q<CaYGhu)PUX1(E>BL60?b|p!NfxH*<=8}ckk=~^4I15<-tCP9W(qMf(p!V{J?l@v=LXJNg1x&&4Mqv7C||D97=w-&r;5@w8r+uj3ZNM%nh3lMWJ91!0p5YXXSSxc6d7+N&~fO?fb1Y4*yZXd{7vyEq>NnQMU0t;l9kR8g4q8EL7*VF2jqiH841pwgQHS$bW7o3!3~Y{)nRmi8UP#9nPMQnG2&WQZHx064X44wvtEG1E51GjqXqCkS7T`gjJ`ke6=LJ+2jmQ1_Nh^FJk44(FOt(F<JTf9>NwOzXjP8^;P!+Z6|RN+-_>-n03=kQ=ApKBNmfxf_sZrjpe*={kY(~Gh9U1TZ~N5+8vC3TfSuK7m>v_%W^qBbXLbhR><qiZj?K@>qC;VQt#xGr0$Qz9Z)-2~Fp4f5k%MWL$26g7*UAuY6-mcZ!XWKBeZL1ef8OXI7MMO9okP+?gsueKvnsu`Flvi!LKFxVP2TT7OXkK=tqDZOMuylBN5C&C6hQMF!?J8t&0_4vai2z}g8-}m8LwxTKjPQ>xCPoQWgyo{KD!K{y5aEt_laC$Xn4{YaS#qH5>YRZF$;o*^D(|Nf&#kP4;IxjwUOPCBFE!{5hPfA<S}3yH0v0ogcPhHt5kq$$=Z&k3CD&jfQ3OWB=aOZ0ZZRJF^HD*gJj<3atH@e^=Y4@aZoxvB=12*?7%J-aoWd8Xc*rc-2<EWVC0&S)DS$L4lka9O?Xnm(*U3c0DO0(4gjgq-K~(azW`ydPnqbkbsL%W<D{fLHMZ{9+oKH0rDQky!A`t)_``93bVPIbd{z0@YS$f#t;m=gy1Cm<qd+}KcAj}MeeAdscW-aP)u5#rC(K73IHq`d?K|mGDh*gjbt3}(I|lZ=(*IZugG_NDWd=WB`&p)r#~YW&GsQq<qN`^>qvg??w)Yg{$bCm58*3PB)>Um8uUQJmtoo;(8t;}5F$GL}(qlCC+7<;BHyAoX$RaFYEzxb7v||Wuj-&vB6mwoj#(4D>Zmn8vKo}>qXV?VN9BC%zL-Z<`$5ph@eO9FkX6jj;xP=V4N+3mej`22l*bj#Elc9rs+itreUqRRb{fq@U?4^~18nadqUW35rI!tAn5R^Q)1n{_8fVlw-8i&yW;&DLBnWRpd=V$AcZ=!~R1W|<Tc%B-dI#O);j&v~)w3~_1dBQK_3e4wlozfPMpzZ@CrHy$M_ZTRQFxp^y11Tc2!XpI4JS8VoG)8vYg)z*=wJG1MJah}u)q~B1wi8ru&H}LcCN_W9@@8eeE+Vo$sS67&euxau;V*tC?9qPLfV(w#u8MdLB1J~wY6n3?*4n-S|6<ROzXN9!mu+VO3oJlfP>xMS8n_?*;xHw=J*$lkW*9ZAuYshJ7AaIq13^aI`3z2i4b(?*20Rf@WXuO=CWRX<N`Mj#<w1|+XXZ6`VHcmgmI?bTby3j-V#tl#KIxAr%k1`KB>d1Kn5W?>0}%6R9#C+C=2x$uo~p&Lre_KEM~Qc(ml`D)p+1&8lK|^unV!Bp<A(UHLZuVPEJh1^+;9Wvts+@D_8r@0)04g)j}U6t;2zs{yYxv~(15}$LEVG+O-U9*kx8NT<{+$qi<tuQR*`-wbcCN6&}L`8Jr3IQPz{Cwb?$}U3XZEjQ@>L<poNqUl3Sot0xm3f66=MFP-CmS#;-Eq-X#US6+X~d*$S}3$U%mA)tyK66EX+I11R6XP+t&I^_mH1<Rl2v;sQspAT%&|K;*&o)X)~gsV@vD)3ADH+9txA*TnU;58`!MI_&^D?QB(WYkb_WxZEvT0{tZBI7e?V+mOv43?yqwt*N_x+MnlfT-$;)92>(|AmYZ|>V_T5Lz>7&v@QzrzcFPQu<8SiJ!u5Hr^@(=kR^?(MQgCFO*=6VL{CaF#1Z!YY3%&aVAi%``P{`|PfHTjl8mS$X47qr+ErR!T0Bhj=pR7Xq#UA!ug3TYzEG_T5%EX!+ibZDPL&~LB;rI2MX4PDQQDlI)tK$FU4T+b3<(Pt)M2bGa*j%Rby8d1eNha0&-zz!9^C$Df<8IsSe{UeR|hc_qcz!J8n%@iZYNzmYbS3^%uwtc!tvVOFfZvsNs-OoF4NH1^#t6pv!Rfnws&d^askFcBd8wqRl=E<q3-wo#5(BVz~S*t*}Ga;$noKs9H}R*XWUZKhMM+d_%wmADaE)k_}cg>A7$2`zIgHMxwZe$lA>-OpFmB0nx2Gk6l#I7P_PW=An|bWVo!2wzn?XB7ce^uns*X7ZFxN)YzF%q1Q$ysvy`Zv`aWspZX>QtVyiP=EfAzUlR{H$)D42}W`&ngfDOv19psR@j%_m7V%EdG%_6B=O2EZi${4B-j<p^`Y~4+HY}^Yi?avSf77XM4YF0$V_6OSsu*Da;mxzxCW)hyu;&L={VX5v|4lWg>pXlsW0~OC0oPIrkuo?{jSi4sGZv~QL51ePgLzF%yg@HsR&C|1}o@_F-cC{I=&YO&TZSp=p*$T57<V{am0jy4ZcuBQ{{-$aI3|pPk5EJ;~hb4&i5)PNR{DNJrNzDZAmUhv`VF!1x;Pe2Cp(oqs&W1q_&ax)+r2e*%62rqaF^Jl%X#72ngSdk_aMx5wZCQm$qpD50kip)1LIq*DyDH3yO*RK0YU_qQv4r|qSm<1>wu7SNY0ZS&CN`HcS6O*7*^Ga})x0|v!oEu~X?sQ0SNGPpVUlcM$Uc^WlGppA5mn5*tu@}KSPvl+QUl$%3~1o#UOM8WR$@>BiI`>cYV<n(GUB;$eKu0GfAM6lNl)A<XDJ`mhbKkhOZSvzUjnMhc~EKz69wI5)0V2CHxFpu-5Z-rtF1Y439L08V#!U3MhxBWm`vP)M>ppbH|G<LXKjY2`hUB-oL#opR-%}z=1ogemm$xxQXX)2-TxPRcAKz(v9$4jr_8Y>n9<Kgw8ncI<1;iG?>G3*{yeXN>=LD6MX+t*71sW82VC8|SUqxgbci2!@#6>h@j=fza{`*PXn5g>Q!lpJy8BdI2QgF?C6vYn0DaZ(jM3I{K4IF((lb1>TS9_Bo}_~@mPuJ+npi<y3u}s=AZlJE@%fP3*Q;2X5R*umH8G|u8!Ke(aTCO<a(m2^6ej^>rQMkS_PF2z0LG-K^PV}_R1;#6uOOyW#9}R|==MX-eTzkh|1s)xf_o#b%?v8YV@mnMK|VE$vDI!Di~;p~cxp}X>;vG=(GKBiGGgW3BMiee*t0z?RMWPXdzIa^p%!Zg)n!C$2brL{UL+Zq<<!S^UF2sXz0FmDLJMf%<XswItze^W)#4881BXY}D?EJ?&*UI<c9sv1sKe^3rcGoQ!f=HrkUrSD>Qf!dNUKCBNbopM(ZHe+cA>(f1#Bb8-cwqCUNtHKx6UQRgFpMUC5Nz?7!-qcvj0{NSlt?pP0CsigfjXP1L`caiUBGXFjNv4a~v3{gxq^NV^ibpk|}zsW0>x%eC5!99_i>VMnVv4M~!JRFTs2PlY$Cju&jd$GZ?DJ#)LA_F5FojH4I0#A9e5WTBA@+JJ#K>60<yNYCgm$0|g)E!jBBh_EIW1vhVZg*#0RLNhQ@~Q|SF3maazVBf$vbJ{eJv(6i7GkAN}@Wx*Y-OrTXY08-ddvhMP=-DiZVu|OV5ivwcuy_v<;#lDH?Ul*QoeT!IEqQSK9a+V&8foDY_T`EDC%IjpBAlKgJ{cA339k!FWmZ0|NdTb(UWxH>2c&RmdUv=w^)shO#_z<)aqfLPz6TPIz?3rU*y8hzeT$ft!Qrl`mYO6b%G~BrGi_{tHW?9|a##3CawFMrg(8gP>Yxl~nHc~aMjA%h}70yAWc=Q|=_2Z|X-4XXA3JlS6gDVKI0GJ(P*y^x1*9ioT@fn?7Ba5<JgWxM$$I@<w`l2u{mNT*2MS*gFx|$BJhv}xRU8<l1)YY239zD=<f+xKWFqI#n1KbBm26O7#KtmnT)>n^^Q%zIu6n1n%6ZKtY^*v~qWmWG*+d4L_-iihqI<6Gx%1vnoc`<McN~gv_x!9<$&%;SKo;`32&9tXdX`7AJ4|Skmd6%1Cnp;!|8I!!g?0$_KDYp*7RH9|%2%1DkrnWdp*e)&w$)vG1B)X(Y#FDVCg&nY$hPX_xd9~RX3?Bd4qsPx)JbC(gaP;`gCy&2Utvqvyc2i0XKjyct)DqWgGEi?x15zm`d>6x3xyaD%)PHkZ<?4nsFS*LFeO-e%Z_N$lU31cnq->~mli!JQHVxk%j}O~t)1b+FV4jj+I@DH-HRP$)m*MjLcvv4xo->qch6Pl#7xn{sc>I%x)7uXn|MrjE2Y2YnL!_GYE1|;^>oxv)?rM<euYU@%b>$qmqL`RUD!2t((Dt4@n<xffpWw(i==%@Fp;d{uN-GDofN(D`myYfQLUHk&{(&BPLAt?SfkpP=4i=i)Pn;i(;c&=n2wuDnNw-;iK;-pHJd_#2UA9Z$gxBBw<G+3P_y6=?|M8b^{^f7q{Q5WI_aA=!+dusB_iujjudl!T=0Cst)i-bc>Q~?W-Cw)5$5UgXv~)foK{ykVgyTqLuq1AR?p;&Wrh__{)@8n?nAKvv471n7qZqn_$1uSnKZnis`pq}LeDm|)eD~YG`1W@{d;RV2{{7qE|IM49|1<yo?|=9iqDbdJ8ADex3T+LCKG9`aL3v3muK45KR%&<s(Fq3>&slsf9+2|DjUj%gIaVXXIfBXIeZg^Xh4r%nZl{gdVwr{DN)=-qAZ{+gmNp6)G;l5jox5xnrg%*QP546y#mLFho!DGrCU4Q6G#3{-$_0hI6f=Y(g|t?d)~QjeA-2<)n=R=T-01{I7;4*?E?_jcPlAz0T8%#wo6cf0*;}v`pN|r5cj_B2%0Qkn=@pV~eXL)c+~K<wiujx&?aEzZ8!r&+3v5boz9=M8J%+w02A)utd^K*qIue$<eTWt1PQ9qo(ZNkA*tmPS5`K9&k!)_ju<L8q$5@)7#PJ|;qD4Oy)LOXK`7j*cWJVb&(pG;mRf$0UsM(BY35*)ELODT);f*GOV&Vp5Yvn3ZxGsv$-@tJPaXFoG4#yUkULdGh=Ws!f9SW>ER@v5TZUm^qLy9G0Z+Ddw=iXJbNKGra*T&mayW6&tU~86G>UZ^0UbO4~2cW8R-v',
 'corpus_runner_snapshot.py': 'c-rkfU2h!8aqs&p`sjz*JuipkNxEECo&d5$-yKdTDHL^wV6zwxcV~CUoSoU54^lM6{}4YId5v)h3?wiT<WcvKhco;?^iNV%{W0A=J^LXUwjV+P;%<LbbyrtaS65Xx4-O8_>Wo#(n3c<@h-Yk;uc9Pl>mt91S)47BEG}7;&DgrWNRx8OzI<}HF5)sSu99rQrg^chOEz7`)5~?9WL4=O931qL)jBUK78Q$iRFrXVUgRrio2K!!O7g5^qQT?5&Z@W=vROQj>a?0A(@J)pmRIs4|CJ^eeqAN0D$V7mUzK?#Kd+){$?>h>6ZDa|USNgzT;7ztO<kl2mPRQXLRq;^ld9MI=Jf34m(QQEfO&kPhee%badCL{nb&)A`in=eo}Ryi+S?w3AFl?ggwZ0-DsRNRuk&9gX&N0p@Q+#l+a#Oi*Cl&)&Q6Z~<A)3iKE3~tz5ew6fIV8T)A-x?;;W=OdT{UW{Cl6W{#Rd|fBkgG(&RE`p96980eifJ5ywX-_W%I?V=tq5R3xH{H{^pT;_vFXtje$|>aq&s><R{RUevLupT%i(14SqPka<y6#jABy!uNadwMtfTUgPM;pBzK|{RjR8iZANfBF55_&!AMIDc7)=A@E5~{eKaqrNqGt!+cGP4VAaD$U-Per#Vm+fHn9}9tcE^AK9yjqnXy_9*(%FtBRXY4gBx*x_OkObrFYEmGW+PV%Qf2ED<c>Dxbx;g4i&gg?X0Vc*CaJSmtMW7DL^X)+W42vZ%O`#T+UK&I+KEaT;eyy$a_^QC8-fh$a9yjOTGRT{^2()vb3C!KbiHevRWiIX;FFT<67QTp(dj;M*cv14f|h3`_h6=2}1KqOHyWCrC3OoWTM2jvI2NdA67?qngG&1*tJdeU)Ez)X0uB6?5E@J0EQ3&XdjDY0e468W7}JRMD#PrNRZ}Orlm9n5L{(u^6k(6lrd7+kG0Lth|nrN~8CQQK?PilVR_!*Xx0{W5m5>v;xsD`_m%2PEij6@WF@>uc9oOgCK)?5w$)I0d-v;nz1A1$u>R~dIRd0%vJk2Oe|bR<x;li=%{t-4cJG4!ZH?hA#4MmJowa`j1(b@C;{F0O_bL0X;I`w-+M_-8A1llA7zVp<_qlCNws8dbbh{$vwoaS^BL?Mfmc`a!_Qz8QOU{*sIh8L3bMe|$Jm)BUiMSiiw10-7Ysi!*rj+soHCa9@HJWY2NOvL5V-zrG4_&~H=%w7b>hJ9X#Utl6bv>^`AH2s!W1}2bcb!Qlye3)stW~IU)};ZqL>-0JA{FxGyKBa3#_<T!E`y{cnkb+aLZfqIouH`59IW+yrO=7{5k=J-`6L=hlE;43=s$7WF>^4IH%7W`XEq!L^&8Ty#ytBKy>Tt<C1e3Q-jGSLe9O%%2WUv9=$qy%2r9aLVX6>00(ntl9l5ZR24=@i?C#_2>Kdyjn)awx_dFCZsnDwr@BjuAb_MKS|c@e5(|=|&C-WvLl#Y^aao3C6;*W^c;7sF9zJ>X{IMtCa2oPj;QIiqiuiR^MAIssc`SzQlX+i#@%-86q3StM=;Xp(7fBTxN<a?1lV9W-4s7-iq~#@KHEgDpGA%l(CBXp2Y-*VdDRkpzd{TjKL+2oDd6}%&?Om5FS&4hoAQ}4!JA+Z6jT@y{AebNmGZx1HYsS|^vweNea>NBgVLT6^J}=`8W=c%z3nl^B04VN@DGcSP2E8_OaR_>iU$1ANjQS?WnV19KjT46#d9rTcG62-_3UE&94x47ns#-hSY8BpQOff2j@Dhw;6GBsNV5DM4%Pm+q%|cWz!TiAH;v<*LC>9gTV#~c;0Lh|)4C1RzGTBO}n9+1A#Wa}_R51EWDlk)I<diKza?ZF5Ya>BXnqEZHOSHS8L(nHVX-LFJqEG>Y$5I$gA?&j2JAX(ua_N!at-(k!8VuCJaYi@rdtIw2HMXXX?Rgp6rHC`wpq17Z4ac(4q>GkdA69T|2&<cQ9C#I=J6cw&^dVYNZ~(0WZWDENmz--#fGh>!9#2qO=#*o%;it4t)`Ec-Fl{hZTT(B|yZeI0zAj1ClKC6J%4f5_DWql-80qnV{nR9|tId$z6ASuh+x}cu1qUj0X%V9nV3TPfEyZ=duG5J7Gd39MWnLH4I0QIoj|)?M?16Hw;NT1X4`7%cm$a&0Z-S(h4L0OzK!79LZEGuKC~~&Fz2g*t=CChdJF>VZhBpSRZV)SKh!(W4ej78|VGwP!AY41SX3z2lqn<4_Nmj=Vru-V6E|>AmNH<U6J3k)42OMF)XC4D*c;-V370;RE=E6_L7K#OGtMr;OT|b+0;C)v|;48Y(0LYSg8Z9_P($4Y&-_~^VhE%0)U%4^4HeREf3EnKWfteeZJ{$|WvFmu6#VkiHzJr0Q^%8~a6&@n)^i>9MV@cx)tvN20C!8DTN`WN`r}KG&XG1X8h%K;_HRK7^QjNKW6Fi`3);BgIVKCVWdUovME)nvE*8WR%W3jfAfzF&`51>ZSAf&0FjKiP;U11QzN@XROae^vOhg13y7_q2K^Xoy2h8rhsPL&Jy*v2<$!6&38>OYX7AdOZRvxpTV);}zqlLFnU4_}<Mh0b0VNe-Gd;RpF!*={6!%qWE|h@%#fWaDG^1|*$y=(G(J@Zr>m0gv?y;LvqPCr%I?3aU8ah3*$^<CCG+pxM|PF9L_Iv3B!=ZRDt_8Mfybeb{JLrF2m=1bT0p3sQsMY6Xr*)QV*7kPS03yc;N4^5i0_aJ8`<c>oSNRtI*dHL|BExG%X#(O(AW`$d~0149fI={g|lM(Pz=R|>FF7lt**<$nv>osCTb=C)x!YTK|o&j!|b#a+UGvAJxRN$?&svAxf(d5Frbu$0SKIE7uX8vQScBAK1<6xk9EHFZ9fuzX-^{|feU@A~@M@2VYa9UJ<qcOMo%jwEFW-LNh9d~HR-wUt28XesJv>xwIMn}M=G5fxgFQ=$}-+UO6Vhq|p~$yZ!YaC>kl)y6gxyRO0U9cREWuf+%?0Br=^L`>pzM(V!H+-fVd54XYgu)>seq|BM^R#=ziWshM;JM8^!ubx{ncelhQ%8t*X6&z}CF_7WL%_%axDwWgBhY+kc>|ka2)AufyAZy`Ol!r4AJke=fn6f}-%NC<$GSI>f!nYqlnAd*^j{R)$<@0_PU!$i#@HEdT@hf@~0kQuC_VaJ4pw9seIZxtz%Yd_|-Ia6v5>749%Uy{x=p(h-4%w*6SIIQIxT!!^xxjGZa{Q==Ns#++{KD8r+~<*x!!lWTV*5@h%8pX=6ZZJW-%Z(dd{F{gHvRE`GM2vk;a?L5;oKWGefRbcYg|>E8N5IJSIw&A$N!*Ue$J|QKl~P4m*`%fzWd?#5yRsqjtk(xYB3>(V4x)L-hRJgvv+TQpZPAx^MgPB%a7j!)OSDpf!!*L;O^*FHx}*?)kAjm<G<zXkH7iTf4}?T`zgB>`em$J3zNHtVn$81(28(kx7|?QxtewZ7qEcG<lWnUuUJY&=%fV<Yjnz{P`Qi}cKPn@|5og~8m7kzB+LS$?eU8J<EuwczdZk^@a*)(^Rsh*HFM6|d&8daOo%t^yrHEa8x-O<xiCmjusfmjDV3ZFSz>P-U=I%u8T~&penHztI?+mGl!YPHYa<uLE7EYEAG~3=*t*#y?^sjRHXgNFhvCetDdaY*SL?FRTMBD=g5Kj2Bj-^$O_G2j{;=|I4s<U5DY7w=^zjmClh7)Baen?{D9E*X_x5){FI>ZyLeHR!969zsIh9B3mu;2uOTJ8#-WJ*OzCHbUczSmB{Ol#HE%y26%=3Q*?s<wImX;Js36wEB&XF>PS7&ZN5O^f!$zsSp9x@TW4(<;jmijsb1r_j=0Aa?$mPGy^QJUt9Fvj8lBI&}9gaZ#KJq70=He<qo_WZ{m%S02v_c1TOQ;}Z!IyN&Gv{Q9HpL@erA_EzIVH`mU0q2?!EE<^>L7N76SG|q#&{x@ImS1N~N9;EdXmoJ)<>0JT5wMa(Eaf84(@jXsj%)?_EDW-euv<5-R`$z2`0UO*;dCc>?2gvie;>?R-pJlqo4z|?Z0CIJj<MBuCyYiKiG|F8L~rhdFGm5grJ|9VBsnIi%_+~7sD>e>SA+vh9s{SoY6PNCLxqUwQ<o#{<Wpwg=U(QOzrOKo@2A$a3F>bFXu#1XN8V%JwJ);QLX~(k)PZU?jYS0VS}8<e#w6%23wg7CyETU-*vU;_2g~kBUo>wmS{O&vjTDxh9XS)bz&1dtdAdk#o|=aWa+}y@OQg8K)kSZsL*ty6es6cqQdhc?Mgqx(NT8&gCmHPj5-k9qkULEK&VClKW6P`B&iFt_{hg9U4wavwb1-s~tImz1l|a>qKc<f~V{x=(KVv7Gl#$&-jW1HeLL~75%nUKSCM&5scZ+Y{*(D@J=>ot@LM^YU<lz)8l0q>YnS0tVrAxtAmOHeEa}R`tuJN40oe~5WeH<OmA?ULQOt6h+C8luoJzoReWK@~+7<K48OMVexh}imKys*DUS&3mbLskdxqYih+Wuj1P<Pu(9xlK!oRugWe33|1SHSUR1AaC!4vnH8=Rs#<OIc>otEWzasjvbM6jINKk!u%8T_6UQs*G`3{d}NumH*%!@DcU%Ci`JXE%aToC$TZ#rggeU=2tI$PfrG^5$tTA^fhfCa!gU(Q8hK@%qCiRy00B5+SWJfnU!YdftR5<!F8IW06!R8Ra6|?nEX>h95Hu-*@cte>$9SYRkd6+ih{KtHQl2FX=-YsF24$&M>C}eMjllBnh7h$@T5P=JKIF|e?yL>Oj?J~LguhO}DF9N)7m869K1@eq(KUh)ma!U&$Wztg<qf}SV;PvLYvh0^mD(nVLb_i(I(vC~){v+hnx^7*fV%_0EUxgZxf8f&r{~{3Kl^GQu>2gm7j*pZ^elYw==_T&>xPUH7nLm=Z4(}0|Cpo4d|qK_y1+N~^(M&5mNj8h*0O93DisW;>1Dsk#cs5IXe8>Y{emlca*FG6q4nvI2`yW@N%zpYQSeF5a@112ZIOB`F66*@{75(vM2hx921YlSH{haRXRn@x=Z`)og>-iM_&IDk;1PpDTufc=;#A8lp^+Ms)ly0*ExWAxc3#fFU!?g(-}_kf*s2ts#*1irgU*-Ht|Mku&g%7|h#DtEM?^y-?NHIR-Q(3GWFN85Yxg}St(IrcfByVDeDwJ#-0X38@wvM9j$m6l8H|tHFwW0jy*v+3pM9f2a5*wm$2EZ7ZcL(97H1SRl?5`~LmNww9PnH;{cL)X_*uI0vt;QnavdS%+8X{Wa~1gpU&iRhsL9tmuhUS@!O8;CN8j?3$!x7~mJ~rd<9J}8g*^Fifl2vd3$tJih-q8;7-A2xC9B3BK|CfwcgQU*UtQq-C`TZLWLJEgZt+!;g9Z^W9I0Po9D`2u1mD?LA0VZO=>_g&FguSy9VvmC!iyM+DAJThz>{KeJO^o7<KxFgyl?WlZ%kTdZt-+KqmN$Yn7%`}wdq-~Iy9PM80#iY$Erh4I9(9}ousl9$n%dI2!ceCBp9m#7k-I6#5gCSf+}K)C#)#hWagXl6P0C8$FhzR5ZNfpGq`n<B5$n$d^E>MSCrAov?yx2Uk8WQ>N<}3El!d3+T^4+cQ>b$RO4^994)cl{knxV`tzkRfQuM+-4aQFBfMVbgz+lp0|GUiP170^P-NKaNM`7xQPK=}8K-N^sBj+q8e6qNz0qFPplOH8U2%wgze)YqF<@dcWpX;tmZ}d3l7>#(gc{lf0+#<GhENbBD>b)gOr>fkOg86E9bJb9$<Y!{shZ45iN&L8$#c9$JEnOKT3b`2Xm|(#U?#JW$Gv8u)c-vE#nH;FDY*?{ZF423fwxX0I1SIVQ9AcUStp_-Gzud{QF&2OcG_Ofao5Gci6uRJI4a3}!;q9+3JE3V2Ho3|=p$(~7T_m3Y1}QWg!Ye&dq{(}{c8K*-+Puu&dlG)46Re)E;F*6c{H&NL)?3QGOZtZ48oA0=yBA|uvKaq*ip9mXIO@~(cvf*KV#yD_Pxz*8x{kHO3>U)Az{1Mcne(Y*w>6N^R(R_<)lRmp>dEJ?Zr^kF)6og;o4Y4qNKXZ#ZxUqmCqwQ!Uh`qPEI-Oh}sT%yE33X9I>OJQFGL~G<SrzBFwu@YQJkxmZ1G{n?mJDsbzc!q;f@ETi%!B_TmZy>RLk;{?@!*+_fIhAnc9yB4jgU^i47~-~u0KANs-tcFHE`sm>UGhfo2y->I9pm}(y$dDI>fvC~+;x_T+@0f^LckxZs?$vlifluLfEW$3})CA|hix|Aco4fGv{W=-;}nL1y^@tR>+oxv5>G?gbxOnJUu!tTJW)C6-+rV)~u!MP}9h@?dG4g(G;ypj?7t-UtXW?za;a^xb3+Bv&7&O42)sj6*prb+#HmEfA?3)tOSmM|Em;3EOcR9{)^NQh(9a!GFGLX4#SMCoY)L0C@VemK@OrlIF@S<4*euv#XVxS|^KoJhrVm^G=m8Y=PRp}r5zrY@&8H`fzX^zMXe^30O)FrbGEgrXiP_P%rG%RoX}T9?ZJ-xx5}qE1R7hRqPowP_4<I)b^SL#>e<kZB~U!$q)@op0YEmK&%7Bd^m6sWcmcffqDwxjM2z>AbC)Qk=^v6C~*Xcpx!q=QFs(1@A~mWsb_rJW3G8lC)YtgVSw6qZO6-(QHPj%wVxx9Cx*I&|_$Z=V13-js8kYz6mISp=sA)Adr{GY-8P81cyWhJN67LcT9$A`9DmSJ2V*bAA%-%X=E!0>y&&7x=tKLj~au0Ru>Uu05a}I&<~~jS~frFu;!@+4#9$rwsu-^lLvPralroa3Ea65%Qywu#e-U99z-b_5?q8Cp2x3~X>9U?c9>g5HyF^0u42qM4VBIbaJe|&3^a*^b!ev%aNat$No+gF+g*hBn}pChSShDj$3d>iK@-1+yIy~=v(;>`+^%rwFoX*pSSm_SeN&%?8qtI1<G|YFr6Nuf;5rdj?>0<C;h`jHi^*G21Uq}C!?kML*4<|Gqy<`Qu0YW325WL3wGt{?gX3J%+;1@XL|551#-Zad+r6@l4?A$VwQ)z#)ANF^biZy4<}*<Z0Q2=IjjJtcbwl2d4c(^`GX2ce(H{b7RUQV$8nu1B-A;R(vo;^nCT_R&sJFumy7`L|gmIG=c^}<uWuAYBt$c{?W$9YSzt(!uoIiA-wj5SltA(i!?|Y7Rk7Qsts;pdKbbeqO@W>H-&|g#lb7h_;i>2jQbg2QyLuiwocL-)Ym8M{t52)<{PS0^`jnGJdo7nal-OPi~8d@c2accoHouD20y}J<Ig>a}ab4`vH?ptPpHI&`NxB~%SXINkKUx(Ig`fg>gAz&xy6<GA!Q%mo<dF;GN^t#NN9b<{)4Wb!232UEa6*ovS=x5Eci>}g5qrcN&Jz&w{s?j8$BkZ`5)!ee~4C~%)ID<BNfd~7=Aa~*t$$cNBF?Q1q`$)YTkDdbC0$}9F0ydc>8g%EIy0Or6hN>5T+;FmKyX!FhcGv>7o#oHPVQxc#XKcW;u}EQ&v+<(7_!?BGy3m91OgxB*gVRI!OmfY-9w!l64iE3XXWRconmu-8OEYcTN<K6>LLa1kL~&Ro%~7Rgk?4xM*RpMN3|$*QBe^R1Vh7Q+NZj0N62FICjNB58Z340NP+`k-BgTwB^x4QDfA_NkQL`CoWE=i1Y&%=5U3}B3&bYJCeQ0)~)UC*%<L*qaJw_6zJ)iv3G=;@DlTHlUWnfW?EbWfA=n<Dj=r=}OS`55%tYveg(=iTR`Pj8e|F~@TNf%C=`5exn6f>PEu-$sjRHG-olw&ylhLk3)XEX~2_O^yy?RU@AokDQA<HeLvT<x#3GMa;ndp1K?pyzlIWp5hFfEUz?nYU<X^>SoJ3XZtA2@{=ga1LzIu03QCSqqzIDmM&ym!panqT7FGp6p<dx{bm1<9)iBO7S~$rd?Jge`c@sV$M#F^6@oLA=be@@Bd+<zb?9=2k*wYWqKd5jrm|r-!cDv!#E#cLR*aW|EBa`)|7^z3~;b@{;OOE!X0wwJ|u^3hj>AA0a0W?@^=gET-%P+;0}8?*ktbxHaWF7`>fw*h#UN$T83G7UZ{A-QT_=E{~5jjvW})P>fiu#``dk+XSDWy64HR>WF%U#4PmqH7!;5PLNcbNY3%@F?0YCH=bHz5$|8Tqgpw2aRM1YrfT{84ID9M(@vmv{M9EYH<n`{o=8|B$zqn*a5D-=${}Wd(Z%R>Jju9NC5Yu-x9?Exy^^RY|D8#_|FXLqg(Th(Kk6nHB@_NB{c*QA?ibaj_!xvQ3pT*@A-j&BG1tC0`FqOAPcux%ehLPxkS)M>mdiVz(wSX978I1Y82`NC%UWbSLhAz(^2IpF{QXTENBHXKs{7VZ%x@C*8O!d-daDQV@I7}VFVM^<{u{o&LLsG3yRQT}4rrt92c*sMKLd2m3odS3t`h~ru5N*9VE`+x$DVOK9%RArdUS#WbD*wAU9ap}wPJ<#|Z@CG@{m2$D*ZU$kC!dE|TKW7mJ8HmJA72KOCv_^8$Mv3l3Z_~a6K(~2skU_wquDB{UfVBwJ<!h~U9$~C+BCwDo)HN}lmW%ItY$C`0*hk>mDK**58@A9pfL9S4@{{&UH',
 'crawler_snapshot.py': 'c-rNC>ysPTb?5i_D_T@}8=x>43?Eu)M98KoI%Zj-6pHe0YB)hKKx5Dv01b5m!x<0fgX3}@-gqnCJj&TkDpI_ur0m#o5^pvZv+EC3>HlDrDaoJqU)bMy+}ro|12aR*yQzxG9yYq~J@<9)x#xW^T)6Pn_=|D!+O0dmAno6utS7U{Q814u$)X+HPJ(xqQF;_i`n%C2YX#F}$p0?Vemsk3dqI*0@oX3!;<<g{!iBX_noNUUZ?s%2)2P=A;^{m|7eRkEOBVe_oXoPdHS;droA=W!k~)KAGKmIU!c@AEEN6=-Z3V+<)L%{(!+5YTg=d3hnntrlTUxSp?vB%_KYSxeCN~eG!E%wHdOzz8lIeUBEux{VzlaVOlX%zE(>K%ptiKnfQfk;=M2mPDnKI@#hL68rOJ>^jXxfh_?d2k#*lzH^sC;jc^h`Z7@V&3abMrjzXJd53{FHz2qCJfkeN^n*FZ2Fzh=FG2#oJjjGtblhVk~``<H;G;8}vb+)BbFjOzkh*ooq~9x4)9XezdT^XwkAoJh07XNA_{SbB&d^FPuRxXYl~Y!b9MmEz=2&i)Y4k5AWVi;#rFxviT%lctz7>H(7A`)Ks`buf7ydqBr=p#ijXCKTVVSSZd5b;bof^f4PYG<DmEv@PXnB{`gKlIe3>X@{y+fQ9S7b({wP4$<TCZFIx0Qli}LhS8v{V^VaLPgKiKWY+er6-nveYW_R6*vZ$X9#y4n_Md?PTy&10EdHsv8-|b!hwVSu^t|%L>-M#bHn|FIRZ+{h~?xxFVZEX#EB3Ld4jpo)`fdA46ots${ZO@YX4TGgX%|WvrXUQl@fhmoqRv*TD7-WN%$cLgx+0MrOOIM$l0_|~hs1;0oOi9$EvJJvI1#v%Kj2Z3ZWjmQivqqTihE1B{krb+Ei-BnaqgD_~+hJ9^{Y5g32ff{+MU-(Xt-u)ry}&f(U9fLyt%+nz_lI%XkRMr>=Cc)mf@O<dvd=%I$|9Q10f=kTsCu*hG-8l#dJ3KkMq&HdZh|%;LOcY_M)Rb-n9jo-O#6NsgBD1Sxv%-BG@4KPgGl=-Bb4H_!*zVg?%O)NiosL4_Ha3!XAP9YSY{a!-+nfT<L;OGlMLe{B0XDlFE!iIjP_+CTrNiIFN96g<Rs}2^A>%4;0CBb)^Jvq#@!>L*<g^Ef^HEgRh(!qvDvcvF@#~8C_%#+OIkIOCZ;{SHwy#cazJN~z^xnAI04>(ocQBu7;?j(+Ej>F6d*%j+$6alr54*}v}q>s*P;fERRPf>aKj;T?Yvxg`%5>%mXuMDW|6~tDLUWcA$V`iI~RHywX@}JBMrCjErNT?PN(15d5#JO3ccwL%9g%2JHFgH5rDQancc&7wn*c7qsh~ENYh7f+A<m3fo9W`(ecmGOKg<}j%%}i$#P+K7`<)dw{#^Q1r7SR5rjBN;}Ldkn^qKK<qeh@a>f1b$2bYT+Mg_=n;^w$BYb_i0HF^06GE*cA`L-5Fo0gd2$ytp@T3mLNrE5!U>4mE*4NQ!TPH6|mgyk!`X{fH5R7O~ztet0=suk#4fi{SFdwzE!5F7{PmTz}jVySk8*GR1kmeIu6Yfwiz|?>~v=>M7C<B4*#b6@9J~WCDMsYM5HmG)!3$L7on}>5SLJ<xaffmF=@K3{lmIWLa8YgCHIh|!@$nh|fLCKcv@7gi7Owlg)eWxk^n31Ald$ycRz#oCtq7lspZGHwR#<4#eEE*`&Y=ecG9|3JJ2@}wBdber_uf<uW+~IHu+#29mG7EF-729uaiMp<wUPE4w;uN&F=Hs)$TX$X!`pd;Q0g*fk;w;FPqftDFLEjjy=V^k&1kJw`9AHDzkr<3H$fIC4!gu_L7K1TSY-tTucKdiky8#Oa%A5B11&C8Vd%Wh(ps5-y1uAjjwiIo2u)+x@YZB9mm6z8UCya?PH(;!0*!h6aqJ?c5Y{D*{5;*MR-yAO%^X&6m8ynldxxpJhCuHt_rRg7qw`M$YGv%1Et>757pGi-2>dk-P2*Jq{DN84Y3~ZpO%gyaeJC+i<HDq13rQ%NTTsPP(4l-L#8}1^u52$n4{LSE6u-WN!%7bu=mtg~@Y-Etv;AMtif{Y2w%b=N6_-W^-PhvqpRSZr(8{B+n*`KV_p#VxWNd_#o2w#&%v@OzjcggPv2dG>oUq>KCbQ%d>y|aYGgNY@nrA<uZesq*!Tc+6B+WTfJ=t!&g$1(K-<Kk-GgxOK~$fqlZ|8bQPrtw=lmDh^v#uVkO?V104n09$mZ1t|n7wtRzj}qP}8k$22?D|21Ri-J)<+%p!6D8bHG%M4BRlBoUoRI+-u$I}Dkg^&pwr0!Ux0~L_!ZthqE}_BH?~ys=Q$FC$q?n>!sYyJI>7eyyy|~OojtLyC99Jl>8hvi}=U~K#jboz!TXw3LtZfBMhC*q9ztY2rKO)He6&S(7KMe)sln?;_-^1bKjJ{c}Gtz3cJvP1G=5jlLf1Z8|Hp$lR2D;-KsT>Y8^TajUskgYFvy?oatpFRgJLyk%hy5Vk3Q|2m1^S|&E{Jj`BqD(G?=Kc9<2?Ek4}1I_2yz?Z7a&y?Nap6dWu3SdooM`;4%-H|VBV}EX>WpQh(|c%5>TS`^?0~0&}g6QPe5u!hl9y7iw~lvWgbR>ZI@0UYAJ6hizdR1)bSTMYOy78LN=5+dbu061R@4*US11jvYI>IHp`nSuGaCzi>bS`%*dza7(U6@%5-q`dO4d#n&R25p>$OT>MP8RrN9+tOOG8$CrtE7gjBw5eVxysV+;PI?VP&iTsW4oZ3UeIITu2G;#X)d=fub~40XilbZ&N>xsVPu4U>LWNms-)W!wb_v+WaFHGwIoR4$g9!i)tyVY#Fd>Kxm?hcq=<K2UC>&;r9b?a7SMMws&Qg6+hE$7|~6{-J*EANrs3HY|$?(m;)MP?e8mJkvVa!<OGFlYwvY-~bXPgcy{bU()4HT4st0Wq6qc*h$@V`9hI5Q3LEPAWyHqAi7419YTN1R~FHm@cOM^Tc*pq(7GOO7(Ef~EGRBV@g9{JB<Xya(K|k&=oPDJLuZ7tUibtqMz=6Fb(TyHA{=@xyP&Fl*>ZF|DkQnUXd1vrQ%hJXvq8d2lEUXl5JtpY$;??2>~_%5PC!#a#hqnpYv^Xr5;6Xk%&FhtH1NxE9sEitgJOXqa<TuFW3Ac>&g`s0Y8L@84+YB4B5W^R&J69(DG%iAgE(1cp5$uFHyynhUJ*Us-to8}Qh!<+;1jfD!t~oTutEI*wqUd1d350zRfSld@K`TY2`zw8A;afF!3@$O6q6nu<wnqQu3G|>W)Zp+lMWD7w!I;i-h331+kE9;XOM)#JFGIe^~webjng<|*{U7Z^np}(&{g*D_ed#48cQ6WmpYx`TGuN<f*GOdg11|TSg*@fLJc7K^KM3Co7gGU*-fDN=(3HVJ&Z_oO-la;YRKs;O0#AqSEYJV0>S5-+ni$gPBpVtH*+)s+7Y#LC*1J>J%%1B?GL2&mTj8tDo`ho)+5}>-;d%M=zyH&<$062)*}z#Pb)%11<C*}S}r@9^!Iusx}g%72(DQ!1sa3J#Xw~9pJAz_#yPdho{+$n7f2xUBceUD$Ss9>+K7b;^hmDk5-m4K=23HK?$Trc^x<=*%cn{iaQT>Wr~_?#(tnwitLhi^#K;%!X3oo2H9>@}bRJp?yPWAERXMOwhcD;7{7{-ZhmvgJs*#WD4uU!+0X@{VNw0Z=S0kJ@V_n0wwZWvHWdYk}WUt&j97J<c2q<ABt!WPjSiI=<APP-J%nyTDkR@{gTVWR}VUNXzx{wPH1I!Qk$K{-4%eI9l5QsEN$-GBjBDGeO*7gCTp$<&?N8QafcqE)lus;A-FSmk=q@RcAkoD|aI-Xi(Vxh&buV<Ll9@&@xxVw3kK<Y;k6ZFgF4!+dFf$};%>g_Iv!1?Zr9S6t=!zAVZVliPD#Tnz_@Dq<opH278DN0MIqs~X{11bz0HSHsv427>MiwQeZ_~?onMDQ9_@SBD!ibdmE*#}R5RZXCaRi;ZQtaeEyK2NfE0q)2skX@SXwkE723)cl;g#Q)YYE_;c`F~+Ga?lp7{BTxyE<WYJ+_z_kl%tyqDSvxbNDg}DfSi{n9&-lKv=y+TL1os$B9LBdnpNnF9bGY3ht`UfJ*qA)e{NvWiDD)FMbj1|{{Yx0Mnn&^B!810fxGFPNv(X0VVqb*6KKpM8H}sLG+ia2TB8ONsur~xmbX~m7iacKBUM$rT+(l2Z`o&#HMi=%3V(=3p(juZ?`PWf415nZh~E>pfV`qzcG7bGUpY3Y5aRTR%a&T~PrQEd8#2-vYENzUfNM(a(`b)W)kmxraqx;Y)%jQpRK}PRr7k3?h3aY>jKRfeahlJ-X5a|q)}7{5dH@f}Vo*L97SOm#Sit(cOj}E^s#H8O+HYz*%~R^M7b)<x7tIdHXTtbToK|U!;X#{axUA#wEy$?r*Z08FVVA#k9lgFjpwphEYb^c7crl%{!2GZV0w6Z#!_iCcbUW=ATNgJj^3&Duq*-)(D7P(B`33g489=TaCqp~A#*WqSEkNRBBx>p0p6$-Pln`TNq^C|G0t_h%*uoSxb5kalcJ1N%3Jg|Scr^#laX%?x5K|3@ISOO3wJ@(=)i-vcF+F3ZBX|K^z?CbPo96+I(V#NX!Xa&8g6cH)Y;YY*bQCNSV-q8?6&;Xon~@CQ?nW=fJQ(oWY=E0QAp0R$+c^yA3$^VSvQAW+2|7DXk#I;(TJda@GzxRs@`j754GgfnLfx)e6k#x4=rn6=h{heP9nX9E*r4ugU)kC2?2x-pZ*P+9_9u0e-JQtgEbEUV11D@1F~QmAx)FN`ojF2kSPRdZe8XSLGF}u%Ibs6k#Sps-3Ztl7P<`-b@Nhi;L`O{5I9Z9kl&E(vgTORfOpb&t57ZLpS0<83EVZH8mONSt4(J|EnjRW9&!0T1)$R}8Spr`i0u)VjQM~VF=%l#p<YN!kc~aWb1T_-)kARAvis)FCIglF;QVNC8{f?|eAZHt7&2b+T5A@)EABQnaa*kS=dgg2g(00E`s)Cu)zPbk+YdmP4SkY0@`670@Q)rccwA;9im%FQG;w-p&zPZ>vGE*=B-!x3_&k70*?m{++qIrWlSom0+Cu{qmv$)Xt3dvZuO3wY5C))onw4hq5R^wF6V2#>_Tu<2#*PB-EVPyqYb(uQVeBWm@^A&ocnmK{?K5CHmUpd&kgB!smvC3J_xlp}AnPJ%*!;mx?gPCxcEm?rV9vD%7s!AI&`ro(uqRjZ6!RlispnB2<<K>Kq$^v$P3F~LEWhzUK*ntV9)e^}-#k8&Zvc~gQuU>w>S-(2c$aCV3!?2pQWl0+4;xDhVx7M$Ff>V*qSY*i?g|Cqd&0^f2!3ajsPJ&!b$r}#($#8XN3LD2@Ec$olD({V28d9q2(39qQ$5qDX`fkhO8ZFm~313A4SM~l^wh;qm*)5{gjMAv6lm%`XO|->NDbOQ_$5zgdW9uvj?1JCE0bA8Al$?n$_)5&$R;{gI$?d}Uddi$u_r_3bJ(OHM5C21G(y3hm)us_gdF9fJt>9|s@(Sz_>f(&lNCB+uze57#`t=cZCrxu_1(H-A7b`|(eyY5#f_Xjsh!u^FOuo9cQ@gXWOO+uE+Ql|KW0I%yhUE$P^NP+p18C$UXa#qnDAUK+L1-`&_xWe@^uCh_QjN<F1G7!c%TC344Nqb-+M1(EB(<u2vF7`(!Im+p;uNa7VbQsY!D?0Ya>WM7r@OKf%#fx~e`9C*x&mlfMCV|X!)d%1&s+}^`bv@&Sw#3Fd0uZwrS{@*5w3X*5zC>qERiLNU0eFOTCGk#CX&RU<YzBBi3|vjr86g6#}?~)Gic6ELgNG8Y;JFDKA)RwU_XHGJG%rQhf*8TyF6G=B0F5X;bx1bc+(fM6YFEOsxUO95haNQ2vvJ3o&xHgz{4d!8vJJdBG>oK6Q~lf$`Ne>^jSnFqjm(voDmc2>JtPKiZUcmK-IRH&tT82%~rL-`MF@TENGtZyfX7_lqWoZZg@xZU);|!m#E|H?ZeZn8@gD~Tg6Jcqvn8^k{K1J-u_xu_qGcQvXd+P*Xo+^C)OM6JefDbd<>pXm!o~q@iX$ntDS<UBE>;4W~w8Ek-A0ZV)Bci6WiBMfCq`NoqcCqiGnlgPD0?*1sHK#o##kV$7rH`hJCD?L|e;onmjJ#iE&!`3Ga?Mdu+|QpxHwwHBFdRvD;?Ha9P}Z^?|Y*?Xf@RsWRB9=U^S%BfG>^cG%Hxb}n5BF7iLEQ_rYV|1&~Kg}&U;m{=yawm}_$t1CT)H#00*J-<N<V79a4^w2ev=34i=9@lh3zO2*fPHa=yJmXYZ-d4eNkN3BN12Lk&ghEJb8d94HQL268gp2P_yCe<ctgyb&P8;X%&`|94zF`Ati{hzya>ij1R;dq0*LBlqu3-D`8zZI?{|&r?vUbB9hC)8_V~?D9!eetfo%H(0x;amJ=`ptBNqg^WVQqU}P?oW<k|P8;C9pe+bw&{vk11hkx6?cYe_R&|KK@g7W+68|Sd9+rSVm<Pt!<s;Z@k+&8?wsTLv5iB_<^gP&Q42V0$7hN=3TbYQURFPNzK{oHCIbv)zW+&`&XC-aBXvPTF4+S&)~SQ!&G|-LRTRWGl3%q^OaQ747ac?t#eJ+(kySMYKIEGIcw*|V+G^9Vh<l(?QUuH&myi}?tnwN+_^-5F4LcXPk&wzS1gahylOV*3PB-19yQW9C*lfgJBPEgLi(?AZ>wiD>fPmQ<awXBBVpb!O?vc8C;;JaG^}KZFpKJ_rjuFYlIDaL(z>yWSspjZbdF;Z!`j6PuaU_sG_ix=wc#&ogCIEI@!B(R#A2+fDYa^^7=FOvsc5HIV-L4}9%V^sa=(1{wO0eOn*#-c#JG5{Ow+6hMIe?OohXE#H!Rb&s-A?rH}XJBI>)nGx@RYVD)7+hp#r}OdXCN&7|k|2&8=VE5Vk9}NVy-_z*lPZEw$@A60gc*0d*3dZ|eYNtoljirN4al?i+!H);!PtbQphcV9?k@o5U!7wXoNo?{xlO4mU^UvyWF)Ebib$vMkFsHS$fRXmf5^s75$D>uGPKVth0{bH0}hKD3-EYpUOQo%mesYQsPpmXB7|s_h*#Rc{LP=IJQZDER3xYzx}5B=Br_^Hl+Q`aw*GVWSpivAfMPa>PWsUDsw~m9LV~w%0dzPNiT4oG-Im)zoCwYO>irUwg8bm$l(Fq^G<C%KNEOnXvj`s!>|3odUsf!#RgqW$4el>!@?O-AXOa)){tM!JuyWp;bn8E#xovZhx?!j3}D^QgHEN&@g>lhb9T?B#j0|h6KtwLRRJWrK`<4qq740B~8yyWnp%)n-&uKQLIm-i+e*g#2aR~!h>awl`<_WxWaSkqx@ZE`ay%KkQL~4NiL{0ViyGki%nho1hL2$OuDr~iK=d?`Fd$#(1)I?H;xx-I4ycUS^cJVO*MdyhY52eQY`l{aowJ3J)TriO~>o;TzzUz5I~O~vuFWv-Z^NEm&6JsQf^H4y+?d6zmn?QXx@8@;1x5ejjxp^ZkVgveMix11LCy`_d(t^M!hYbpR-Z3ind6LHFgVuxh;TpayOdXa-aiWtL7O?D;#vI09T#FQd~T=yv@lP^O_Kr0%0=~P(#<|*0-rzL}C|GYj`QUjs7x}s^0U{ne7TYkn62lO-W1#We=i!m9l|4fB8HNaRrJIr)Orkn#XfOJ{)C|9=H@)(x}s7!OWydV{#N~u6;HL=GcNb`#=sMCtI>BTKN_(5Jvl+rm@xyRj8QLo_Sm=BKj%LU&^*vO3um;!;nY9g%;gyRcagbXFXO2Hk?FbRcS=(<*GPicNmtfJW?x0$gXXQ>h}1h+ZgY)juW=ddsOQf?K-{nD!IeqDnttC!Bi=5I;JnQU)4s+rFGp{iRn}>9Z6cpu8=*7Wbzy@M+$Ls9Zft1Q~q`=*&+%Nn!ViO@<L$&Doqq-D8_+2M{aXERx)NPOXaIs&+|CB3p~$kA9*P->!gCrm%NWFyJJw@!(TPSHi?s^v)Xlnl>=Mem{Yq(kwvSqZD`6))h4S~-<B)O<66;FR%=6DVLUy*^=y@)%yjjv7P4A{*Han0)YN4xUV@g*wH2CfSBTsx$Fvo_LCzp&uhjk<{d$E|b-5^2IT%?Krs}1rdNK+8gb)0F&(LFEo`dru=LbC;tkqfmlxt@ZN;^>J5LKgsE~?)|OGQ3U9r|;Ly`SjM6_K9Ju&<qE)XxU5Cg739+GX8kz>LBN&<IN8RNqI2SXjn5NK}m0%USy@Y}(VypYHh2!HU7j@cHw3y^;$Q>)#<FS>cc!?yO^&l?pj;uE?P(2{wO)OJ5c$nNqB>*jAW^a!*%2w>~qg?k{i2tuV>rwAy3SY(HD9BO7r%61{J3t8Jc?WS!64zGP#?pU$kr^PFd7A|m0P4~qmpr42GQ3nAUoVN{&NITs?8&YGTqnIF%ct;*L_y*hXcgf@3<GS%Fl6CBwylNS>ypuYm7HbFLQX`-3k<xq@W;-C9z97)175((f&44pzem_+NyMmFt(bc~~A$~lKQp|oNWLeO%CilK2xUC3cf!}x$vAm*5+P5yg{|6b<5SNQK${`)-rU8oIzV66KLkutbM;YrT^9Y7Pgfc|#qvV6}5lfx;cV4TvpGlaGsP|z(HrwOa+fS@l0uTw_Ch~|A5QyYD!0?|%l(skUDfpe`sM<BJ|22@EDSQW6B$944g^!PKCnx*Vy<ky8_yt}diGJP(1E*3`<-eB=4g-}dSv&8v)z`KrAdEx3V<luFcu-&;^P^RssVX-bPtl0^90#rK@Gzd)3s#>;CbItd!k10p<jDh&|Wm&mEiB$00Hig(%#G-iZ<|V7=cEMzc(uk}abMPPto3*od`gE=3BQjgj%?j&erB$zM0b(=cLa#Z7CbKtR?wSxr_V#TcBqGXp^&PU6lFyo<HHy7qCK(r$@U@VSl{f{0kO}t#_ftx2M2UK6=i5!K0B9RiQ|-e!ySi*_T-^RbJnUXDlpkIqiwiq>vN8&IFm*Z0$!001G-)K(e4i*&1F><YrM7Kq0h?^J&f_SyWPGGfX-G}#VW5+2jsdhrVI52<RZN$u;q1ORjE5i_$n|-9@6o$Q!H3`Z@Efzed)dV;{J(K;c)Z!Vbh6?6d*!6r`1~{XhD|R4uy&OJpW0DZG$<ww2jQ5r8#ja6CYPD;bHk0#O*cOG8^~PCXyGRkKgBEEZQVyVU5_4}KIJpyPBaA@8E|ZTI^`G!Mj}dLhZEv{OsrL=Su-7nNo0#=9SuD*SvAO}S^wupHkUt6Df;cAA_q~)Ykx0|I6ijB5lEc4PAl-*4x?;<?bx4DW~6fD^BD<w>4atvrAWm7MMLT`U&0?GW|uNci{#yObsDnYfQ__u)+J#`uZOfJ(K=;G+rt#D>+WI)4Q;ASd6jINt<EQT$V=eXD=5DCrU<moAnaC?4Y4u4vP~_6{h{2tGjykMA-K@W$<%r_r;ONNVAxOq;w%4Uuz21nFw`*f?ANx|f6B&~9#CW3UqHb7sMuyc!<37m!Cg?<YIfAHG=RJ8Zhw<l8~iyZb>Ydqd+T=a>a9oLx*h!bTi0)g=Wt#mtdwaXwB%{`HD}UIe|dPGhAEP$5z$2-3_r6E%+Hv@bAFkI%blPpQUda>LIK%+&9-~aXmN8hKr-O`JQ8|oHaj0FuKLlOEBQXxk{iGN@2&^BcC4px_BeVFr!Iqe#!aq*rynIu!U4VK6MOSgr=RVUn<Jgv_wV0tJ8zu*%1R2psHEoTLfXO%qSA$(u%wx_b^T>o1@h@|co$jo_o-@O)AVQAeK`_8vvy_~Rqa(YrYDmfUGa3PI-%U@Ac|}R4GV<A+Y7_t5;Uf5Tb#Lek8<nCcWUb@s;7eUwuTlU)(j6jbt`)MsF<v*KpHIAmp476Q7Fga3%2EQbDY~>2+g+X0z&THF2g~iES`tl3rUtMGbP=kSvBh=Z<s!9s{c$_0ZB8yLGguT7_wD*@APTY_YPPH(_~(zdp*d#smZrxURHHc>*-ib+zc~*&hS>$WS3Oa8fCfWQ4zybyS~l(0aDi`hgiY@x5H<3hr_bawi}H`-1fHvGAfqI%wV<;HL^=)oHq`c!64##bOFn87ya|lWi^70H~G6)!!BVfup=bSL3(7lG`pl2x;7k^HiGt-&O!fv4Zbo(B40@CN0(q$iZ+UZkuWOd?nAnNO4_8k(#l+e;h3exDymeshrNZ-`kqYj%l39q(OtW)RSk|ef2sER_0wa>0L_j`5mP}m&OUJ#+Z+@lR518ew53{n+$I%ITbJ|YApbE?^-un7ys6K9A46$CvT$cTges&00pJ}#PbC|iUp}bS;ytyTV3Lj-_MY<hrishg2Rbx?N0*l{$NKjEyv6~WLiat%VJ<R*e<-`8pb>uZ-~Rc@H~;bDZ~gTzzWzu5=U@ISt*5^F)8GB%2k+9WX0yuAvdGD<72MlxqdLA4D^{ggDq}(M2&R8pzp7KzA;*<RnLWnNepuP(`qMI=;?6rh*o>(%U-c{<aOUIwWZ}K1-!&n9WhupSV9wS6LQVI=rXG-GMFI3tQ;NZ@7aiW>`+wee?ba*nxd3r<wufCf4*I(pPe;pYn%`LNzJ>y`{z1GanYvOk=8&~GS4!#SVWgYEjWOmXmW{^t=K71stAsxnn;V}GTNNjq6(}T=)?!?v1~JNIlRCP=36}E0avcFFw{vb!Gxm9@P@F?@gj15hZGVAabuS##KPMM<HojoQd}pZyej9XmfJ9kP3zpbTvZOV5jihYTqO|_@P+CSbpK_SC9aK1o<-W%reaUrdT<#gYRc#N-nt8*uotM;AEQwdbl^y8KI=|W~vqQ%#qynGUc<RG+EJ-%jt?Iy?a_LCrl$!zObFO4#B^g;U=;92Tjj)1|+*>z=N_tjP`82hnDd^E%Q`w%E+GF#f9^5dAs}~&~LUW;Cs!|(`p8_F>65yI6!myM4iYwIc66P&z5|>f<%Imkb2sz>t7btx47xaBQysjO^em8yI@wDs5_8ib&tQOQ5d{HR##VAAev^htZVg-F>^Ac1|4%9tz;FojvhH_(8bw&A>)z;Nc@lM|2E!?s%X+^ep!AxDRd=J(DvC(d^Z+!uB-63bj?<C!~Q&L@S?QD`8lS!tM9)a4S8Ls%o%gP_0b@vI$b>`Z{*&aHy?7<ej#5tw3taP0#U$M$;3uRFx_E+xq>{$RIWyFBxPTeBuk%ND6nJ(Q+&}%vwu-#M_FF2d4pC@_5J*{o&Q|97-em7<gS8|q<cc$%3HiJv}7*_z#kDNRoxZeb|{Xw<qE$1N@c*YLjFj)`H$w>c6hsrSIy+9#&4f^9G0nEq3>%(bO__x<Lc1}jR;|MqZn)Q%>yiHqzX08>@_Myah>V?$m{~DH2yYg>w5ml=67qPfhT8NhF{LA!9u2O!JtdW4-WorIgSttLt2-oAt<?>_a23Hm+9Zi{+@)JumROM2QHck6ez>l`y5BBkOJ7f1G2TMDG>u&hDH(vSDy2{su{J$bZJ7CIj&-NDTDV()Ne#D8sl&h09HKFv67M;DaLTZJ*BJto0v2zmUh%w4}5PHSe6Ej~%S8R!cop8eRm2W*8j&`cV;hf2#PT3o8SPWDqT$Bu_vp;)L)~s<EE3iQ_1<d!LmZt+2FyspckGZ4XT!mMo=AFxREHX467niq+*igdPb>~*Cl6WkcICm{b3$VR)$%}Q@awgh6IIY6g+R7BXFbneSXFV@5Llr_k8@$27E1TmG!awPgA0{(K5iZKLx3<IW2+A;VbD}F;C>0=NX8bb=e(j5wE^l6v)dwEkI7(yOx;_xAo`TX#i1{*d$iey+&Iw)ofBW8rd*ROK8{7BpukSq942#<sHE(%*zgFLJ{p6pb6`9!0=P$o-Wyj_(;47lpe`ycM7llo?eRDrG8dBg^^vt~tN47gGWLa5+eAi8V*5B{#(!^0)d%OGe&*3^fIIHWlhccTlbS`(~044;1o^8G~`4?0hKx05iNvf$_^N?((@vZvoy(O?_T1;Jb@JbQz;;H_{`}!@p!}=Ajp^5ljO_KfPyebV}sjoI*mUM|Zu|9)Ab*&iDT-3T46VV1MUY@(x#yG20FJ(GkVCK$D=L;j4jRAg{3=CsTJ#Qb(%y8;Hm-|za6UsemRv&pHOtoi&-MBrQOxv><ij(BP@uXwBHyiQwR3Ut*#}+ltQyh-3W20((Q?1-I-N@0ZSqT2A%HznjBUUj;YKbcJ$=mjntg&?&s4Q-8f>KJ?vlq_i2<{_h*Z<k9I@c7q?_r8H!66Gj+q7?2D4z$OhAv}I!grPT#1pC;>IFv3-I*h@TNZdL|KqFfhIGh@ho}V3w(NVpaMN5Vh4e4xx<fa>R_@^9YmbOA%Z!_!ID+utn5^l=64HD2pf!hker5xfce*#L4)(kdeOay4a3-}vlVKr={-WjRD(okHQ*5P#5TrxTX`YV+1kZbM^0XTc`TS<Va+YhI$$6igZ8%1MHzP5usEC$pQ`k;Q>5?P%eCDepa<`Y<3s~02-0|7tXs8tNr;`cAp>vRnOL)&icU!@kEzqk8OTL$D+<neYPwMu+G7g3exJX%*){skWYl$7X%8jQ{-}sjz@;5_Tw6dGjzZeet)N8`7&*Jw;h(rg(*@<_=PD}MOSwb0acTuNSd-#^!4YuJ?M$EG~mN7vWgAKXaGeNOY8Hf$Q7H1b-mU-YdxytzH7-Y%Tc;3cv=)gbqoDKPw-)_nBn_GD+T~jM8zARfyfM>mp(GHS$Kt~30;wGBTz-2xPR)4W$%C*2Fb7dGHq87b$#ZmsCxo)J;bJ{HD4=yA{^iiMM>E?u8?2EZ6vTJZxbkUc>RVP|WpWCI`rw#z5F*j2GoUhXNRZy$6xn1!IC?bct15_dC&KWu`rqHf6gc>yS&7kJ37(tz7tTJ$ZRwPBa7O7(VEURv|lqSsm+-|6n><;fr)n=>wp>#K=(I<+}B~luhtApK}f@u4Bp<CzR{+~t+&@Kr7?{*VW?-c5XA=vbJ=(0D4H7}8p-sAaM42F|*m}@~v{nz#SsSBwQRdlmubf4$bpJ9T+hO)W=xfxF+!BiQ52xI<YlNHZ<U3(2H!F311lR!h?&}(s>VdeKqX#x6g&3Ykwqjd!%>4M(f+9O<cLYu=v#jxPp>{rf%jmckK*{V{oQC%I%;lyp|wsNnN-O3G#%8TZ#t*evl9X!w4RHBk$N&KUm^O&98#>0-e5>O_s&>u{pKPAMlfmo^zOnGsWa8vbK;+0~I$<&Uf4{QYmZl0qy*UYuJSsqkJ=b7~{5hc*ue0vh~YT>=*<tv>|NICBY^an3q9F3sp<WMV<<<a0bXONs~L?U@;bJqz9>I^xBEv0HeB&KGMfmU-NPRZ2TpiNmP=Hq^&S%6=zNU0QQ44HAR=Ng%*;U!!&%z!P+3C%BQggCm58`ufhCc=if9A=HHHl2p)gLz5txC4Cdc2qU@CP&Q$W4bOZz|G-DFF{QgXXZ8#4i$`dQw|}R0CSe4fnz>~45ew?ARJA<GhX<%wJ+s}Wou$FzBcYrYocS~6l%FXE!_U_8;}0#!|y!${-d9K^xj85{_wXS{q~1{_~=`Y{`jMJKYD-11QJyPK8QWQ?(yXogXn|P3rW$u^m@3`C!XS3bUx{6S}%2Y#%x8W=Q4wrHH#vtk}9_w@0XP(a{3ZU3S{uq^uDPsXbP^<HeJ?5+49xaXAUk0^1RUTB15m^O6Mfybh~uNFa$zFkc|`wB;YirFKM70u5rUHa-#Ro{gL3!Gg}q{Lrc?{PgX5yDmZ0d3H`C@p+WRALpo;Wq}}1XsrkbxdX;j|XPj4M%z9@MUlGNR@Qips3%5@FLfiZi8iz4b1+?tNg)%Pzg>|qxe#NjX22>-8H-J;dTkt5YTGxVax>?stozpSG^-eeSHY<xd?hU5kl_KKg7-MHj2?mm4;1WVgjKG-|NLa9iONxUPSTciM#QW?Bp>!13lb3tY6TQv~Pb0q)tPAjD-7C6fxrYG=Sf{gD2S9Z**^*9Cr{!Ed<ZE+S=F`KeV-KgXo=%-B_H-Io9v{@3+P)Ya645jXg7&!(-Wmx98LHw3k?Z?(f+f40D}y7C$vBfulYRW<E60f}09ojUKJ`gNDwT+n(7EAELOc~7n8WbnfBO9=Klp<uKmFO0?|kdYkG}Wwe|eXP-IH(r)6XCN>BrytlTY9O{ZIbzcRu;!zvs7q@bHsA{>vwS{<oie@Uti1{Pw3m`RSAY^0%M9|5x<w<M00GkH7nupMLxOCx7&H_Otz8-}&SRfA-}4fBE$HzF(H=!k_>2$4|cfr%(RuZ=d|=yA1ljfBVVr{nID^@ZC>;@{=dO^UWte`YFBr@DC|wHJFVbfAGUOB3K_k{6?_*=<jB@L34>$tSY}Te)L1U$R53i#n^rP!S|A2if_&$(>ltK&0_J|<*#%;c<)@UTejT<_R*9Vq>M*~$G8@>uUxHzKsUchLCQ)MCmtd*d*GkKer+j((}lT$3KVuNmP85a0KxRxn^8iMUNT61)RThhqGkbG^MaJ6Ym}i1R>g=(+gx2$4}nz`y<J!>s@4vruKdcg>oYj#SDf3|aBQ!peb&$G4Be~>d@7y0O)Jx_JVt6~5s;K-Xoa)%%+uR&zY3b1=9x|_+(m!&wE1#WX1<_S1g^g3MYE6xlgse=)1%Z-kOC|Afyd;g;cM{#3bbUjxCH{6A}!O+!}$b;;Pk;XOV&BN1wSq1@!B)@w$WBRdT@<jy!_xA(CB65hShdlH|B(sd(hcHRB1T}#poN1mgK;5%IA~vz187wHa@?F<m?ZS6a9g3T7hIMfABV01p08Uq0{jJRhhA&UI#sVfP72)q?0u_u17Pn1mQg~V{~jBVN#I=D7lZl_23up{f}S##b5pWpTGU&hu_5kpua!<^Y4HB`~UmN@BR3}g0hzd`{PGHp@Zx3!#@I{ef+_D@dNYf(cevRW<CDkua5#Q+YHMy?Tt?*P9sj~w24jvXhXgaMx$dFih1rM>NzJu884^(S<QfCxWU`~gT5TX3`{5G!7sl52cN$G^-tdY`h!7#4m!~H7q52pWCZvhs!YID?HO*xq?C3P2-X0>qOxWZS5-kFl7zKJ6|iY5a`mk>KMR|v9p1(nWs*f(f183vV9VxhD`*C`1hTInfRV0XB{Ph)aL`*;0?TiEAOzF{Z_8!bD5;Kn8g7|Ww9tZg@VS8K$Si0qwv5jmGdt>9r}s}Zo5rtJ@=3x*j0*~I|BPJ8ma^GZS60+HWWv07k;{adp4h*)t#})oozsxeffz~Apsc)6%Cn(s&OGf>TXICA2dZRiY0sRn3fzD)?q}ot8auxF$NTO&3x?*NA5#%-L_3r*)dHdvskqC>(L1`A_t<-T80uLs?~z{HMA4|N*SwfkovU5UgL8|IeB};w)#7qgPE4wIlo(Z<pD3{6%96656jyQ6?N!2ZS{BQ=?`>h~6mGx?R;oq)2_oHK%_moV=w3m8$0S-4IWA#In&N^t#yA4l5sR3<K-3aqspI5^Cia{pLgMt=9%(BX07@!WSJSBzVQ5Y%S!gNKqCYlGCzq$-*BrF=t|c!gO#OWhl~CaXhy21!5qn;S+b(j1mTwO@LGT7%oP>J2OmnRa87pawp<N{jXHIu+?_d(ZPDW#s%$S?vY{sJt$4mkcA+dA9>5DJ;gg}YAymE;gY=?W9(zHpk*ON>C^2Dt4ne*G}r;rlXbu1Ia(_%k1jnaTpGEsgI3-Fp3zQfxp_ugsb;4QP!4`^r8WpM;7+5@Rmf+P2KGpVA^we3aRILhM@-9g-y8%Eu^GT$_{v*|v&F%xw$<wR8GQj?Gk7N+L5ezSK?ts2jpTr^;6iNg^m@z8W%QyV99lW;?n^ZtC}75UnX`s_rb`g;3@4)x=;{P^>)VZh<yU)l*gn~^0$ul2+C&8zKB{+--1osXA0_*jDOUNeD0FLqoAPG)iiaAJ}w%JHVB6*(8mQ-kjp<n_}$!~AT<fK(C}{fW{g8xoVcIAYf!z85^9+f=`L^Uj;MUcXJ}tEfxtVV^Is^?_&495hM!Z}axc;i=Ai7Wx)0Ntgt|WPawY>lKwHg|R8(<-fjo(VX4+`i$be3Lx_%$Xv5tK2CCH*I&lhK+<H9(t)$%Ng4CA3MRx~by+^1Z#mBS*0Wk0AWvrN#N5ak8EWmQ4>Z3osSJJJDs`N0o}9P=t-85{tCHc$MznF6beczp@EXq4u{qztY`UbKXcm9f#=9Xg78}xt*?TVqOODo^!QP3~uz?&{l*<c8p(gjCOkT+PHKSN~vsWFOE4+;9g4xRV+4|%7z40ZPk{gMyV8_%|QkJ%CI(Ee{-h_q=^{hr>9a3g7H<wWhXr8k|qZh7IGh~^060`N|Ob5l9mw(XIwyWWDL|OlUeXjE5$*YtTpHg9Ii`LIJ%&JrQ#!Qa=$iKF#vuFFuoR89y1gL=7Xx|R2J0gkzdmmt~6VVmY9yu2V*37}FU81`)RUz$mrtE;B%W8x}%z+}7++K5s|G^#dK#4JomvCy+8ukEV$^@-4^mUGLJ4Y9N+TL;t3JPh$(^lZidr_9ZF*e#8tT;}IeW`j}v#}$0$As5!ATQZX1m-NU#TwKmC8?$qyxa->A>kt^=}5{;zAcupoIbQODB37hrV+*{8GTV2b9aIc)=ztnB!|&qOJVbUQnTeW;-vq5qP=3uKs>HAj&T*UsT1_B0%(gjYvHP2oyu)SV#Dt41kV|JQl41KpJ}QfXjzcC#iPvFGgm+o<$~d30jUI0WObuZEhpgc`b;~~;n7Xn6_uNE;I6bImn=xwOQ$X|2k$dmU5q={Yfeq)vXJUnXEYc+i6E)Gb<*4#R-^YgbuP-=nv-xgn4S-p(|LyU$eiAt1a=T7D;9(tifjp*bC0`~!_SbMBG(3Zi_nsL(ByT+e4hz)Nr_X7YUcuHG(R;{+oE|r6F`|s2avolSdh=g2}C9YCxN@<ARF~M$A+FPv#|t{6>*xkeDFC}p6Ldizz_ra#i3eCOV5M`mQeJ_n*v^h2PVVzmLV}AT_8c>6+;s$rVX3}7fdXfQ0A~mFz~TNt)FK3yx%7kvkB!H_(GCK*mTmq4Ut!JPtaxIR_>xSzLMqRZ~;LHjAhEAC73L}pQ@27%#h>c^*8xe8_JV<0KqOJ#Ui96en-r?WbcaUGBfD3$+;Vj_b}jaDB0vF;Ono6O=C4So?%bsH7F)fVd0QZ2XH@fISk*S73L=ubDSLF81|prm0(6Ni$mAMOm5Wt3JIQ(Czp-<1r3pYXGRfGnVkZa($74*&Y7XFvnbkcbUZDU%e}Kt0n%PxzEp4}Q&;V2b}&d6r(fhE<#+<vdt3WNmmMT@j_dbVztX$?#aC~9g$}@~8(s7=jM}>}geAHIo&chxGDX)_&7`5v0pH%h?nuCpm!KTugMl@tV9!M*!T}>w;=#-JSCm;Oor}?6R=A$3aw@nOmV;1J=AotUUVZ(>S9&-9Bl2N8zi)r>U+qkkhIbT4li}LRTfuA4?mVb>K74xXb|pSlUiEd#(!6!MX>(o-_9;rWsJ#*=CPR2EiE>1vyoQ3xshk1wnJJe*rD#T~imNSBOSqLpMl6Vwhr;6?erCet|1Z7QeaGF?_`dZ>R+8j#d9rI#9dTX*hUVYASSiv{g<6?*UL(e(-rCGd`ytIG*Bm~xCyv-EU(PJI-Vv+OtyebQy7THPi8U=st5wkN_sENv_;OclZ3X(%ELi1*Sf;i*PrK`}Rd>;invd9^L+(&&(lHhvUv_4k6pIl2v023TyLH+vIL^F{F5F1cbUA0y8j34jnv)CFGJ!joc7Hf*=wM5;EP@CtQN$NP2ACT6<&u3+hv6C&&$RB8vQ&0O-I~+(uzCOuC@}GP*fpuhxklgPph!t_^nB2F)|?2IdQh&B#*+Z0K8LGI22fT&^W}w<ZZhAq4A0i{c&OAB`uy1vz}v+swQl&j&^jNJxG%XTw~q@p-7zV3lR+;YGXKnZ3>x^M;W4baW3Oy!(7o}8V}OR_+7)c>oH`tI$OgdnOpAQ!m$a$yX~4xnv24^VO^Dd_l`Jy-H8dIEmmHC+WVw}hs97!z!ZKe1g>>a@9rL213Qz}{CU^iww)A-r?HLczD&X6h%15AUD1~>AdK3=gYc<7V3)y6n`EnvgNID8+diBhNxfS)q<A8!TbUEF)_pBb(r3FECdCoOPWP`Vj=)vc;wOy>CgUIhwQt(@LBw;=j%3@-F@Et%CQErgiM34)aDnO>$3fa-wrILQ6eiu8_uo7Y(x*`D2LSfDd+Yw=zd>HMa+iMq;xk&4N6z`3NbWxuUx83R7!L@*RlVc&!4!`ZR?&PeKK{nukSAt|jyW)f;O7lpMv>0dk)<#DiKbM^fGSPJPs%)Zi)|W#%7@?X~D~*Qfz3~e1%4<M~lqosNvTo?o;h`>?)zzJ%sq?o+1zZ=ssQG-DEe}&T5uPD3WiaZnbUH?9!BhKXy_U`bQj4AnRH~U>ekK$pKoyXRsumB@s6deD*NA`BYSXIP@La0O-BqVt&+n?0tCO2ms9T8u?DuMqJ2~6TiYp;6S2MW47wh&4JHR3ecAdK;N>I$U?|I+=#nJ;kyLlaqO3!Q<z6}K`o7J?SndjOmcYx1s_CrNRjbYrMsUsX`Hi9XI-!+s<7*2vI#@T?*U%V=FLUp$JqVz!&Pa5qPuY#y?oR`9;{lB?$#ugy9Ejd(Ecb%aqFXg&4WeLd8Ny;`dd&U{x630%uGB>EIPbW3*Ck9tVh;C~N5~Y47P~lDl@WbUGCRa+0IE6zD(l8-54KAgJA~51Y8jcJN=FpS{>Q0cEC?&enliL@6O}%Z#TmYj{thp?nk6+mi*I80O__pR6G;*z~3DRo#cPgiBU@MvGJmZtF0H-oA#oL$wEH)Q)Jp~hEjJXy}g@{63r<5DxB&PdzZPBXhB5CZFzrih<LSXjQF-80XV!m&V1YvRuF>+w;@mNx3dpBjeQopoU(DLD%v2uaY<-lC>SnG<Xi#bha6iC+TEt1|P^973J8EZfM54C&>?Q4p4hLLbLqHU_)0Ye2S_!Fi26DS3bLpP<eaKKWmrGqI%i-ykzUwQn&&*<_NM5BmaO7Cv{YW&4GdF|F64wQug?816*WBmBxw`Y)g9n3rlpg(vCTjtTb>_+|93;oV$eDYE-c=Qi6laC+%1wI(_q~$K4Cb>|U0&3~YckjLtfRk{mSngs6P#5W=A0me4aOt#|A0brW@x#C8D4ai=k@E$L%_TO+^xx9|>K@#9{K4PMf<qv}gnYx@jwnT=Zp;8b4u(s#vHSSpe}WdLTv!kOnvvdp{P2IGiHE=IcM=4I^fCxn{T@#qPrC(LbFl4fwJ&|Hx|!SKNB=mbS^KHp2xn$N4^NEc;PD4PfLIzoe)zsDWH2C{xb;e~_xRy=1|_f|L;aqUy=U_1-Tcy0to-<+ANMIj@`%0Ze-`Wm_BW^(^!D+?zX5K*-@YAuMf~pPDD=G;W0qoU%z-!)Rt)D2fd1$`uJh=<c|Jo0RiR)QBZE#d!mWO=G`{~p53Ckg7JQ+cD?ph6{GQQO0-3-ihmcyu*$b`mwFTjW;6FsWe51_+TFE~FP+GPJ!E4;kgW${gzMvv5*5dbdGjk7Umpus9*VhC7yS4hCFc(r?F2V^)rVVcMdtZ1E93SR{`XrFwuJmHw<+x<t<nn=i<tSy$t6c8En+pz=$&-k4^5|zV2bgrezU<L^u@aZ?w|?ul^wf4tofh4FmUySdX3p87d#Nc^!&c>u5&^e&1C&Sh)*_gE^y9}5zc<@ETeF3P4AspVjZoR|Jj6YAuim=*8@)R>-+2Ab-S(8vC7S?8V&_YGy&4vW0*gk(1p6BP79=~4J?@wM{Z+Sb%2zoeRvQh<oE%$3)lIKh+xN{YD~f8hK~*}RDm_glcL%RNW85g($4hr>>w0>3YqQ44@RvoqKHHvsL$o#0h{e~n)4<?+Bc-4Z_3BfnMX{5KlNwG{I<Dz91@eVruOaxelfRheC=uTP4Sv1#inSt}<TXWOkS;E4Oz3EZjR<4JbA`E~$`D}8*e1qI2coI*tcRk}6Ia@wwCiW_Z_bN;R;C^SnAKZ+XfCGv7)e8NgJ+I^0pEc^*>2<_XtztP8Vi{d!8GPf(sU;heb2`xd8lQJ@VzG9)ENd+dPB>|VFQ{<CQrEC%2Ceq+_!Igo|kdA!pFFdpcLtd`E3`>Sts!h4y(KnoeALEhf0-N)=s8CRDn)Og*a$Q73*8MG0O@W4afGFISEZ_QRvL(N8=36S)V1VkBUX^2-LFIZ0}8y-9~s(m%mwz-tf(LUhZ*njFhp8CSC(k*p$E~95*ddRme|;%?=Zqfp#dux2qBYt+v^zrPB=5^dYyI6>oXtFpPq&jVxN8Wo@n6M#u3_?od}--YVQ#)!?T6rFM(g!kN@wYW`b29-GaH_Fe^9AmawmyqD}(_;eLFJd<{TSegJY#;dFYZCM3qc@`>NMn2OZu5v=C%7Cu=V%5Q2Re7dXL|s*wcF&uwi=%8;rf#cA+*OkYt|pV$3LEiL=kKaAhCf9<uhTkImEWtJS@5**w|Y!CH59Js*xk*g;&P0hf8uek`P9p=23Qq-#?Ewd`1(O-tDFdp{~+H+1}4oMaFVM59kBJTr+dubN@C{*U290^FZjJ~kBzWBV|m#n^BtEWyFapU?8>i<Dj`oY<v0cwhAkWA?ogXKo0xXFrc^@~2#lehni%alf9q>^8sYl7kjK!5W_2lzaDRg@<?bmH4giRqS=7Ryu&e40m77C~ObRroOBiAbkT%Uy1Y&I^M0HkHGTO>=m6-0jhxFrB8%L9QH@rn@p)-ohL-yfpk+aX?N?^&n0u}-^6_$k(@a^O{{ty*e3o(sgNn8g57aB-+=#6#!z%p{G10-_fV^^{OtScEr?&KkP;w@9Yv4$i;+Y%g+dOAHBLIP%ku3|tvim>|bvshdWjHq7U+NF|MHmw8qxxQPgU5gYpx~;0U%e52F?Bwdoq8OEBc}3oli!_`ck!j3n_>?v@YH{5JCf1DbYW)xu5V1?AK%t1mQ=x!6Sl9T+YA7$B4n-{+>l7$?{;nXDW*su3Gwbze>v~Dn<E~KXr$OFwivnw9TXMvlLQHjEpJ~gSw}a9Fm4;G|ZPMd{{~_2}!wO@C80dU|ZgN^0;(-c9%x2^Qor(g^&JiHq?SRygNceAUY{OKrfzc)``lWyy1J8tgLRX3&7%Ij}2^z8Wb;Vdr)H<^?y&h*p>M2V@p1;{*Nb))z-xN6sBrOYbYbN(}qq+8f0ZJ9pUj',
 'diagnostics_snapshot.py': 'c-qZeTaP2hb$*{;Q63>^_KL&V*?4V%a{<=7BY0z7R=cYR@nq0yHj89yn%#6?W`>#&NT3HJ4*{}BjKDC2NEQS|c8u6jf*`vOc^KLMWAaP#ol{la)!ig#Bnv_Y*k!XWr%s)7>fGuyH#ax$r_#@*@I^NDqo^yA(I^UIDTbjxj*~181M%aBkHjcVrWh2Y{#hi_BJPSX4x(Zh#$z$_(>(OKIqPk1Zf=CrS(4_$PsixW<OWX;k|>fv9wu=n^nm+G5$7`PilH3&MU)T2AUBhP>{QJg0=+z(N;AoP?+W_={UnxV$|Rpg<~dBtC;o`SL2oK^AG3Ya^In$3YK@tnPcXtP@d#hdpjqVSqa>Z0@7X*vkF>sdem|Uz!bomxY+x?}G#z>bzC)0^;>=HDP%3l#8v=iMI#&<$m&&|IV@9SICH^quIXwoE%k#V=;~*hA?>j|4+Iq!tU7jn?13Ak@=glXFX_|nDf9^+x<Y!m-8TD6-7zBF$Y$oGjXXIS)DzQJ6i@vzP;DzHhak^ME_Y*DOP&g8Ck~0~5@;uD)tmC$Jcy!DyX>yi{ee>-s%qMDI0>s*TrJZCs)}L1Jzzz`?(*a2i%W4=4i3XWMG&*W@SL|3AXdG#95B=e+)97q*#r`9V4dE;{5j59r&Cq<QxY1&U{kNVm7iyag4rzK4TADvNEZ$(A%dA6<T`>w}G&Ian1Odszr1u1ykH-(+?8Nen@ZWbd%(FDaNyH$4>OH_Ye1{uKYBEGZD%)4UyVz5n`kBv1dfy45EmB#7*5i)}WCws$2eA$)62}NrOF*p|=;>UX$a!BKieWLGWt~%^Cl-yfB9)$>1!1`Vsvl+2CC-Lf7-zX32ht1#d$Jsw1zT}lAu&*J&r&XwVue+^YMe%~2YjZGW?BTD)T;KssPRIPsW+B6Pwu!qmSovca=ev=V~|EM3u*&A@Umh$_0xGLDS&32oV5;>Je6@yN5wjQ7%aAh6D8x!p@%R&O#+{lYL7=a78uQaXlFifM0bn+8a(zz+XUtvhnsNb_V<Ou^Bm#F!>S)gH08zg7`6t^v8@8sa;>v};SBsN%$&Y(ir7E(!^j^*639B3B&qpFlgs~rEeS5a5ZIl_Cs!{%48=I~V-Z}v_{=Bha*!k^VtV!B7vZirQ4_LaE>d6QS1+Cg;>%}OFMdwXzZMx9#95M_K&qTYMG8<t;dr#QqcJaOX!?#Ux_aSI?{JnT<5XtLXt{<%X4VS!G&EF6>N)~U?y5=Q^OCqnCO4Mp)>$}|FtBV*)hD!OQlx<t4<BSA!MPBAkS0L>@Ku=ylkK2LqwUA?CxwKv-hLt@8HdHRrvV?vW74Y{Vg^etiIYAWhrI7dj{RU>@+4z9!5JJ!a!cyTa3?TVfy|TKkGwS$EW2zXm_{!PAbMKpDl$}(R|ns>Syk^+2P+<ajRS#MjpJ|w9swAwR-K1T28mjp+DQhF92xnuOb$KWr?*8aYuUUGbcNU|-Jl*hFkzOEkK*SYEx&&59J_8$ZC=mSDj5C@cn&~@CWHhbS`v+w<%K#kK{B01G6&oD^-Q(*vfaxkVdhy(=`J6K_S@Q8X2C=bi>L*&Y&Uj}9MgRZh3%3IK(}gU+0z{d;<m#74RQ=^4OEh{QblTUkJXs91@8=^>k)BDLDjk3w^-b*Kr-1u4A8wVEFNI~D2&4l8$$<<O4K2~yJ&B6y$``E-~s6GUGb-`wLnF0KldUZSQ~a>V%Z6>6%qRtp)Ei-_n8Mh?!gEZta{b5Ozd0Yd$(y}J_OWIlG5I*LXd#~S_xqX{GmzL0)QfE?Rx5;cd&FN8e7$_+!F9k1(=y$jt(=nKC}?`&=*^@V0UGjl4LE#sO;`9pVbx0)v%@*yY<vIJQJDJEEhfhF3<)n`onLpUVM`47;1d={IgiZldI>S=DW&TO_IxxFo#;i5EEh5(#v1P${vPS&;KV@>-5Co(|aNS?7tP!<wt_x{vy!u<8jYXXjaZUVweCvQyNUdQ`vz-gOCNTM_1@3?QkG;djyG+=_!bqL#qU4?i6ZJoJ|nmh<CzxNFMwk4dp1B18F!zpt)ebpTa?eiyQ?SpUTf^#hmS=ACG0{70dZ<Io&08J-v3vAXyuxPW48^>3$?r=)w$VbG7FU0RnS@RN6rwPRtps6^#a$Ni19z9w6Jf3rW6%e-LMD8_@FKZ~OA8A7*vml9h87NjdA_sC91lZr{cEx+8AgGN7Old#||0EgTeKG%T6KUSXX*b`#Vh1*tR)x`ZEVWFi}}U)czJ?U-n}SLA{A(zPd2MPpMx4oAv5R3WVPXiOhFGHo~0r^C9`Syf#t{2zHfLsEcE!37K}JTJIgebF$@3dBO|^B_q{Kc?DyVf4FZ3p9%&D4v#(QtlW;^i0R?jiY4Hac-%txV2D~hSav-2zWOvIU+$(GwK2X)rON`#Tulvc${_|NDpZ#D+xfmwZ@QUb;Wn~%K=BKS5XLF0}VK?uX|0D|M0=~+mBxp@fdn0gd6?**G1_@=a;{UC*t(-qol5C9C!WBNJ%3EB=&K{W^DoAT@zICd8$F8ZFJcj(Z?{E!nt&mxrEE2oAWSNQ36bAUS#w%$@7_4ef>${M`1oMTk30=-cjU_d3Z4KQ;*$R`o<A#kn_=Zj1FiS=5EcrH&!s#ox31n{+UN4H_zF`zjOC{ycH}~eyzh$4l@n%gMi)La(hO`mo?QWQHNNL;Z1ob(Al-A<z5FAg~N_l;7v<|&Zb$<ESGZ^O;I>9%sl2}R7tdMW@WlusO#g}fHnS_fdn66OUGWr?9d^7;4TF*3^8Fn3^im85O7wdxfp=fFJ>#H3>>>XxW<$gRQZG2B4&`=a91rAXO6L6*R3VMI5bhQ=c?`Z8x~Xb8K@TyyR$X}x-d@a7&G+cr`Q}>CGgsmEjgJ=QrtzEC{cSNolHRYBY!#=`lR7m-75^-de8xnJF8iSdeF@fl>pL~$oXi8b=oqiuuenKy0^BDHXqOwV>U<a7@wD`#fN&gtf!4Ofx9TwEH$Q~q10G9=rbDHbR?F|RNBj}^;sh5lK0F0soKI|?x`b09zc~?@91oMD-ov%R0&Ht|EOH8QqPVH)04@FPcJC>^1j$<W%d-M`qsfIR3_^>(2W^KR`_TwWeq^yt+SfaKndxl6L>#RCl&ymdLzlI9Qv`0If1N$xR`Q(WmruC8*(ikSyFW3z_U@c;(>7Jv6b-nUL=DjhH<6Oq|JV7h}(pa%)r*Q0BR%LT3n{N(r1;X*h+|3MC@D2j)PErY;=eIwk+;6(W{1923dMaJroA@m42o8&X&DSjbF?8v_z6z8>2dxZ?evMxwvjUORe2xF<T3&m13y&n{P`U_Qnd-xJ(yO6Zer~UZ*$6j@heeHHXs}6cpTLS^{*tJ(L3~<f}>JkS(L3I6xiNrZr1w)X=i6S^!HW06-V&NpDw?SCL9}bdBz)y!mpvNzJ<p3%E+I1M>A*c6Ny1<(mdfGLN=6bLtU+{*xGP;RGU&!29{<<R2%O9|J+{K+n+nNho|=iQ*>G&n5^7&$h|y!?(FOzx+7hX#Y1c13-vx;&l<j(GJD1utRrNseI8u4#v0PEPq{hx$cy4L(Ro%t#XZjz0{}HGInDR+qDV!GEwjwce1|?Kikz84>HfmT3Od&TCSH}h|$&j5NCS-j+(Ac0^Mz>IY;J@D07>fBH<HC$`>3}V!!ttsD^w#lYFawut)zRH-?+p&C_u6V2|$71&K!{^ZiY}<U%Wp;K6<SfT(oW-nQzpXz1)^`5aAyWH`SVA+GH2+?}1%{W^~IRuS$3SFo&q=ex7>UAn8>n#gcG!LZ)lMNZd87Xzpk+<s7H0f|)KJnZ5=B%kzmZr}cs1%mf{s7Lm9Xn`aOhvKE-aJW13hh)6^-^U6hJzEg}aomsODBlHDQ2XFVx(#M`0XRggQIwo*ou&S)@5l4AiA?1J9J)YyESFmuJayT}cOJ3OB)`CQ3)EMCmV~6w7kk@k?-n^m&fa8a8R^9T{`EhK3!;BdW`RHB+z@*oF5qBJ9mwp!D5`#aR>Prj7d`PqE8%!ydj^Z`3(JLA?BZJP-&t?H_)rZrp2DKHw>_(Kb#`!Q9AVB*e);V36IOX2&qb+0c7^ePc?=YfI0>&_d@w~qkT3Ior}7U_iClg{*MhkB`+2T1rJTFawq+Yc;=v!l+Z`0+QX><(p92_gazOc644)O*GL*mk`^#U>8I{bad64Df&}4JJe5SEOV?r41Qik^5P*HI8;->|pM<YK_$&(rF;L`w2(ENnj&q5#-Dt_XI?#pMs!VffQV>7vPU?-JVr;Wk=I|qA^wF5`IENos|ib9?BYxIPR`3?|CWgC^0aph&t<y1sWmTi2#_$T_Lts}qvAtO_p%x4rjA;rJR!NUj80TX_FngRypoF<nahh^(q_shroBxibwvJt5(^}G#;dZFWH=%M|WD)iv%cyrrlqTH4T+Fc#4b(a(jIO4B_(|Zqx`8uZybnQ61g7~Ao#3OAvkKOuBqtf>(8Q^G)bds+Mi;S)|F}y?6CE$)m2MQn`9@Ne~t*a^;fYRosOIibC941>?>Y(VP<$!>?HRFJ)o}3muqF%+<A>9W@*}<OHaR*Ov0G@%ZCzrnxfA-|fH%NY;f2Iz=XV7?hm=f4-O<t9SBaci8;of!$eVgr%$qbOgHl;N4$Z?#P#1eH0t!MNjE<j$tdhyTV-EtQM>s<uA1s}(8$Szxa`FX4r2njfj?$59b@H6ghY>9U@-kd9bw_nSSq6Ezr<*s<^ewV8c&|>^ot1w}kuWhcm3*t3RvINCASklz3>ABE@e-|Q|t2HZ~zljV^*tWEkANa-)^})~(`D9TBov5;%%J(>MgtJXv>Ht)U#hCP!KHIqDxqALzq|a#c1$nj%R9gWJRHv7N$sc~}H_#HqhE=Lm?^tK26s*Qr-Afqf<n;XT>C;C-olweIYh}dOZx0`6cshxv0S1$T$KZ1gkg=HXtr@efnkD1YFim2Lhz!3U&GQMadIb>rslw-P3Ne96;cPkF{*bQd6wPt~6{lnpl;}_e&^BO{HbonNt_l;Vgo=v?2~bsUnQ8kly!>y_mPL^5a7Lg=oQN*IxVwZ=+lmkb&+djye2kl}lgodZ67LXj5En6NmnB|-_i7hBiipV|$s5b=+$O-kGa`mhHL91NV?$G3pTb0Yig-Qz0RuFWOd!p2T{1485>5YA({ykN!3flUU%=o$|Ic{0(Ws1V@cYlfN1XhUz;vvUr>R?5TAAu`XjQsP4e0t~C`o!zr4>rtB}K%uiE&d13HzLjPGd77D7Zmw7ckx4#s>Q-d?~9B-#6y4h{u?rK=9_3hKk3Rzr@zT+6i$5Xzgw&gjoCPt1LCG$qa~kJqmjlsw^R9R+?vH;3rT`FaBXl9Qs7Nf57z%f19u1Be)ijX%bVd?#k;VmR)GpFO&~dtIp!>)f8ZlX;)E!c%b}@^1$d$+exg*R1dd_d*)V)7O&J$Z^QC<`!{xZ`%RyB1ICBVk-l^7q0E_7#`FYAr=z}P`%hCYSft3+OHTN!4#><EH|&^!Ky|R3T{QQ&Y{AGJ_wwmXTVqQ-QSQ>2v2LP#%slJC#?i$$CG~X4GCIs-&pdD@6EVt(Kjuu%#bRMyn{u<t^3UczT#_?8oaG=p^>MnPcvGLg&gRo%HXLzJ#IY|GmU+Qw6w;UIe9EbbOxKq=>u&z7-5pgZ9v|sPx@l?Ga+_^;^eDGk60l^-BQ^AmB!87^Pq<XzE2C0g*wpsH<;=Pky!Yd9_9|WeSOpgj5tMSORJlWSv*p-_kGuzmufBHg>EQz|dL95XJ1ftbd2UlhvgFn|j}PxXc>S<9CCV$;s9tyKH0qOE0BW8Qaq2H)?Ma#*_bWoQ-6a52p50f3VJRUZS5@3>uKsm>^EF^BpQi>!x8D?u<>l1CnAG=*^qhPBE-RO(u}#&{f_%d(H@8_Tqg|n@&ZE3VnIxH{dm>GeqyF7x1J+){sUeGHa9_Ue7Oim|S1xw^O(CMx`LMlnN*xZ0A!0A|!Ft}pe|NsCKkD^D{GJOeC7Ep`(GacG>AUXcYfW8AqbLQ>qs?k#3qC-b>&k^I7gq;o=0>*)bZ_0_i`>P!8{-w1-Swj6Rg7BxQbVgsq)rCT-+~r5qCg8q;AfJpUvsl2n%9l`+i^SZtp_w(kGm^h4bUihMic-+XHYl=-E3YgC1SPp9<7L11mG8B;rmkT?c2h)oTRq)=vc6z#5PTMpb5F%qM^H-OCaP?YinfN$W>{-*HWuTUI{2W31=RpoYX(TGVNrJC8|buJoul^+S>Q3He$v5ID8OI)rGl+G9?$VtVx6ly5*bZ@=;2J0!O$paJ0NUsVc<$Gsl9Ut~vF1Rmag^DI^9S9dgXD!Baw_0^0LYJDX(T`3e+Efzj}A;A|bV8Uu?Lwp6a<XvJAgm9|_~YI6Oub@BAix~A+e+otrC*^9dth^3Ra_Lm^EcCt;P9`gE~o2~iv=(Xx3w4t_-L%B(PB$gC`++dY8E0_ygTu4FiGQ1kXcehH<C+WPsoVJo>y%Atv>$_O#m4@@8dn)gR-caBOA+c^JuXT@>WhF4*z5`QeUz6MZioRzv1yo<s>iwgZ<yROfz~fj;N4cI!WAtWocJwGo)NE8Kqh_m>J*s!D7E`=k6DBDcK)gvXkWGjx_28)&jGCQLUpZ!}d(=i(F__{MqH-Ru8LjAqE=1KqYsE57Z}vpUOyI7uXb~to*|~GO*1AMN*SrlP`ZD9gcKI4DUsH0*_&S{8BNfLNaxEUOszm0LS-TMV2zQpp{gwFrv3`TbS*P^~>#{s}op?5nppsWy@5iHXTsErr%}AIypf%0L%R)6jXrP<dNab5^-+S%h)4%W@A3l2X@lzGrnv+luEuDLBKX~}GKG3|7)>Ip77A&Vw*ag+a<uH`Zv`$-#0<{Cc3a#p@AAknVl_;<3pu-WBinZNt>gcPU1|zH)sxo7YWZe=3q<$U0>^e<Om(2OLPgI#BUVW-8%~|P;Vn(lJGdHT;TK%;bF)c<FvvG<yl)d*Lzo@3ErYZ&4Ag*pj6qs|ReF?0tOlz~&2p>7eNVLq`5>^(1wCzz<O@fzTW~x-=`lZYyXmoxjqzMd}8vODD3M{S%Wh!VIB3u-BX%2DWC_FbMTBhrr!ykBuj~~DJ_z54>wG`2{GpCtn(uU?NoFLmd_D=o88x~2xt^d!Y^Cj^kyzoPJO1Nze$5fsr^!u=O%^vk<B*X(sKuUUjF~=`&sQUozj_EgNmt+}K_}#{ts{!#k7Ag$*aFnDp5w^u`tErU1^8J|Q<fzeAuXD4IeJRa3w)(neZ3TO(SolIAh^&6~223Lk#<dLxjg(Pp*k^uwyR^<t=dGGZ39d@cTDl2-uN<a`K~nr=#8~MgYV+*NC+>2OSGo@6YW=S($Bp0EaMHo@-_>Y^1?oo)GAU2Y`<cu5wPmx*b#KHkcD4CVuI7bw&6<=Z>$vt0l3e=^qptYFiHod+eDykE5u=C<8!^$5eIsLj#X%`F{X!loFhf%Q9h;QzYt(Pr^zNl&uAF_uJ)I#c8SuLz{970vmDPHlYt5vH6v(S79cOC`Qsbys*x+&uEmT&F*SlOk2Lnln^;G_#$>xpks@%Q)jv)={>9-2$r7@e3uxD7<ihM2{gF?SHyis3X))-oq`?Wgra)4<%UhpfBkY$fAz&wv%xA#1vzvnr9{bnLDW8?n-H^}Do',
 'export_snapshot.py': 'c-oCtOK%%D5WWl0e_+)Exv;c0j3h8r=un_8(4t5U)X7B^2x>WNG1-?%u5H!ufA0**)x%2M?m^;m-ro#A?qo8#e(GB5nPNp-cdGDH_S-MB&VmE`V46GJwq5UjVqXC|X5qdE=b5*v0LM&&JE)-DIdjk{t2|WOd@`AgN2ZQ`?H%qr*llc-+f-(~d;j9vW;`lwTeD7i60sz?!PoI<r`T<kwe2HQ4i#VT<`@K1^Few~R^yR|64O&!Qiw{B_iR2MF?@bx&fDB|mGPBnz=@2h%C#))IQHr6^5XL9{L|UR#npSP(isixK~+Yx-~amaBN-DkxPxfq(dDD@n@~aCcF+iDidLi2g7?0hUGa=ELpxA)%7GpWzPAlqQigfeD(wX2ld-b3j8+73v|^?VJ2?gKM7G)&Su4|sp|pUuWs4p1`Oo&Quc7fbK_WD`0zG)sHVY|rTS%E5n&wJtsgjN0vsv4FJ`Jrm7WCrQ_K*xv%VDHs7yif+;f!*$L^1VI&~}dRN#fex7J$No29fqT<8k0}RTR*9(0LH@WCu?j6u66)&vwa;=Bqig0WKS`3VvUrG9uQreuzbgQe%>bjQy|(8&RAcc10Nn>@#)u8i&gYeqDEdn~nlG(mf7JfzytM#SOWY1}aT>2+C{9cyBBED@~&+nO;0j9E1ft5v*Ld)IM6_1|BQlT6*dFb!FTJuobAb&n-UOsz*t~Myj-S8+G~q!{GPeb(d-Aw9$?>Rqr=3jD-gSj~HdW4H<VDRYS~F6rL?AY5DvWwup3o2VUSZPAD!2^jP@IVHYfVNytt~ZBeOutrfHABesnOIG}dix`Q@g`%w43$8+bm(s|{3$5+%gJ`4w^W>Ak&F#x(6j>r5MveTA+Y@6BOhP*&%aip`M_>@JWtURJB_OZet2a#p@2A;IJLuwRZu9C{oht+>(bBo#`4y%o;l`l3!4zAcht->kG<612D-l!K-5}C$-c!$9;UFymLK|+)nR2T@wI7?oNz21@H>5S~`BGaM%KIBQ`qiB^>yjoWf)4io-PSh^DEoC5Rnx?6wMf@AM=RSY522%+3BK%I-F(o99$3dmL3a=<f9LE1~e%G;AB$BveaE%}f&hsxV@@1f0-PfHHcFLe}y+!hJg)xg?@h(6PK4bjvhVKWIlFvx>6|<*<0z_q0_<bHoEf(_a3bNd{118yytda|w9}M*-A>E|w&bED{XEgFBgJa6?lWW4|e}$qO?~k_vVI%sAi2fPLtO(bJ<MY6+2{If->9-x=E)N43NGKgX84o_qolM^yc(CV;?wM2KO9;x4kEDF$16YHTxU*qAl=rquX)2&BpI=_~Jk+Q<+_>Txr|D_`4iH1eq~n<oCKp*8fc4hH?kd~^wM?;BOVV8h+=6PtjQkO1i96=6=_kk!3FqZ_^dWrE$yzn0M0u#;_y`=1en@GGMQd!^;`00R-60Wt!<3jIe5933N-9Xoz$sGFwI(G`NX2iI@#tR>HBWf',
 'longchau_snapshot.py': 'c-p;PUvC`AaewEhnD8EE_PiWYy3^r8X%DARI-g>lBuhNU0FM~V?#}Mcadu{JW|kC>CSV{>L0)1wFM%K8i(wcEV#Edl1d$N<L59Bv{Ymnxs_veie@p7@<X|s%d%CNutE;N(Uyru8w;yNe+&Nf;YiE+BWt5iAI9sRFFps0c3Dc<)Iz{x=8V}+yLB~8>MmzCx73D>ihGmS=pB^4QbMkCmM*jBp_SOtt04zzONf~Ep;h=Yxm(Bs;mQlW?eonI0x%pn4%CKozMrFK=^cekn=+NiaSsF>dXc@+dzb@m%fM11q5mEo3%mO`5PcZ*FUg__PuvjGVSpSSO{q0MvNPl03<wB-f;TyW^X`ayo^*uN6?YY)zJjtenN+=N4c>(}FBdEcktym>-xg}kam&-&yT9nIVYwHIOpTBtY^ocWa+yq-R!4B=5-gdV>efr|?1zIoMbOBV>*+f6~C)v_JP2Hj6o}HcfeUC4<?ms_#bny6L@Z#a)hX;pGpVJ&igziqch_Jg+g3q)ZZHIXoPm*Z+Se_P>Fo||%V3T8a>%pTJ5AVNt7<}^R;o}E{kmpXLVv@%z+H^vqIA5;HY)M~3Y(|!3bMO@X%;VyO{!Zg?o@PaKKb;=tQCOZ+V-lZ&snGN+k4}m0P9t})_2A(r_dkEER(_V{C*VJK-?>Flq9i=W_d7n#RF)B@rDyNqC+!Ou6Tgmb?cw?Rcl={CkJr<AM6I_zMC+Dg{pVJ^9if9eVUB04qcT4aW?;N%8f0mZN5y(ceGc<=bh-6ckDnfVFF3saUBPiB3p`~?h$P7B=yP$4KX3Erd;Ix6f8OEG59qVZ>2sW9lasI3S;;8O;`t;r1fG`sImMQ4ZB3(@6BofED@qT{HQ47>0q1p|I-X`Xe&kc9G=!98x#PM6KgrIb+#5J?>M(mLaA}q=!zBJX^2+FCsUXr<EYM<dzv|~c`Q!jNaUTUck5)-I!J{vDJzu2QObbpP`Newd<?hiJrSrvlZ!g?C{tk6?6r6#*4%7RK^y0nYr2zaQRe=33vpDsVI7P1#jEEJ@fCV+s(g>O_RA4j)1KmxncbY8&>g1(ZnOg2DjwVoqBj@Ote^bA+Fgfv*kf?6-UrZE(J}Kg~D8m$MR(<?3oDUt$HP}}RF=Oxqu&sW2FKnpP3HV>QB3{f<<$1y!QJl`{Pdv**;Zn$x>4nv^_>I?P36{BRO<w`GxECa2w4fEnIoMlB;D@VKlukV%c!gj6saZ+I0j4MI#bgmDQ<_h8+^@+V6PCLSYVGwoxC3YvC(RjbT9s24q6MwGu>$8uPUoO+MFYl}8OCN*OC%Iq3WR~?HSE`%2w;nr8cX`$*Utkw9*9lTxo3!a4|?UJd%N_BMNUX{Mgfi&QrfI@^el9C1@V-$XnF3PhRIs<b(>lFX`M}-Cv@QQ4)Ebha<3JM0MoGKd8Js$aA>p{EyMFFP1Bx9{Z@l%0ZK~UU9+soI;XYLDIa=e9s%u2<<VY{z*4*ak^o@nt_a1^$MJNu9RyDv{_rq(aR2ar5Nsd2W)(@4@>UiD=PyRiErZ;}<<L1Y$8fNW3y^IdXf()fa78`fgs5106`m(qNK~U9K7B6){?XoXr--b=d4%q2;E}t^vsK|9;|m@<lOLutt)4s2BOj+3&R;aR16oL3-76yEG%wC#CdW8R7a%935Eh9AAR?&(OI=(JY>*pN9Ca8v(|A%6*Mu?P{eU>*VV-BX=N<@&9A!y3+4xJGg^r@21$h#t8IBZ~n9ZGo`aU`OIFFLiw&}Eey!-JYk7gEtoQ%|>WG1b%J<-5kfhS9-j!N2v`Vc<Wm;gnx2Aa$dW`>$>qW|cS0qYp_gnimzO$Y3M9ps>{2)p@C!yyKQeL7iK%%DcdcBr!|dO3+!rQ>}sI_K>eI)~@0NPVq#$(p-o-W0+fPn-NMvIs6%WI61i<eYhX=p^A7d^Pe03NTrq#MNZ*`_m{|(HBc)DyHIYdfWZo-SCLF7~*N<@RJbyx~0z&nVD2cYp83jNxJz{NKZhV<!MD-#M3qN8*B~H(E^7AGf>WuH&7OXF&12B|L)LNj29-P)QIISc-8(SETUPKOuc4zu|%j3AuJ@iWSzrbMG`Mum~u69M{u;j(r7iSR)$Fwk}*1mRp{keRt}vd<d`@?Q2|cb^Jtl!M$XwHgAfWI&0-)9pJ0VGr)9CrmS!<@Qdr%&0~32%IP9A&qbUm}lu;ob9;1kvTANCZn!GgO6z54ihbg_zl@Vo_ljuP_z~qP_i8R*Q!*`E9W~U>Wta6eh`DO6srb!YO1%HdD;1he(%Q7$Qt);10qECjTae4X^np<=nPs7%9Z5C?bN!Z4kVu~*NEKJY6GCPS<-W;Bpz?&w|RqAhN8W#WupM>Ws;<DKku1hfM4oE$V$-rx=noa+~LrGu>On20Dj5R<_*OPK5kFY9O6j&fR<!l^<W8jp{b`-PMC}?e>HhV5GK}fPsZc&Xy=EuIQpMXP(@Ut?6cn9jTHMp$XdEBKbs7gS{!Te#gbjrn>e`BNe@Xfz3oke{0`ft}&qv=7P|B2WH0u<bN5iXsR#nr1{WUBMvMqIe6(S$XX@R3+sB##B#F8drliPLpd+4kb$)LF1cxzyRITch55a}*y0bacv2%VyA@n~<CjkAw_ka+z|a$pCoJ&_ox7-9W@FEcI5$H>H&`tw)B6)aKO@BAi*H9v#=(-*FB;djZu~WLSARg^TVqSlS5-Xq$0DA*p+-kM4yZ`M!T|H-x3U`e@hyMkmgf>jIVu@4N3j1-TVX)eh{?4mhF~op6TG6&~PF#g<LtQWojg{TFy9k+o3p2F*kK!4|Q3;wz8ClQ6-WY%VuTpqDt}W;1{u(%AiqRGI;*o~@2G(i<GQ=}$)z!)?#^=ixXlcGNXs>Wc7n&-pFqt<+96%{9u$rm7myB^Mh7=N;#9G!G}|%&X?)gHm+zc)loifYLk#cZ-?_8I=7nK5um?vUNU*h<_qq!G2!rppRuT#q+g<T6Zb#31-b<)#JpCa*z$78XN@$)E?MTTLv&p<DB@DyTM`qKi7sFv9Ltnt6~^-l*myH&0|(6)6K_;jT&=;5Nn?)>(Z(O3lcJk23bzglll7U^-mEds*Z|%w;^?_JO+uXMSyyzakefR{J}1;jVzCAV#|Ir1GM!78-lLef)|KIHsT-p+^8eRVaq^G?%j6ojl`ohqX*RVf#{ed@mf4hoyawfi#0@ze0AM*g0?VOEkcjIr3uk;d&m9bkN;^~;yTno!{rp9Z`5K<J!mN4!?=X;L8k|-P{>@1mzj%+ER_Z6JBP%?^K}UDterKN7Y*p}w5Bl$0|gSoc^<A7#a#zlTdZ4zj3W{hEki<EBK_GS$|YL3h_u)0Hx(prLCw@RY~;BoVtBA!boJ_&Oa1K^>B1Sm`6lC^i>uc^K_~kAch2vBa`pP}(*=iz|L~i&^X7kAmRNg#D__WVZQK0Bkr-h=70_C_+}JH3()I1-)BdlTvQiz`*Nhx~d1$SK&&Aqtq?Jeq2`pC7rygJo-sYqd)f*T{)}c{BKiau<+;Kd}qWVsSjmOjsY=wA9pcU}9u*%+XUMwIJQ>StnoF$ns*<`X_orAmK3!PD%J72~rIc%Eshs+LPC1+=0o|2hldVCxG|JEwmYw6rWW28H$Z+`9+Bqh!Yy7ym~&iv~2FJj1o!<+-r`FZS|##gU?S1PethBWleuOU}xc=gNDR^4uU_xd@FACao$V61b`xwW^qcXM-_{J5a*$Gzp&+JtY=TmqLrViAHo(iaK#(>ERL=f|gW`XjM^`o__JYZaX{>qrB~4LY256=V=ASp0h(tS;6`S=s5WC4eYU5S!8?+7i}5=a$W&EesRXr@R@b%{XhuNi)uwamtKrY0j^4>a3P4tEJI6Z^mh}{icUO8ES_T*%bbVbBvXo6}uz~kgjN1$f1KXg3pIsS@<|qIPhqsNn$DOW23ef#?->=vSLt6;%Fn2m^ZfS+C{789G0eUz_bX{2S|x_Jt5Ydik|E%KB&Y3bF=-ex3)tRs^}I*Dh9*TJe-u?IGdi^(fnFKo%j}LsRW71i_jRvpeP^dx$APa^P!~3azv$FrGj)U^mEBuKH%Q1Q0P!#M;w8EMk$EAFGPyeG7K0Je@ou^yIM*{d7~<bO0+z^)e+IDx{xs$odu#;D1+S*gT?Yb3~+uCo-U=bO%KkP<N?f^B4)j$DhMQ^xQN0jXw^=PK%`Vcmey0#=X#9JRy1r^Qkc5SfcWiJ&3Khiv|0>B3cHgw-1j6-Ysn=$ZJ~2NI(3V$KuE%`;zq&PtpfJa=CYY;(ipRKl02aVsb)YTbkrQVLaYqr+dS5IP?Y3XlkF=^xJZCOPlGX)B<P7K4!1^RqFCTbNSsnlYaZMOvv#n@dNpMiw=qmL^B{{0)D|IEQ0q;xOC4IKKylHj?D1VH6@ruKyo%Qj?YM26;ylJ9lfblcB_Kihpk4%`<&TF2O})acwJZR0;l5@YlW=Y2-XH?uavaas@V){O#TuMyz8e55!c|rc7iOf)Zy4`j|3G&;O(lLsTaa|Xx~GPQHSx!-I+~h+q>@N#29<<tSRp-2QPTvfc<a2Pj%&yCil_mP&wX)+Y*rw7wet!UR**d_!O%dCSxX~VB?Fo^6n2F=sA=ncEE8XKGc8RQ@_qq@S4v<bF4$vJN`~VvRymS3fK|9?H=IJwOEN{~RSKlzyl$mBb)K~1@qw~lI((3xt+GZ}DZO<Hj5sILuw2`F#(cPhum<>FbE9fvc~rync+j_TYQ0r;O~EBN_`i`<OsKP<{hfHn2Yo@jK5}3*pc~C)lA8^_ur!kaQrn0s)D%(aAs-CQQnyMJ^dW5**cmBOhxL{TgjYHNnR81YSW~&RvI3t|_rA&7`+R12luq8Qa@~5*2X{><;?u*=9y@4D5Rmwe^U3}17w$Dk!}9=b>qu*h9MmFfvtF#X-(+;^rK&O*O*Z9QynlplkbMvAj?ye-l0Ej)2XDjsTH|V*d7*)el>&I1Df_KlE)5lwdTiL6s<yhs5WrqJdB=IC;!P?CL2+8)OJ{Lgr9JMl^3hNWMlO_q5<;$v_jVWW>#&+isn+H=jVXr_1<JCjA2TyKXQ+tf0+?Q+4&u6V=Xe+XQai+tlcq6~m~b6?$<AC#s@B^ZHWI!K7*y9V=;AI(wBD#bw1b`0EPpyJ+)lk`5u@wOy<USUX4b^es;biI{T$w{-lc4Um{h^+R`%-Qh9mig+Fca)b?Uu_QJ<c_5v8=w*Sfx&KL5Y=MArdRLunR%MOI&B22g`bhViBXDe5%jXxiY0rkr;5Ffu46;N)<l2}f!kT%NO#(QNx&Jl?EKr2(pLt3%!MV_75@C|QQi=H6jioRJ6BXB6Fd;gs9Wr*1blck25No!@UAG0I08-OZNK;yoR6cqj_Rqv%stKVSJb3c4ZP@pVdd@m^O9`x@@QsRZ;|PoVQY2_Ck}-otpR95)p?sd1b(c!qUjWSvsV2<~6RgUyy&?JfJHmk;bgTtG)9{babiIuL#cQ=gnRGW$bgMH%Z~R~xc*ahD==;-Bx5&&Ax6XNjp5i0D>|xDOnl)|Fj>O8p~9aHQi}F1w0Vz^8IEFONygEH{^9A<zvV@X?s1luM>4JSscOZDVS%qpQLz6w(S>0d}c7mTHotYN}(JqOQxk=F>;B(gqp}RlI0s<oNfUy&JgAmCrdnuiR&<h8iC+y^h&rMudqC9%>}2ajQg|3<WB3*FtT5$H<y$F!-tufq~+%ao9{Y9x<#4tf`{7qTsmB_ul+^vH(Z?;Wt;We>UNK!++4Oe|Giizb1CR?lGRj_WQ#(sjA!?XD9*5nOgy=U?%o1dha9{G$gu3&4o8mMHtl(s&Q2KT^d4LQ*GERs~0@a62fa-OvMOV^Bw7M)-i8ab#PWY($RHfr_EGcuIgscFr5M!Su}?7IH`GHW0Ip!<1Pl_rEC6@tUz}{;8GzY+`187#VdwMO`SS3iKdqVc9WQrtVQN<Yk_Y%bv;d^p@_mr@g>bRq7O^Kj?6bY+mK~~-w`02n32gn(h=`=c~Ey`c%<UO!>WT;>Z3}#n`c`0j>rYxFp14)ysan!w%Rs~tcSfvN@fZiMXG5)=7b_A$qPp1#i*{dHozy9J*u>2ZQa$WII@4Vp;@fSrT>&EViati1)e9!GF;oM4>~i8HOkV83i5^?&JOEw#eW@>XwH=WqgrQ;0cX7weB5sHsFR(>=jT)p&*zJExb!b7ULppv0c5<Kza2oqKPVRA?K>Zg)NM{*etLY<V!B?g3X3+7C72YXdx}XMkGOPV;IjlpRN1|VUQXjVj5Tktkx$vy()2U&^HzsydsK_L^~3fpNeHw#(r}*qX=YR}-hEeJcN{wJzAK$Bw~QZ0-Hyy9$YXQafMYVP#{*ps65H3U+E+EIi@-SyFzeDUWjv9scNE0YEx&PoP@X0(W~ePFtNG<t<}v#VjW#kc{5v<pg@;I)!<Y;@nnS`TsKA7JC<zpqw^=$ZEW~QGFqcAW#IP}|dJu9qLrsq{*rBQ}LoEO<tP=QW9_4=hZjqi%-%6EBAh#X9%y90tiFwN8JEgL%J(iGDj-<TX0yH!$QCC}!ZU21JDijFR|MS2uQtMS&tI@OBmt~m|^?lh*lauvsZ1xHhRrF3fWPu$Cz`m{Rxs+RN^f=3sO~`ZuFl3<NzT2O-UWO-#Vn&EzpxR-Qhi3_8`#?~Nl#$$GXR>it7D2{=uVQ4CHpxwSBs+u2Y!*A_RyjM4T+hr8b;FjrPD7U-aIq@+q&}^$ep?&AE}|kBQxOBsf9xn&5;aUIDH*!7yj9&L!%~8Pwmb-M^d_^0?+%BIsyTeqhgZnLIg)i6=^h%vFi$G!B7Nr738Ip%&0tJ2^Lg=D0a-1QK(1)vzmRs%W|iu)HnZ3Hh!)uacrN!Pq0>fupPyDUaCU2{c%CMpo<84qszV7+xkg=6o^I>$wJ5I+pDp=%6I^w~g2>31D?gWQ^teR(jN-#1nMwR|aC0z5$syOQ?fHYAfgz-#@9sa@+$8Y+G6OV9LU`n#F2k1|-Hva}z9W6<gDSad%%ZkM@ro9W*0$Bg0G%SzBU)X3dEA&sZ$w6cCK|(40~AnO#)bn)q`+B2t;|%z6xJ5GQKOD(LY`h9$Mq|M5;bK1P46%0AIwnd$y1ZAfQbu*UM7mgxF%NhHP;FoSDRy!x;CZ5l6RLT<XzQNw|1x%QP&G=6?*b4AC0OGZD1V%e%6T^{R8@IbYp?kD8*|PJa&1l7lxqxl`0OYxF(twdb1okO77AIMtRO9=y+3yBdXPJgw7bjB0)yMBlH{lK*fSAYRz@jOP@n%I%wR*IwfnbfG3Av;Nfuyb2*~bDB?5WnQIjD(AI(MQsaP7oVkcx<6H&G;~6?=h=d-j!Z7Kax}%Ks>&W?zRA&psu8=MF`bapvCq>X)Vb>;R?Du`mfw`Ic8Po^WNy2Ytz?bAvA_I=SDfl8n85KQIS0&?nUAXpgWJ}vv-1#Rgs?C{iiL<au^-6)xO;DqIwy9Q{*0_coC{3gom%fpTQ0JjB+iptA-g_}fEJ|#{l6WOE{dpigimb~0w-Wz8BDdCz09^#{5-F;oHQ}yvdr+CNjbUBnm-*=!Y@HUQG+7<<ZCD)G(-=7e7#-~0RY!rK(wyt<E>`OJAeAR*_8jRiow8eJVngK3YJs8y5no%?yJ4&yxfa1$tOF#_9)T*U808Nrn4~G?D-b>99jX%;q+C<Mg;eW$@9%wxH{;y?OFqW#?ClX_bauYye6Y9I&5Qgx2+ETP0%EZ|C);}GK3$!YltSlFlB4P@L1h`NsoMJ`S~*jx<*cmu{!C;OcIQi?)Ld-nV7W_*xYlf+UhulA)UF`4?$|$t>cwi#R1K+($e!OPpLGvV55Z_Y<kT^y4?O*}Pjz)|<4|uBqTu@NB%%%*l}EC;y4fv_oS1qpZ4Y$|`73z5|83<#+#x{d0$`^bUrvVoSRE<}?@^gGWw&9x$4Rl56N*gzIna9X;&HQ*`0eo00B(ZV_0SHoG?jSgjeySRK1tGX+|}F;vSE8X4CdDjqm{Q~$z1Osy{xmGu3dCET|;^|TIDP4H>SB%9)^+-+mSZ!m!T;tZyte(aipC30_q{5N?GmQ!|Pa+ca0!t%2z0g6$l%MkE;)(ju3!-0!gwP)e-r~)w`uK;8b7$fe+H?OnEgNqo7CXH_B!Ly3}?)de2;1!AlyR!WK|x<*r2T?!t?o;2lH-Dw<8IRB{6_hoZ^J3Yq|62SiFwV=i&xNXQzuk#W-oUZ{TcgDf4})f3L-PGSfc@7g3#WGyW_(c`F&dfJTA=|Kq1ZH#K^e*FPCf<|+iaFVLO<g8z-be@WoIIN5&&2|)`3MlC81p$8#2K<J%l3s;KMf-{|wmRO<V6z5ShuaNYJg)hGpXwj<q{cw>v?GRcY3+^<Z_qaE*6YLZ)Cy^ka;`>H(Fxj=c6h_;%CSpLznjGx>GauE+@qN4E=dnIina}LXGg5K4Am@z0MUS0TW?o~>{}QO4LZ_WvRU#nfuedlff8X+-|`lq;>d*>K-5qya`FjO%CNfJB*Pehf@{C&$s%^OariOYzL0@OR9A6)$&}x=>xF7$Z<YuVq|P`K??CJNrwph{{)dj;LQ9k?FVDtFD>Y4nz6e3jo*K>JJbJME`SZt)F6#2zWrsF4Vv)F9cBt>nz)3axuKV>}aN&VUP`gk*1KsGgpMK@wdY-*Q2p)%L%s5yyPRWoiZIqOM?>6bx9;i$aeyM`F3YJn9$96bKcURpcxm@L*Y!H0MM#I=-CtSM|n0uGYAV^X5Zirf#IZ*biuL|F{tH{;vZC^~jr%)G#{OYWka%_!ibe9OsjOhsCYsb(MdBjDubjMW~XIBN4@d}ut(ihlubJw=;8+Z5RuD*IdSZ%{=9W}tV^3)*A+1w%E!9`=|^`Jph_HBh0q7XKoD0CD8LxZyQf;iDkI&L8|Z#rD}EoG*@9qp@r$95J^J*?PHbS!oNIf<f?+Rz$sXeqnFR%g&f<wn?VdCOFZF3h1diW(P?u*c>k&a@;l%_L~?OtpHh$*90~qS%dE+Q71v2JOp*bsgU#M@=8gv^uDuM8+&htx@Y$wMN@YYz2a<CY`@~n+Qe2=xo{*mU-E=U5W`5)_w7R3fwhuZ0TyF)BmET)4RfCJP2kJ?ClWgG@XdkA>(Tz&4i>whm8(3D`S;PP4cFmQ<KS!IaE5%DJM|p5-s72vRnnYCeLruw0m@=nFd?e_$4>C2lO7bHs!{;xsnpZm2Wi^p9qG(VR++gZtR6h;`f~0?J!|`zsJzFCtIDaq~<%F{%N62kr0^MgFjI-DXnA(rF^}Oz_hbhchE%lD?6#n&!L2Cukk1_{|vU{b=G0axRi_|SD$IkBpW$$mAbRv)+<+#=(TH95X`nAU8=|K1@fI`MRCH=HD5%&@Z~GKxsa~5nJ2nomV|TqE!Q@?6rD>YUQ$OxpqsT{Ra>pn3eD~ED{FjJJJmgDv%(5P>orsKeu2(nANzqRR<_!7U0wIQ?b@pfX{&n52G~(;*I1}7km{bY&!`^u-bNqllFwo7j`Og7v3Y1;SMHpdLw&?1^f#cQLj1sl_~!!P4^}0r!`6XNnd?K_+8);IW&lpY6=J-BstK}R5u1@uaB;G4>#`H;U68`_m-Ya92d@x<(<+=nw>2T#^FxHTkM<E8Kkgn1;o2~8hl1+0kyg?FS>5p^=}x?+T`CnRg*<hK3_`|@*RvTLUvcKY;I}0_dWtv2opD-R2jp1=a;B{~^(mk5H{9UK;GvL(UbKUvL3u08S;23xrBj1cos_!UdP2$ku|0vDj1qhH+7AelS7F|18TTz5h-2A;FOio6`kpEAlM>k-2#UUr4r5^%<`VYNM+TxQT&gJwdZ6BG$!7%x5mCCBADM-*g`>5mRlG>}xA@)-Clq<L;WU(^RLYxf*8-Gb5k-PnpR0dcqlRG}Mh6f}fwv)1L>SO)C4W4)9%g-Aod~v;4wzm=OJXW(6(gI@@_q>l?pDU-4Up0*xeJ@N-e!bwqlF8t<VyDSQ*<fhCPnK|s(@4tjV?fp5w4PSP275ENPU6a+%d6wz_T~vEDeYQGx_D#MmyI!?nJsI$~e@*re5btR~*PVa^KOle{cR}?fmh_|LC0X%bmVj<NZx2QMh{j+lf=6NnJy<Dti^cuCiWO>L~>QN4Ez&tWGcYQ!x$XWjao(Uip{c-BYJ;{u{`g<MmM){>2;k97%Td>YKQFJ;0FONcGbt-#fbNsJFb~D#)wf(JPYE_3r17oXOSeUxj$_@0*|F-aw83r{BWhnUL3+<IS}{iygeI>YJ3(s?G_$>I#c?u3rDMbMWE^yMO)S>66F&$~b3&kGpcQg=YNKIp0C~m6Kk*`t4G^AWPY${30!UTOrc%a!s$rYUA+xe*ld2?gXp<Nz8B78uCBlv-|ynp>@Nnz+<_aw5x9;6YvCxPBZmxlNt(LdfDggJHA$@KYse;y9b}%|2%m9@Y&PnhyHTf`d2*)t7d!ae*v-sHKG',
 'postprocess_snapshot.py': 'c-p;MOOND85x)1YNScG~!Q<{7Ew36C(kwfxMJy~DX1UCctSmdLE4N&>Co|pMwXK%8aYP7l;<BUB3ULDo390A6VG#eL`6Wc;!%jb{x>^hLrTxf^j913@Ww?%yk3U^53tlopo1Ci?c3D?K5nAR%Fqt)s)HGYuGhUv>gl!fq&sk1*Nz{swk3S+!T~stDnV=U1DJ#VmRkcozkB^V|MoU^zxhnX=-F_jf@@Ofl4aimag4h?ofLk|6t{TOQqobpoEkO;Mr#LOd8PV*Bz@K2M5v7)vpaL?)`DR3-XDD{5LRMx~4yw<CMlB!yAc!I>^g?jOI;k-^r$xi8aC}1Mq?4$hDu&8Yv4vOoG`ZkvmC9zh<d=F_Vy?+|NRX&a9So)=W=*Y^);n)DwG8#b;M!6)f~8buoX<a^g@k9k%vq`Ck0KQLc^TlSfjxsI1gi_0F{`yPb;%2sVibJ;<@}6E1(w+GByvR`efN8z5r~#~<|kFn$}kX%AR<%}sldS-BM1MMl^~0v%GQ_`V4?^Mx>@9OW=HCHgzud`dY?QX_%DjdA_$^RLZ?yE)Hziw)bb)f)b{;~U7D*fYV#wjL@teqRih5F5OvmJ29+hG9LqzKK`4Nnkibd}qU5Zo7Gdxp!I(h;bz1i|))`RJ4NG*8Sm;6o!CM5MpW0YWn7UO=Cb%WQwr~)w=rA9gGr^bJRBvH4AtqVXluBA{^>c7Vx0QOf>erCTn+@iWu~X0EOA_jr$r)2&Age}XEZxw$h8#@wqMYSmBA2-L$#c=LUQofy-h8MLDHImH1+<M!-d~7h1<RD4@U6Ku^F{ac<({cowW$lXXRa4qLAhk)8I0f?_Ed;UguxT-#%%Skzff}d?sY~syj+og{_ghepU6)>Jo)U|C*=IyS32Qf{>2H}nFNCvnF2czByhIVh-Mk9wMJw<C#RlPIuG<M2yitD<EWn**-)STP~$)dvUvx=EBO+WXb@&M)p#nevStIdNA^9{={53lx$TQsvu!V;ddI;-+;CxUHvYAeQ=4NPHgpXD#Kbu)7n7i-;#I>`DnJ#;Ai?V{b>KBxQ%t^$qC2Ac)F9q;SQoc9Z%T5u;>2mFL#!8ihzlh_S5R>GYx^M6MBT{5Dvb)-Ct6Rx^wpxOmJqML|FH^4n09p?=?5rT-?j4;74QNkn(*R)_8rC({cgr2T&5^5j4)94nN$?22mDJV|B_*hwIXJ9U6H(c<;A%xPexEQm}-h_uT1Xn`!e_##GL5mjdRF)_3m%w8T`9_`^VaPWJa#gw(FoP_pcfbNQnu)HQUJL^!(>l=`b7^$TRi@>}yi6Gn#E<vZSDUjz)m-1um+q0k29BAl6JsvS1KKMyiFPyo3@5w1y!BbHHQGX?ATqCAOWx=vy|m1?x&G@Kwg7OmUFS2hY<GISd?^yo3QIAh{M6L*>uZ-98OIxV`x+stq3eGyLySI_q6{^`<7p?c3ix#&I{Vw{Uawnv>1#+pjb7{eS=Z_diVfDj6kgfVJMHEVKnRuwHe844~9h%Dsi%=9D+l7OAP32U!2aH`navRrT&G;HtMbU#r8+HKh)sr?q+!n0v3_p+8(b1X_e_49YZk<8;b4H?o7Cmv=(X|I$3r-t89WT--GM#l1m&;Os}l)%}s(_}3Gl!Jso6ZeRBBpz}jaJXc}aw;ql=h!_=@UH<x+Gt~IXmX-DG%|8@*)j(nI?T^F!=gNHC>K+DtNCiz>o@Ab<t|-Q6DMjln4&YdQ-wl~`ruhq}h*~_LZ|RN-%>9EvhZ#C7-3G)!9pv2QWT^qreD?-UhIFgp^AFrr?}L+bMCSYYIa=OYdH1gQ@rq^Zy5i+nJ0fXg6hBPAoh;k8V1mQ_h@9#J^)LWCIN8AD!JvDXJ_wPm978YM%k~NOIv}S;bxTLl1tk3Rms+8i{0h&y_WGx6Yp<SfYxnSp0#3DPl)2ZZO%Sr5zdc`Z*wPKu!rPm_XNYgV1!EVxk#MiUME53OQA;;?2?unp4{F-z2Qe`aIQMFkp%#GYtkxU^4(h0z&qo|7xN-<OvvS%EY(Z~E?CF01hW7cjW9;6V1M1-pmcCZrwNnlb=b^r`XH2gp(mNgQVlBoItam^)vA0C6DN}E*ai1xTpu#1~Aj}<E`wZfWEhR~T@SAoL6S~|&PXpW#<6F~p5^M=4gB1+Zj;n-?D-(}N$ske9^u2`5g<}|A7h6ys31Hwxvl<2mMwMVp`K&o^Ut9H4S<+grDwWh*EYx;7Ts?dUBjKXer$;?T#PL-FN~&#Y1Pn>zfT>`TK^>p{x6t-_s=Bqzr(cDvDmKMYm4KHmd*~M|lQBl1XN&Q)j%={YzZUWO-t~N{?@X!%F6Zu3#;qB>eTl~O7_3v(Bwuhh2J~GuFv)KA-8S8;6!+?Y%jq@_G3k|-IhwVDcuZ57`3T7d5VlL2Pj<#adfW)uQXlN+BW|!`l_@?nUTmuA72P+8Hc@s!G^Wgfn18f;TUZqRfJ*jMU*d^%2#k{fH${JdPd)@JgDVB#3!R%RO41amTAE%1X@dvXzETLgrp>`iS`=R8^HAfCQ*MhF1|H031vtkyrODxoO007(G{o{eMI6=_s)FM$RbW1Jf7m&kjjPgaYU}CPNt5H0uxOFyoSv1S8qefGfDPyYYy>*%iYzK<6jPC{_&Ec=p92ojCBO}##2Sjd0xjV&aa}9j0nA$R+3O#)#`?Gvn@js9+Kcl&@G(2<e7fy-ch6xDON*u#xw_uhu)smKVWCg=ct6oQu*o?XqMhg{u;V9#lgtjtp3L?M-eY9$YugMOOg=)J3hG0Fb)U~k2Qp)S-qm$%h}_Y*+u#n{8ppeKpY^eM-pT$pd1gM)Z5lY^g2+l|6}dp3WS9cEfI49)5$;n3FYwGxy0paeJXBTD)GFDNU6wFRCD@>l4Vv^ZIXx)TM$~_&0$rO$R2S&F7cYCKbN5xp>9^}cPJe-s$)<GR)Sl7Z=fJ_?+S>tU+GNkpS3QBxD8QoZ@Mu5|4&9UHd!>|P7J1c0Vb(gZ$M|N@8!bcBCGyDM40<=0;c47C{b4t55_3C<goT<-C`=MK*4l3S`P>_7xIIQn3pORC?M;FD?LvV?gA}M_(MTx5uuh8*frg0TI{=Mt{t!K2$QV$99-^QnJ%nkcJ$PU|?0GVnwL5%#n)BWa4|MITPF06v@OBJof%{01n(qTM^5U*94KEFovE>Jw5Ig!43`!DgC9~N6LuR|<zwtTKZc+EMfppkObN7+`qU#Vm)$$-o5U-C+G5oN@cVa(-OZXZs9dHw>stwPOy*$=-!RL1K*;=|EF`RR%516eQW006&6F|2F>~zhdBOelWv$Y9Ju;VavaMw20RI-DOZ{$Yjk9kW%P6!Y|5TZX{UngJkIvD-p!5^D+gVPR5CXU66h8OMG+x(~_+0Zro%)#h@t_#~$un6JWK>Vvpfs4f)*MZcAPcZP40obtZL*2WK`U|zcez4M`o%DJ3e{%I5+}XIL=K}rX5p+V7G$s5jXB}?1rk828_A(5%<|v0I+xezpHXQ4>Y^<}6{sZF4s&o',
 'priority_probe_snapshot.py': 'c-qZ<%aYs3wcws#0n0G~P|ZR;<o2{FQ4U#>rxk8X3Q67ViRv<gMS?}RNP-Q3RVtTjeb;w6yNQ^X-7ID?tLw!qJTd>_KbiAPWC970CAIIkqeDdi`8;{@<ay_V2M^w_Cd*~8*`6=c>A_}lkpz?JG%1Q;nyogIJT0>IFgVY%>mtd6MKYNsc@d=R;9c_LHYv(t5M-M&&DN7;@IF~4>vX${A3S)_o9EdI+E`Dwd7iAxc)l&Sc>+~en=CJb4~rbydYfg-mmibqw#@QDFe%2^=`tyknW<kUAIoKWuJzX2<x<}E<khsi*(3#5NLG_{8E?yUX?lW56v=E-CgU<2<KHq}C0h65&*?^g{|Tn5zgLrTp}#X}djs#Rk-W8v<fg%5ft{PD?fE9p5S-@CjrnPhqa<*o>kIB`n=hfKn1-(X!Mi_Y>3V=aX30FCT+q*AvrJ2c)^eFlaWTx~pJ!Wu6~Gp1Ot#B%mQKrF@4J`p-oJYDIv54vCQq|GEpHBBoaf2G)$vi-`}*ax7f=E|-}VCdFMI|Vu_?o05F*?U7v*X>m~5cq=>&T_{D^P9_*j3tT0Z`9bQJ$;@bK^<{rV~#*bN?hGg)730p+meKQ9iRzaIQ~AytyKTjSf!MGm-k@M>KEy5z}$;vd#MhEjLEUJoW2q{a9uyNUoZ{UKFdWJL*6i4<1!DaN<!$tnr5JP5;nyv(kXJnB=8JONZ)OGSVYv=?8kLoAB#;$%H5uG4Z6g|U3<OXsV}WrE$0=SewTK!ekKa=k?OXWMcRJj8WaPHq6z%Q9JQaQ6V%v$7cTCMrg+vvnc^hh6k$odnMpu(k=V&pcfw0Ze`w6o64sJeb3f%0&Xtc{*99e@<q>cW=H6W*KaQb?~R`oVFr1#s~s+$J6;d4Gi8g5Mk@_M}5(wO4H>82*Gps@LP92u{_I2@xLvMX)JVNJO@lpa=@)Mg8oCkO-woH87K(Qi!wp{o6_t;^SLOq%{W_+bKJ1Ybd{P0Oc4b1c$}`&ay*WTWI4A+K}~EoxRkLeL$D{nk80o_RqOEZVdO3aF8A$SzwfpU-NhF;HlzkHG^7I9K|eSoL{SKiu~3X8C3G8@T)`rZ0tMtUo6N>oEOMLf#}`Q%h3l*wpC_=)NjM0O;(n0MYh#F$k0~rz)DMy+5R0SO=?;oR6R;`Lbx}^%(<Gt+41x%jW)RGw`?BB1sbVptdU0CJ5pGhFzVV=+1V=8QX|GID==HnFa+|!&^DK|TSL-W4MMlIa(Dzw(y$%Q(p$C?0ASmNWd8(W23}AzVF<xabqHLW`VP;>z$Ua6#@jym$z(w$%$hSa*%DP8JE@sE?`F&&|;z-eO5M04dn%|5N<N7YjNqd5EN+cTnId!JXtVpy*qiD>ZYyBt!WwcsQhsgXOpv|CojAVl73Bmx-PhQO65Lu?{#3mySWN2w%#zX|&&naz4#Y@1x@q9UpsOvc#s^bEv&$?U+Y!EH9EE2_LgVz(<ySj4t*IseLZ|s!tdJr_eDMrxuzOK227X3P=>+AvD37ruL!DKz-p*$T`1)7buVYbfkIsO$42R3E{1R@!0doae(LHFe2G)ZQKu4Qn(oxzkI!*j8PGiM1akk_{Zji^{A$tJ?_x*s<aps#~qy4bESfmjtl>fs4ExAV#La-86}1l9ycA#?NxSBfBfdIq2z5LBGD2~nk~h<flqW1X#+8I<yN-6)NPf-gn^V2rfbD1w9xOFd=8ZuUqAP<@8J<;Y;qVpB6%hI#mVpp|eYPS(LKHGBv2L#`4)4{Ts?ep4nz*!T7(Tl&a1)>hnV5GxDMK4Vf%9J}Y38C7Cn$bemGTLh`yvP@-u<JlHMo`moSxyD~Gm*@ny`Y*nW`}@Lh*fJ|*V1S7TIybGiZaTNBKN-QHb+4`52Ij`Bw;-rQt8^V*4Xh3igOfh7WmhO9)vEaxfcu9Ao5urEL88BaFxfSnV6!v_{=OLfcCtijptPTdQtxZP($l&eE5eQRhj(>cv@}5L-smb2R!D5l*`mz1Q<C}O=aV8q#(fR+KD|m_fDjVt%wuW7J0?>In3r;k;^8boxJJJ?`U>#u<X3bKeR*_*3+*lZB>3ggksrwry(^MEVrD3k;qVdkE`dS^&*ng`k;Wf|eLs32z2B4;8|3fN*Q-}N?HmORf><Az=4uo5U2dqgUm0Rljd2is0Kr*)z5yZ%0Jx^dzB9raOKB~S%P(pbs7=R}97uX}>DM+Q<XgUhBG@1}55pMcUnW#Fbdkxqj2^#4Rz4PC@uj{&c?UiZKXo6lI0Y^E8nBpcC^2Xs)%*9~2)O+;$DvwX4MeclurFYfev*NG5EAmK<C6OO3~P5|&$IQKwHkZP6_62@%5!iz;s!FFXG(PUltBoQo9C<y{Tj3=iBFOn1h=*rH?()Fwt6QrfNTX@;#SqdIw`NST<~vvnc|_Y$^`@mPk5G;z^miL1WJx#!-oTVpqGUQGQhAXmOPU1QyP{fZK9brKji{?mVHtu?|^c&_kilQM)Q;PO+?B~0VAH{^bCSH=!$?sWy>wL&Qk|k4D@+2n}MEVo{^M9OlZizf-(!J$`x=Go*~TEMGb~9s#ONc&$JK#Ky%h`P*p`D<1Cn(O%b3b3Rx1RpH_`lz(Tz-w^a3QH3>~6<-YB3;ZIqi_SJC~#?z;Ojd#X$0UJI6h0_?-*HLw1WV0L90U9>t_ELAm5Lh%^;L8E?YI@|tHeP``<&G*+S*V6>oeqzWPJR(Q4DipHv$5cCyG)^fAV`~%if2pUaN;7F<kLk|&F?&X5|wGWO#ZSs?*DLp`p2hd51;-}Jmdz7V6{bQF$t3O<Qz~4zC1bn;TZlr1-zmjg#$@GqM{YlTW0p3(rxtXA!@+Zvg$u>fx(>p?AQF|hri&!mf3bTUrzF*|Lec}2}d^xR-h_bmsldcJsv!|gD;`omlzOcU<|<4{sMGtdWp>OO}5R|mf%$PmI#<)iai3pp8oO4*~5N2Ok@@E1eIz44q}rPF(VXm2ql0g{Ob5uCnUX{hqG+D#p%Rco3Nf5btpZSjm|?9fe}_Q_fZwbz3F`3EGJW|F?;{hv9$#4xN2chd){jmy2?DkwZ%d)Rt3rqu;8gU1+QT50n(sWH4t72>&74P6pF&bL#%Rmh${T3e+oxZgxeGpuK93SO!IUDN=%$*?Mr#HO!{ZYbYjn&8jD$nIYL<mf`C3gsU%Y$2THR_Xqa!25zc)KQN0hNPZrAHa?8{C0f_x6p1(6@mvyU>GT-bM(&xJy7hW{?l`|At<|nk3o;tgs1#hT4=otZ#SBT$aya00x>lwvT(p6XbYVJ}O_Z$!@%aDVWC@B^yPex*t7@)2`9|NsimzAmE1spn)d`9L3IlrtEVFQ#UxL%+E0yQZ~eucC7`oo8}#ukBWdt&SjQ0gkpwn%cN+gOKTZ`NX9(Pph9TM_?mZXyt5(P|Z$ua3Y7XHN-3VbVSf@H{So_0htE;F87hOFqU~y^CluP4W`|PPXL&|AMp%!gI2|3Wf9oPJE;Rf`m3%;Xq+j*pE9JDb^Sz*J=(P8|gbBGHPIE;Fbnpq;QLEy>g|~_5)Ql2f87)8&dYxAm%M2Mn5!!+=k!63LiW}1Oa}v!j?O?1zjd-K_1;Q0v;`+&~nQVs7w(CoI6&8nwd=nLR#4bG@VySQ*enDAM_ZnC`loxA#Dz<in?J??>dGyVIKh*uJ~)SagwWKT~=y#V@Q&2T6E}C>of$WK0Mn`nQqtMnXa>Ik5tmDDcVn9^>j=iTNvgG$a=Plx9b$?Py|Zg_{+YTmQN#(S_SCRhz-=o7eRbnQ=>zf3c@mK0v;&T=D?)^okD3Ud@j=eA<EF%d~RvTRdeR*%ELNOSRLk0H?(9xRWq>3Qdc=xyl13A9|Vth{9etvG|Q&No^5ym9b1;qJgZ+!&^bYDxz(;PinxJ{9St)*uM)A?pj5+lM%ChRcVp1)jY9qn`e{R!-!Fjmng!&TAp;?-7ZdL+om{K|W728C+GwEO(8Oj-8rB3D!b?Mv&y$OE-G)SFO-EAh3z>`n3&*9jPvHO*KZipWP-mo=wk@Yfu$9{%q0OPvTf&nEE^x@A$tpC~=@2naDY=1u*k)k{>P?^&3uvgFhrZ5qkyoHy0(i5<Hv#c2pVFB+udR}_$uk(Un+(YrV{(}A_rdFX8)7HupMWZO7;k&?s2_mo%}@tOiUa0(rH?za4Vyr*te7F&cE~n~%XR8}JhKiB>ph>v?4yHf16o?4JbN8EVqyM-<+4fh=N?H8*NppPOzL;=FG-@WYV0IIe$2M9QPUjOSfi#Y$!i#?H9at^z^yn7z~wd>AQK!j4E?5JRJEE7yG5JPq|sk(RhXlIwc-AO{QwzliimT2*KgPLx%xE#uAVRk#K+ZPq8=4MK#u!f1?&ASX7@D3zH?3z0OJC5Y$Y21SfHen!(-^7g9RB$LjR#|3uF>`Qt(cG(K{?+)4jW1>yju*I#Yuj_0=4LkkmFR?JNZF5nx^Mp>QXTvU=Uttb<r+<EHI@U_fwuB#ICDGiWt5?D-%2G1wdCb6Yg#Fg=ZpH-!FZ-C|KgO;xuTfQ95e%ciZi19wq;%5H;XO5sS4HBw#e=@2YgXM@0~$j0P=HA4SJR_2b!sD~Vn_kdNUCm!um9eZ%AO^I&tU$AByJhGb1S7U=8C|(<utVR+PY<n1n0k0>l<)}F}NgI2rw7X_zI^j9#Q@gcU$Alc1(#9uEim0wApCe+ljN!A7U_Y>T>km2`5O_*`))ZHhC+Ir9feqUV7_ifIO97^>&Q4-<SNt?@LSjN?kw)fMtNQ?NQBo$=WSI_tRiwTl{j|tM_dM#HT^tU`!Xr2@ltV0H^71FN)wi1H9b&d9%gvZimysSa=6seWMyRCe)Vp_6Hdq~_z-Nt)I5`|3k;KDab+Gsw^;B##C)giM)P!Ew5yEpm0hYkS1p%lWU(XO;9GdeS-9G0JZk26LVOj6Sw+i1`dI6<aReKj-{P{tv^>9*VD-bNmJ1Me#O11j|xvIzN$wX~}Qyr4p(NTZL%4j1a{Skw!{U$5Yk0h+Y%VIG(`SO<|-@seiu9Hb#@mA*y`AqH@9<B$RejSfrDGmx-Xom=Ale#rsg$-TvdYieO47+wLdGnEW6S{v4nxbS=c|_o7Tt~iM8xe7VB+B*1bi0p@VH=tHM;s-4P~6WVapz!nz3jD<aUTs)HzwTSBS+Q$T=xD4+4~_XwUBig0|P%&-nE0GeAdS3Y`9MM41nDCs@`+()NG_jM<7@|Iy%9B9^pU#5&!v$%`^@qzEP^@bKOMs9!3p}G=`*N>!p#?s)4h9hZG}<oJx$j4e`ivWTi5LN9_}yttEvI{KG?MD;<Q^?d^q}w*9ly672JfKdBL?NAQfgSZ${X%%2CpLt=n>Hlh#g4n!gpE=me)p4#Eaa>L3x*3zX#d=FKE0G(Q)J*GGa;K?V`QjP#EXYUeZcvP#v_`$8v7HN}#jHu%u&juusUN;X`fgjuk$a`UAUfCR22WhOz_x2WV=8R&uf?8wC!k1d!0jz4P6%PCN@cX8k%6`p5T7T0J=sTkDab@7E`bAvy4JE956M`xFlb5-Jrb4r*LeqAC+(HaMd4sR}13~>!D7_(%+w(Cf-~w=AJWwO<z%A*LcAQ~N?be~h5%nq`;_BegqWo&e-|lQ>LQo>s1EelxA0HvJ+_MjBQ<`kVO|8<fx0mRlik>m6p2kM?6x#UD!6m+}x8K~tEs)3!Z7_q5JjZ!>+f)|Z4TD>FaThu>cUwjm&SK%!<=5o=QLBtkb_0~~#lYKijB^-UWA^Z{&=ffJK(XF2*zhYc*%S#5Y)i`W`L=Rqti|1N+ZZ<i?j=(WoY`S^L45@n_wfse5H~@-Wlt+H+=Ghl<l{%=5FQ(AIr^|{;B<u}02b;d5Uszlzwl6K1a{z9EIwigYkJ8=toN#C*Se!X-0hKKN^uN7TTQ^g2h-=Uvp#s{jxl7=Hi9<8wchCrr5*J^nqU+QM3$jxZkEo0qr=nLfWhfYLs?dn{F2<IaD0bBsL&g`BjfqZ(4nEcA$=?NJ%-kaMaUHf$mR3e!TZzGg3$w@C>Dyd%0*Agc`Qs}wbbFzT6b30{yZ=c0QwBX86%%($sr*7JcSTjgfUo4Rw<i7z^5;TvmD)$0ue6@BO1ig;)ew&vQqSbf+*euIsN4gvV`DJ1BimhUqGTL0y)L_+<d!1t*^m00`WxP8QboOj+t1&AfoVk0qC>Mz@1=y0dv`+v`7(D;q~=(-1fEKfT3-f;e|m#8O)rM9EsWmhi>wx!%*AvL_X5?>nvdtVXH3_)acUP%8k|^L0L7>5<a;jHnJ){p!mqn@W`rg|2|=n-vBTmrgyTfR@<OY{p>X?adWQLT6pFvk)l?i?l4ggt+whmJebkdKpB1u43(V{F;xPwD;)hiH1o&S3Wn_ny#4=xT%W#n!F?A`;9bMzt36Qoir!ygWN#I-WbXpyF8lWl{N0V)KQYD$scbH7d%!hUari)T0LKcnEZ{o!vr$c3smGS8EBihsx9fc6LuU3EmFPg?*U2><s>}gr`Hc-IjWNa%=|q<>!6@PbwmQoIkCxea6h4%*f4?JNd<l!LapVa)Ccu$<nS#iVkVK!3(=+3L#k<+s*Hzb|H?24nWwcsZzFv(uI}z@+cxE1uY9j2$>Mp7^wSn<wY2pNh-rIu^QHQrFwB1+Ek>4O`>l2zd0~olD6!%7wS1%5~efLcOOb2Y(>B7c|`j|wsLSwL`INaEFv8HV95q*V7h*reu#@6Bx)i~<2zT~G{71XvVcU*C`M)@QfI%ucH8C&R5wSmkpc*9Q6h5=qR%24jT5YQ5%FR*d2*$UYAx;U^+>;)lSjg<2lOgXBCfM$pDknF|4(k7FY+qszC`BRMXC{f+WPCgX&94jC8rwxm|?L=s+DG@PjAhJd?Ji8Yt7L#IeNb%Y&I9XG@*{IXdd6K({6QA+<*z@z?E&C-W$gLl!I$d2{fN#$p2N|gC@@t%^IiHpZ5Le*TfhaG@3sKhHQq@t633wtKf0>A#Mu)xj_&O>vAY$z~JLjZhOqVmHnpCtUQrzj4YSiK$m)>sCOV96nLj`16bVIe610XHX1?tkB-AMI!ZAM4GNryVN!AI7;8<3r?qiNaKO1ikadZG4M#vQ0{(5FrcYBa#fd*RXp9rzAv;M&)u*ZWY18qAPVXgh7xBC277PzE+H)JRFy7P3UuA}TfcO!#)wmz@?<UiP|;R9o+|lQy?vO#c>9zt!+&Z8)k5*>rHn$#(Cwu668Z_hVhd>fx8`N|v3>m8(y){O4JHpvp68i1#O;tvZm@UPH)G0$u_ObY%395jg=i*(M@KdvZ<f{bQ}_pQ1pwPby<O3?=Q6&BOIPGYi=bdrU$6;9?^|;$dUD`zz!hK2-N==Df#fU>n&wv0j_WQBuwMJ+iwjoxOIK77Meu{`^55;SPPc*2G+A!;1WJPv?f#O>l2zawXZc^BO8SlFz%!eP&rwRE3Ld#Aa`NBMc0h+T9gs)rqqdDw(YuO*e~utmsD9c(yaZQGnH8PgdIMXQTi>6ojY5aO?vFk{zfsW~9cQmh2t`nS2iBlM)VgLA(ItkU;!H6Hj5Z)oKm_pT28H(x{Axm9d(14@g5Q0fd}x*1+$%Zd0no*G4TbE#7n7Aci`V%5{CAQU?d}R$_=fG)FqovDsze=%$Q4u7qQkAYrC4PBj-=CGSE{<0d5#TlRG5_PBIdX3##7j3&-7f&XnwCP3HJl!Y79igBgWvwHwJR4b*`gYC>;^@PEqQYSP4pv`z~<h5_cRM^H~a!l^^B#j{A{@Xmez%6c2_bufJ6cY-_yqLLi1@=JFnS(iXyMhgZe8EO)5_6)MO8`LFoFmJSRM^|d5jc=A6WC}_aJF6ICS7J1aN+_f8@YlkYTwX|X1@1zjd8F54D$g)Tz3glRbgXW36Sz4cS`F23pW`UrH?S9#RqH1^jx(#>YSF_q{*$Sc`pR&@&t<nCpj($y*qL%GZH7b>m;XWR2#7(<^Cmk^;(O8C*LJH!fR^O7<^PNHCrAnFk{C}JqaXDA|k`oY`fYN;HAP)3WqLmLjw~1VqSFlA5_)GC0Kd@&92mYrqK*$SY^Z=LsEwgg#*c8I=e8-sZCnzEBPh#3qN(YCTjv!ID*Sw(oRRFu)eHL6ZMb#HN|Ah*A9K@-{EF0;P@I(ifNj1pd7k(Ew{zOvYuC%;-ap*1QotWU>}o%>HyvIv}3x4hc7!)HP2&LC1h~elZLU{ijXAlxcOb-dCEV|T8h=df7&=9Y>geb4O~LJCg!*4ZjKoN^*Mm%b{stYX8aVwq$LH9P@TpQ?zeNvhX^;5S&49;01{!?(GIM0k~e5;PG|N9()RAtCPxdIdL%y#bkiB1x4oJ|hQV&jVQdDJy&x~_0{8+_WdQ-*bFekcu$rpl%)Oe^J>7yTXffEYL5o)zwsSCyVf#1OXv=QqKJ-TUUUwJ&hsL=K2Pm<#wS#w5Bjzo8#GGu_X~!-)^|P*v)HKKWv`H`b7y)P5LH2E@gAF*=V2}cOynd;<YL8<p;qoIVa=O<M##QB#)VS)=@mG>bb`0h!NqOrHs_wPLX1a;C{a!#Yr?tqoQVKzf&MAZoHMY^)7uus?nqv%UW&|3l3y|&lW1Gu~;k>FdnrNFnwrsNk+zO!GmcH+~xb0zH<ru2-phaBCPN+WAM3SiSJHf%QRo}8OQ^KKLw&jsyTz*)KmG2(EWQgJkNI0hspZlRNIumOGb#u5Kt`B028FcmDo8CrdJze17tFodtWSRE|3Mu&kt*)JusUnyyL4Zx>D3g;6B@WrvM4FJLT}2b%7-lCgceC3URLBE0>~x)6Olpvm?bS$h%NQzO>v*b_o;X3j&B187c1Z|Z!#=GM@z$`ml<6JuZ|Ld9E^PY3!rH2)QmC|rns4tXNEYw7wo!6ok__b51y-Fxwk6e_S5d_1tv5zhcr+%@z46#{W`<I_tGd3anxSAo!CohT-qV5F>4t%czz^-b3CMQ+BQht1E=-{tCte;?f}&4eTUSi0+r0fe-D#G&Av;x6iE~5AK{7N6Jv0Kms6xO24MM^Ylw|w?%;Dw+a`TZ?cvHlJ2`CbRX$mv3$OnBK1P<cDSNrkKkTy~L>(w0$PXUhlej1M<+m=)LIqcB{l6}SGt}Xpm+6btPK-OqqH<xz1M#tC&W?+&@i9Kz}H{~9FDSPXVZN*TBN6OxQC9WTbx{T<PxXazor^LTD&y7!VmrIG-hHaur!7Spz=X5LKZ#x}OhQ|NdQLy(rA_O`r_|GM4MGXeXuhXHFKhq2+OBRz*Kd@azt&?rKD@u!+v`f^GBUhmr)za5k9nuc8`sXBh+NzLA68vfv>*;KqdKMeXBf9{{{0-$NtFHQq$Rg?464HvL0P&};a@M{zca@eBQMwBI<1ulzUEi`|B&4I_dx>JbNl7%9G)H5&$tI$<EUf=|fVbnIy#tsBnHji#0B&Buq|xpmwTBdyCETN651bwVU;ylv(t<2fiQU4;ElM9|nPn@j>5+Zg4WrJexox_gUwIm)AsPQ@*VRG0IFg!l<35r<sTm{3L2)nbX(wHLp-~Up!fMgq0`)}wp(nuevk0!$kptrnDr{Ky<&4s+LqSUR%ZA5$I29PrVolU%EbNCnjnmM1Uk#60-D`)X(QC>r;ZHfnSaFYxWBcV!smaRjqazGko~6FEFaq+h`su1bD~%r0$GEFdhdR`^_>o^dCb8#_1{!*9iq*3??1q+I+qCZw4Rgx+Y<i7d;mxjMO>!vf?<9t_%xaZt+h~BQ1UB3C7}XUVW`&Zx=q`BMTi54-O?u$DhL0pX)#1or@>;?apjM78xy}z*F8mO~O?MEx7gh9nAe4>>3H>y5q%bZvXWjINJHdMP)qs)o?Q^W7FX&MQ2cQKwiL?6+<hE!DbNzXs0H7=UD9|4R><b!*z~M0;Sl~qnw2KapLUT%CSv<gMjeb}pCKNyu9IikdXJsC<2H6Ue0AN>D2pO#^x|^Gm%WavcE|J_0pb4B>!8Aq9$g<XioAJwws%B4%O49LKIBgR)8f4$~7ioltG`QuxnlXpMndmKnc!PINf&%D@q8Ry($_@pl<lYu#pQCN;?N3}ayW_rE$I6@I9g7i~iCD7if@28`4^R*7`deVu-fqu-k30@ZR1jYVk!cu}F5X(k60Z6anU$+L=@9E`@B&3bUX)&4%0HjZs71|+eK%w1T<HdBJWoAdPtVhMy<ElXbP-=<SD|05Y`R0xMO(ZlpnLQKarmFmg{dRi+O<f1<Xr|3u4Fj}d?HnNM>-GpSnrO1B#)G8l)DrMc3E~U=NDZIsQ1DiFb?ghAuxg_iZ~tW0((RQ9gef+0;`U7Lhw$>Tn|o-lhTB_9MzPCCez*y1w%Ryx{&N=9Y{9p_H+N|QeFI{VKJYh!e}ZJI;fJa@zaqO8Sa;B#GeLN^VeE}x(4ctJ79hLc4v;U6Vzg?8?tou2HXt0cGjF-=Tv*r-g=<_QRgu0Id1n#VSA4ptJS8{UsimJM_GSliN(kJUgz4Gf#sfQTwA<_eK}p!qZ+OR;M|wgYIR%V{+VDqlTqF?F~&YZq%8Y=^2+Xv`xL_MNhzWm)sYHY0^kPXp$OP5mov-y`{Ufa8cZ_hr5wgeZs2G(Sr?NzU^CqE1Lz#~`#T7<9wn;oa!7@7rAH2K6yAd@P=b{j){N{)G;s>kxc6O;;`%WsJhDj<sh6CoQxmUA#hDW0$sP{k#-l<FnN!yX6S~+NlN?mO$iSF7H^aOBRxK;(IvD%c)$$kKXXP5_zd6m*`Oa9$rgN=T;XeMgGT@rB{64MU&qcZ|hV!4?OPbw~t-*hyC$f49yO5~6I%%(38dpbAxw$x|IwgJ5I<|*h--lK;=ehP~xq!!!n(u*wYZEl7tGh#`@Q&SVFN8^>gLJ`UiWPP?GD5M)s2~i(VK1E_e}xxp8rOfg)uFok4DvEA>6#T5agxb&F&5nraA0E%PoN7}mdmi)Z-Rju{N~-8@87?CH~#SQ{fF`I-@JHf3!pd@l-+O?9!8%pK)@Icl&W%OM?n}OMn31OT4|1V0!dUL?QKsqd?EF~_h@)SS2awTb*J42Nez2;A9J5|tnC;KOKy5UfV5A)!TWiZd}QtDkv({`B)GstQhb<8afqhq4s0Rroys)C7hEr&g{it+6bcBsQ^R`im(YL+?o%y5xn5WqB;jX{8MIRC2z-%oo;r|WQEp9FQ0FkEawh3x!8%#g#_%N_(iLP1VqFfOX?hk`)f{}o1QWP)I<R58u2hOZeR7>e*{z7M{%I(m012(%omX=GF1e{+58K}<l;V{=XvE?k7(e)J?Q#J`47<=O0S}3pSdU+GisM)z0-CS{)I6NcKyC@*!vKT#xXud&?r>Mq2XoW1-IBRAu`x`$6s_;88DqI31z{tg37}Lyxu`BQHWD!3)q8q$hPA4K2EXuA1CJRM6qWUbgUsk7ZAzXV9G{))UO(&Gw<cSDF*QaH_&88IC+_jFKhNMf(KUgiC!=QbpsOWP{EgiA=O$G29idf#W`hnQ+Xmrn$pK&6iv>XJnGC&GZz>$>IkkNZbv<=v>yTJ^I5`L44$qg9^<}7xRE@bb4%@>vV!xhTVRC9rntnaW@p`0Ycx`PaRSk7(szg^$b7!qEVGhdBESUI*{BQqby$EL8o4^0{U&`Qe@wfj4+R5eL|Lwou1pnt>|L_0&`(OXtG+6)rum2^5GVta6Z~rxgEU16~xBp$n?iHRUA`rn@!%1=tLu(X7!sf8+mFn=-ULYRROVD4TnI;*t4;|U|u<2V`rlFy=F+GFA95_MiEhXLfwfh{WaLrx*0SX2~An}?L)M_9wA@7onRa1};%zI=wYj=WLiD~)rO&49=>jTBbtaitr@49j1GxxetkVcNB`O-VQTa&^;7h6cjXG%Dfh1`DeCF;hGzdUN@Wvwlms9Y**3a?Vla{UmURjT`T2Z4QSS3NB?1*xiJjK1rTWzxFBK`+e|*I;AL^VGdg&IM_T?FK8w+f=K6pPav-Z{NdEdV5^G(yjZ`KEtkCv1Fd=wc)wBsHWeFiDe_@jbFK>(@RR;J~osi`=L>V$c#;^d?ld0i%5|TjfGaJ1`}d6+%j6>HtH+q)nRp!F6(-0zWBg5jyj#V20j(6^k_JZ`d#yIFe6fYwlTI3H|dq>GHOz;opbe1lF1_9Wd~-zRI{o(mt!;%wo00FnXg!xnQwK@E-gV1R_+I7V;ZD&I@^q=s~Na<3Y+#@EcyY`JusJa%KQ+$IFkqbk<OfBh39575NG7)<YB%$QLZ}2s<ZaY;>~-tj7riNt%@Xr0&OIS7{Ha4^MkL#9cd`cUjEcH6hC3c19#DXqI(=LXNQ@w5&BI!kCyx!lUt!%ckyO@QO#CnhBf<ml_wsnt4pv+Mf^~RA)O3Q<Lt5)Pz9WiQZdMPU=1U}J#Mw#_wSv08WN%o@MM8zulyi%gdgI4#E%1tvV(XdMi(#seL`-JW*&>~q}pbNvUx;bk7@=#8}DSaZr&58?J=`Uw>IFfjx1iccit>rJF~v?g1bsKw2%FXm8o5NZac601J)|GiX&F4_XiF8k~fQjD613_uEOnX*n0Cpt)NP0zCJ!HvA5yD|9P@{TQ_&L=iz1tJFimmQ*`UqU9Wn>ce`GzQvhle^xCAIM)cRtB#6$1H}%?I?wv^yuEo(vOEH|fo7<e}BfjIa%AbK=jo?E;7m!IAy)q?0-wBX0SSNc3j(Ryf%hh&;WutNOW^n&_g$3!5)SB({bfE&xRWT_bRD7t4?UjcTt99*Pt6NV*derx~$dY(Wz5`Jn{xSOXP_C>tu0`T0@)vT%L+J`+rd*-g|KWU}7;|fIEG^QYs+X=_p>EN@!zO;^pMd7&tlDbN%GtAh2{O#1VJ)kol#{PVpk~e%tb=Y|%I30fd)0`%9#G~m73<O9*sGEza;hyze33IQc`)eKGPDGb$Xf}s5y8U=J;<3c436(;&dsH-!sCxTHf)-Ly(T_8k}u2-n25Y&OVZwW*KFnhMe1GE1ZL^nhCcbvsW+d4Ib=N+byLlR+-vt|X7o1*k>$1ZQ7nfbarc9ebB`MByJv64FTQ>AobOsQ60o#Xk$>&9y9LXCMh=&DGl)#y?_T4m)nqV!?b+|Xe)IZ2jrW*_UFrGP-#!~Z|N7Z~ZXdKFw^i1){j}O@$p28Ui)J%vZIFF$udcHzt3uIp2+u;t*R?u2USVwXav6vj7Lw9qC19TE#&Nj0AcxFLKpc^K*^~$8;5-fK>p`jh(cCQ_pm$qg-*D0JyMz&;zJ+`lTf-iS-V&cuzf%{pa_~44Oh7liL{kV}261+0n~LrB9P8K;$Lq2fIG&2D#@pW3oGH|dZ;z~tW-l^`H7=1`?>Ei)_?q5Ua0QT<axhLvVRd<M93RC+79YV3BY6j}4vxxS{(>$ytlrr9Ikfh9o`S3R!AY|b`E=4~%l=Sra$OXUpdoDrAA!OT<l83yh4T@f@vGCq#rX<Ha;&A``Pek^@$y`>d-h<>+oMKne?V~=U4Ij&(U0bJ)%G7|6ST~^<@nukv)y_65lk*n_K_(|ud3<(0q(h;<yhpz6rooqjn=@{4y?3OlwX}#fae!=fKM?t!mfxftZMXFue1R-T5!Kg^Q?hA?x%waqIcCHtxgyQFKw?1B@M<fsm4hzO^HKIMVG{S-Rh<&$G{z<FTVLEk|t^ijahDzyOx^`$beJ;YqTNzO^Wd>t4Vm~+OuhbewS+p&Z;U+Rc&fO=Z<oMmxE8if=%grA(SLt#@CA!Sk4fsD2A2ewOfr3R3-G@Py(Oz0H;!IZnN*07Kz)$?_IKP=KMI2aU7a%gDpr;AR{323$C2t)q)5Y(^Y{o$r?`$2Mz2H7cVq6jBKXKHl0Oep*`|snAy&<26XUc=#4fb0uNuiE{AvLE=QYh`qiZec}97q>1I0+4rfaY>&1S2sfVixV+nxH!Mb+6WwO44jkaF#tsi7108q%=LGa?`Z(conJ^t;xH?KdueEkAld)IXL565}|z`uDue)i2bqY#exo~zMD2^*Y%<sgh8*)d70g^2_7c~Ywj3q)o~Vpt3gi)12S1Xe2r#5+O89joNqS8rd^dyvBH_wPTvc=PRt{jX|A_F46RsDp(S8HK7=BMFZbLvZp=lW_lL0jeVl-h=wx@6Gk&C^<R2g*}=jNJL;<fEJj$3FHAv*&R;8=5Q$n4hs2^5I!)I5QVyNnt<-(#)3RC=rtxDD-3y&M&z@14@h~YZ>Y@Pe=|5c07*7s(ZuA5g~*7Ybxh!10AlCxtcH>{I-U~A3t8ucgyZ1HA5mKg*_vuZL)f<_*{B^UUc##7q!s7rIhD$1eyAAY*4z%KL;x`-2YGSB@TL8;)@`vjmKu-ymR;1jJMP;{V>7jHl1E1^QP9B9khzh3K)|Xv$w^c4T;C6WC>};Xy#JyPts9uS_|ld$y((UzFlR}lhz!%S^AuCKXmre;K9JH0G+$wRdUeGw<qqVzJ>1tHtlo?z>DpbdGdiCYD=Sn(T*YnW<#Vj9P-LiIr96xJwszQhHk&-sZX-X-gM)Q;Aa4&)CWI`yYf5JJP+L74RGOi>G~|)7&d5`W9~#n=a!P!o`dJ5WW$2f;qAz!D2lc}98**8L(AvSVv~=ZI)tWLO*>A&rmG|sT;41GWE9-t*`~FWTW15KF&uYpQRW<1!h6xzOz8f?9&`wDpk*#vFo`azb(&K6Nyvh$@g!%m1bK&iyzi2vos85IS_vrKm;kIMED<+*LTK7Fa^2o%~WK^{nGaA0(FM`@qE(Rysu^#df>_p6;h0Np#BFn<=qL7*$QTSk%TpbjP)xnk=G`s8M$qLjjQ007k0p2GU2b*-Ws}6fV9N-NbqvMmWjt0LvI;#BHs%5NjysafHd%#PM+pelwc?Vj$7$V%z`J<C?chf+@3LyRG*~e;V4nj4X;P;sEfcY8r3~?&CfT`SMpdjG(1$qpG3B1iiry%;&OedSkc}fP{@FD}=57<0}I2v8ZvDI-@06GNxyN0?|y#HzV{^j`jv*%yG9KU_`;cMf+f@m)WbKIh_(PD}ySY3?cZSlja-@gPdf~glUX`w{2GFp!mqbZ3w^t3@v##q#a1s3Ec-nWh8*dR|lzPyH5KOzTcVbg4{7Ekm1-OFbmUXEYC8Grxk^@}&(-!)KSWKe+>o9b`NgYfLONG3CqzXM&Ik><5&ishf~5!LX1QEy~63e_BhcavI+^-^+s>wwsltWpFjiBh~n!NgYme<vkti{<9_>vohBJ81^`Rj5SiC>zZ&a&(PHO2|!Kp$%9}r)ND{Fsnld7HKPfz?!yhy9S~CQk4!(ZxNY<MdsUYub07tv~Q=$;M20^dJ6k_nO-c)`U&??5zQ#gzhhrY)s>T`S#y9CnuQFfj%ss^05H~8R5Mg|R>gS}gXY0v@>TI4z~<Pr2kG_kBPVF%+cJ1UX2+-0(1DrvlS6*}1iiDK%H)V6dU8mwLZ8D9)gcOBYcAZItus{u+^wKBVKXfvP~6QU*>{PDjfFHjuFAqK93DB^)~yV8dwOkr`Z#)?eEc{l!!(*a)jPdC(BrXd7WWxwV+(c+)l=suANO9U!Z3S`nhXpf4}8om3zD@%F2>8Qku8KF^}OkA403yYTZNR^39<dGv7JullXzs;;dFG2Z#>zwPkgzmn|GSv6+y7CfKORM&_=eTplan(4J}LPhV#b(GYfomP=Wk7%H;$$L(3#suG8?RiDyiv>RS32laJcDLZuo0r_mve{MGHZs95<xi$-1{6J1&tjoj<gTGI-3XMwcr$DP?4^?uuW9!5Ukev&hymNag(Bk#Ja3UYANd8#36lc(BAMb(Y*_TPoL+=FHFP(ym?-;X}D-e!?-T;1N64OqzFEd7pLo_j<y=znm14dvSLYGm~iRrHv|Ts_foMSAG+$u@^c%SCm+0FhRVx`!InA(-9(6F2a}0|r$b-d+xC*M5tyhFtoMXmD9YK9i3S`i}4w@e~t^?y0TDEU&wyMH5j}Qb7APk$%l3Ep@R>k`3nEiX?%_39yJJ#-`ul$y-~qVkBTO0OYu}5fV1})Y;xY#QpRhYP0*_huKJP*-RZ$rMvXhZDhOjl<_Z;9a!C&%A>)u__OgfsnY)gCW2^-',
 'pubmed_efetch_snapshot.py': 'c-p-^%aYs1c6)vWEH3~^9L$XTsB22pEm<1htBNhDr15<e&7eRMoPh@fcmZ%m<GEa<vdB8Ce862~o2q0Jdy$1x`49g|&ZE20-Dr@M?Zj2qkU&39pFVxwr+M$*y|?LKHfhmhN$$Pg%)Uw&-q{yvGr#nHcXsaGWX+}b`l}buy>H8`sJ*0Ec#BPy%<|NmCD>|yna;1)@Up3+d-v`Qvemk*8gE`-50+KA^5$ipr}HK&p|^Oqz!~}NB5BelTczri_&)UT-|xyov`be>mPeZ=%hf3Bq^eVFA2($z`^o7qzRT9~b5_bv-_~UzKd+MJQhqLLo@EWs@@yt&c&#<5)X){+pj^psG%0o7R5|pG2rN0mUp6Jcx6U{95Z~A9JZlE+kbs7|snvsbt2|1ZDovxaJYB(_oWo}?slBuF!C;Xtu|eFV@0y?}7il;ec<>+K{B@PqX?2}?tLz<YUsmK<k$TvW_OIMDUb0wZ#f4X|lX(glEb45LdN*aYs2>3wO_CMfEHCF*-mKgdi=@g@-c6h#TY5O3m(|`&*btr;|AUU3s_=ZDo>2eV8+&*D+8=s8AR@uPi|iWz<Qe~4H%D2MuJFsO^0(9#*LIWSS3%0_4%<1_S&9&ky@_}i(YneCrk$!*OZYy&%yL}3CmOb6DCl4qY85rfMO4vl1pXF4ISnUAr_(4eZ_+9VJ&BFl7y|G>PtAc_AfsfxPK!n0e^Y!@_~D?x1+A4J3B&)RDdak{oN*O)cP+vr9dI{(Enm^-{Mz?qyQFF&Mp+mV!tn{BidI7uDc<`C8?J@>G*z~SK1{-}c~vrR;wqg3^#%M#>^Jn_@3=qSc<lMWkXGs9Y1L%&JoQ_w%(EszmI50pdMq<U3ve%_{b%B}w(5k(Knf5qMHe|Z0j$OaS})3m=pE<@4m5shiE@=KVmcF@;gDCp*H}%~<f$A?Kxu+@jxBghxr*IGCbOE5Ovh$w^=tb~f&vcVQ_<MxO)YRobuf}V541+@nd@1iNxBM@2sR$+lIDEXT&9(4Vgs2pv(=P06v1!jS(Nh)9(bD=J}=9w=i;443Hs*AuAR+OW4_1cS%Y<V@4!L3+oyLtWd96Ec5R>wqF=Ou-I<*mh<gC5Q}RshN!Usp6p+z!ljkecGlI&WoF08Pot(fw_km?&7fnu7Nd_qSLy~XOv#Kg9*#4is|6Ac*TxQ;%zWUN@F0-G$|E~?}GM7L7@VB*>|MbH@3A&(t)B$QYmK)SUg&BItUwCJR@c%$cABm?Su-oh+<6|Uo*MgZR#l<GM0R83)5QHaQw23ZKV4Poy*C-wRaA>_fzg>$rYP^f8+^pjkCPN~!)NbMElEXQy)ekL6SuSpMRJ2HF7con7Xo7Df-0*}!ll<Q3F$eYRqO5L>PA~J-Gx2;!*F^?GCAp2|r2H3F7pOsz6UomXfYkW<<<qxsU;OUn*>j&yif@TZxtmg0-ub*~{z&W|YASLoM%t$N4Q|1RHgDLG(Z+v10ZP5$FLW*PRV=}T$MPc%in}l(#BaO$06cZ5168`r-XYV`(E~aSVbCAJnS9{Em_D6y1BC)3ogp<+l6VobJRl%KopET5M>fRb0HFLf4}W(V#%$bLQL7L0220-y?j-)VAuUIKReMWIwYHc?7CcLXmRrd=h{6Gjy2hh5bG#*`N7sHP#43j7kW;Ds@Hm%A9n1H4=8M}vkl=(je@a7&XE>^)P{DmbDsJ$A`$ZYI_z`T<7@cqu_^Yh0LE(=HIMtHb9HT9n&(n32E__4v1Kl0?Di99-LDHoDBTp>SSF4l&!n>9?N=n8*=JD`(3D}xGKwBMbc&>kiw<$7lBw#QRNO2^)Do&SZn!+yJ=4Ao^3L7e2#0YTRDv~q*@dsxw&tIJX*++be4$%fk4K(~+as#Es{KtRyxVLw8`Qv|p<@DeG{6GH&Qe%#?<6pd{g3l6jWr>@5b{?S_hzGE^kN^sG1qUYn+%i>7X5s)5*hFM`z($dEA^@dRboS=Wt2e^hfd)L>t|bTqp86pVfb;Q4>hat=RDzafM}7MPTi<@0>PNOS%aSlft5-Ps@%!Rp<lV_Jw~DyLE&&c|VEy*4iAJcq0c9*UU_~H%<hO9NigK0Y8Hu>N?U05D4e2XPK8ekd&Z?B%2y4VckRg?uhQ&~>992g7aBC?-5{Oo4pcFt7-4yA&H5>w14{orS5J!bMp@9a0#oy;8I9`g4rchQEvqxJQmTj3$(oR-Q(|359we3N-0X#f-4OwvwZx&(#!o05?Lm=w_h70$>FBCh3dcv&4&^PI|*aSrcKL^heK8;)L15Hj(QxmC~6I)~#fy;#S((q95+5wVvwg*Dfg3-u-f(@qvg^Io<;1-O!E5_{I$H#rYhl!&)E7><R>ep>-zWqf4OBrem-ie1D<0(#0YI+4)yH|j+{=+|PyqQwr=Rf`@Fc?(cA9?eizW>((RIs0Z_{SA=NPsonM8cCK#F*kGYP$x5!5_}vynXTNC5q8Co%SP^CP&w&{^0!0)0b~wzj|{X_cnv6iW(htN7oO1&-)PViesLMsEy<=py9TLuPg8+X6Pj)4{DuU1M|B+WuGY!&*1Fk^VhFlygbKgE}LdukB*O1@^41Pe3nI^Ek;Fl8C{gu$8gfC^gHqTn4yW5^NY+MJU{#5>DOPLzojjw!Q(pn4tM(W<OFsFTzpq)h1+rp-!76h?s-`(@JaLu(n14twr=qICzi-Y*CzN5kvL9IqM`TMNrXf)+bk|nGI$@I;8RdFirAdl8fTTfi^(w*OOGc`hR`j+7?~yWt5}Ws8IOr43)L6c&gwQ^Cv~0HJtM<vU`c?Q>oi`1y9=0uKr(^|W3>)ycyQ;Z1v;-6l5)4(!B<cJ6u&ux&42m!EdKn@=Vyq`(@(wofWwC$i$4R_LWoHsX~VS3>>}VlLZ9F-xE1&B%Z(X&_wV!bt(3TELx_$Oeb1($G{XT_kL<U`JM|uqJy&CV4tson8i0QMwx_q+)QvX-zcMNxV1t0l4V>jPieR+J51mj-GU|ZXX@F|P=>b4(U`Ihf!I8;tJ#=aC2z>x3GVO(+ZQ(sW2m;xmMToa9>kJ7=B1S@kEchw~Qw?Z6TWwaP_Xoyt8p3e`og|RQ1HPY}hEqcmA*ewu2Od$()!JZSI935nIrNsGG&bN~6`PMSiYTkWYzJwew-2?I<9ArH0L>CAU8U^>3wjykxr6T8NV!4i#Ct8pNgkk-&pa&x@i4@_eu$8uL1QiM`k*CU)AMVj<rBj4HKJCIf50Y6h>J8`)LwH_dOUKwmBnT?1Ipa9l@=4jhyZS;gLToXesLDKdDqZ8Q3LMaSU9*zGx{muop+M|uSf*wO%c0qVP)z)akC=r7&=#v9Y5N&n-U(mDI8iPc7@sxkLl5v9*^k-CsFiKr~%!3U^-&PI6mFCM2!c~p4CzD5kY|%DwR|y!IX1qGHFUdDfu;Cm5WWD24UouG-g-Bbms;_PLV?u?a>&*jsg5H4ig7WVM-Z|q<8d++^pn7H@vRLf#!D&kAI7<R~jV3XsN=!cy?LVO_4x6_XDLqb@Ulu;|24}U5FO(=&#>=iS7aD!}6E(z{qr5&{z>UdV<qXV4;He@Vp7GaY%sx;nG2Sh94`gKwtwm0LSFB&tJq(U%!a|?(EN~FVpw4ub#g6Qumd@8HTxGtP=c8#dTIe*aH2UIF9M01@v%Z^IQs0@K02w;GqMy$?c%~gket?Q}Bz*w{19CGz_hlI!18Z$>{Wxlar}J2cF4f39beyw@lpV#wWy-H#x#b3|vQ+f%SI*Ds0@C+!y8%lrPBVbP^3TTQz8Rdg!5Rn?7m??%2ZCq=7R7KtSppsa%4nC~pFpP5?Os^DvB>vPp7<gs@{syMY^euddSy5OQO~s(}ZqWOS3gkb&|Ze|&PHJVGjJk5uK=<%(dey_+goLu4NGk?SmlFtieF=)m`2gh!sR(WT{#XAWo$cC#k41NdaMN(%HesL(wp7F;%~T&I3G;`L?*hx_s5@1IQXKS3h*;2&%kft6j)6OcO6XSM(=Ez-c>G|QvkU{gVk5X=dqc~Ym#GG7F(LkR{?lZu$fH?!ckBLVhbH^mj~L;-(m@NNNI=_355egG}Lxp@E|!n>2<!!3MO>-y^kx`74$%?zeR*uzfp0V(l1+0V){4>~XiWd0TZE?_SFjN0EVd<Gg_qe#}KK?X9<LC|=gBVYLfr1YjrgEyN32hcYkH#&}y*y9)uWxgDG=`|!*fDWYdiVb=5BGppL5EgW-1w2AVSHn71n2sTY9=q@4kP~4oPeY9j<;zIEM^BOXXf~juPgy%{3y#95H?Vd>rZqIHKOpKLH)7H0ZM^Ln0Z;@`J37l3Z9hIZ4skfr$(jFPvJG0qLasF^yD(!Xh~p)oGmb;d#X+y&v2Fy$bpZ$M5*(E4TVK~5a0mz(K!OS|gwg&H%WA!;<91**UJX7fbl@0JM);H^GK~dU9u(;ge75Jcfoc-5QnoS%y5sS49udb5@Sjit5aH1U3nqOB(+Vu|A5(17BSq~@mCy1_<<B_idk+4VJOyS5hiE9&4s^;w#Gx$47(zCrBv63rLEoqwKbHoxYCz~l-CquD5hG7RZe;#MvQ>Pvqv}#;kKS^NAasrvU8abt8hq|3GQgwvHKbD=J-tZDW#B(8=7^6!!fC9JSS2}PL>-?-C&&<=QY;1Af>V%uUoh-3_=5edHRT2Q?wGVBGLV{BRgzXljLjm<R`bp7u)cT({I)`Z2n5z>t7B{(m~(`D%-T+9hD716xFw-eA48o8DZ;G<u^^S02B=O3if-n4S*HY(O%M|E+#(!kSzu&qC5u}@Yzz2|5xH<Quwm4YaYWKGBKaSBPsXa7e6uQ{by;NdAaof==PVvf{T_1VV9Sr+AcH)5idqwC2WV(A+m>>e_5Vkt1%VEk1MZaT0ROvzT&&c9@UguDTa!Q$fF{q*mDqvTwwr0e{C}=hx)Gu}Pt$b(23>F(4R>$Yu!TI-Xo01opaWrVCaZOxf?pS%t@!u;ki%g^Psy>d@JJh?Nj{V!hresI02Z)3lJZzBvI|&ofCnpDY*y<!uz;w6AIH$K9-mjBa)K2=ijuBPwsQH-7yU5OZ;nI0dVG%Ak-;LvObuY`IC5Yba`nWAKWM;GyqFRd^|3<BHQ{R>SBWy^a2M7QeGN&ilbXP&#U^_84H-W&5o^t-MEXdf8siGma~^~Gj0YcHcO34}4vo@SdJvL^yd$7@VQWCJHC573>{;6|Q0<YJe9?{g7qnSQ?#ieVZ|h_BdJvc0I^|V#Q-KYmaaLy5tukgzG}Sb}_UNPcKX7n*3zkdRc}NB!XmYUlO29(kWBx1=72snm^zRTOdtkn%iRI?uXAHAWv=S&Bszzv&dY)zC0AOg?$RuSRVQ9VWs@kJTQF!Ocyx;O{(glWSuPFgxs|Q5+nAP*kDZd>P%?D0IUd&J^Gs@X=X+YzlKp0AActSxke<-s%C>)Ix3s!_J`&slqW7^P2Mmi6MtwAQdikjm9&mHEl`{V|+Jq#EqgGKwg^G{~ebiE6qBOi#?JW&8l4Zp+$d$oaBB~r%da0}SLhY0Xeq$$eRMz*1RI_!ucpdn40oA}DhrdjgLdO*i)MC#mKi$7E@GgFChs1~~6Cs~upi9F5FAbuhKZB0a#tjDf@3I#(lQ5u{M%_V5Tga_b~?QDup(uUrNPCu7kbktoXXes(I-94SbdA>3QYvo7<qv<rgNoh<*OX1hhg}*aJfW4Gp(1WEeH)5Bf<=pwM=E2qmH5OvMN1&558q0J8U95LSK+k`s;K!|z3)1Ajd}Lwin)8V7d^kmyCGiFbpZP4C;F}EFwvkhKC_ZU|oD)~vz+$8veIIeZ0K<TAtODvtSX+zL(+KE1+@%t#EU9&}LFK3a<kTCmU}`*hGoOrv;f@nTBytJf98`h8MNjTO@}R^5upx%zV+?}lz>LI+KN-qgf$nfl!tT+6#T)~N>9S7&-lGJwrYpSp>AOq78cbwsw`QQMfn<l2)ZA-LjRisfFl+-_#vxH5weGMVdRj3S(A^_+d0%D=5Wlo}m;wc30{v!?cK|#D#-U0cNSqWZya%a~kOA-J5*AZ>0HNAWL(W&wjWiTHE#0%E8S|f^#Kc(qJ?Ma98VyojtQMklcG6*3oksQ~3Cq2eDlJkri}*>5YPfszEkYHN!r7L1f%FMzCvq$^7%}Gqy}KBq8!Hf`Gl5L+XWZnd&D?fd-*iA}Azcr>dt}X0>l>7dqTrlz%99uc6Afbldhn@CZwN=d8%I$%AQg=K(s2C_1hmYt^%Jj~Q!&~}8M=S6%N}-e_+5LrH-;zCqO4uW`}ivR9PQi1W}a$!6D~HO$7yY)=!xQ4qXR&5@*@j4SgOvUb^*lfxvd53-8CU1nDMg?%zeWi=4r6?Uc4hA@ZrQ1fs>Pd6zCf-x?qCl$0xrTdLKXhY>WI3LRiSq#p}o3r|=yM0Oa`}V|avAFL%hLA5S(Rwa}0vr<MSa^o~J%LOAAQ$^}H_+iyW3-U0CV8d@}%+c`<kUb6JR<Az&*a4=0T{okLTzxHI$N5I8Xk$GkjR3GVd)8=3$?!;>?akqn1rA&0_Lq^w>XZ8($IHTtvY=)j5c!Qkk*EqY3!)b>k?A2cN{h+N7evYMoN68eZS3)2LO`<s7LP{7s*g6mbo2BKmK%-&nP);Sh;9GwCr7IZdq;HGb+>8!WmPzywpVw7E$_9TZ$VNuza#1uRi&rBOZ8eJVAb>1*jOl^s!F+fq&O!`Om=A<Tk5JtUq}asIA@Zm8?eN{`?t$A+X*`8!GC<_e5lQJYW%}sXVs!Oo(&G{|vin_Kg@=FX!xxdTd3-0voQxh$x9ZzAy7<n0#*K-Srd(z7cy`;QweC)mlMSijJwo0L_^8`+x<bJ;KI!3shK?i08n=TAARm76=~#T^X9UL>H3EqWQ{^A)U&6ueX_bZ7XP=CXZ{6WA38!2xF(iVy0fx4%=jct9CG-%oGQB{$0gk}L2Rok&`yu|Ff#$TL6a&cV+ZQfD`%-c|lbz!?ImP2%|3W9MqSt=$!bb8Gv3f|hzFHG9IIP(ZMB}0NQP_J5a|nHc<=O4rQD2muwQD>~@~AJ_l%3jVg^XFZ9b$yBbcezN@4yWdi|rY|&ws0;THt@u!(t;FtZnGFi-P``UHwaZxBGp#Ddtih;=nb1Zc+uL0lJNZLyq3sJ}yDjqdsTOo!>h1TNnQRAjdA>2&J7L6fEfPFS!?$Ue<VO<O9rJZ#kOrUObxa8rKm@QF;=3f#he?SF?tCFe8X`CkaOc{#||~(*eX_>d&V8>}(a}!M?0X@RLyN1aRjKg|~@`lQOq;5*TLiZt6?6$p@}j@vjnx6_Ew$zx|c`j^xeeK}QFwU*vT~SB$z<1)VGOsZx%_N05?5p-&kHMJ0qhG8t4^mSr?>NQqW5XfEjSxKA?da|pk$Pt~RbAcQYO2CGzTkLfBN+X3c>JE`q`<+tX9l^@!C%|uq91!k5BuSq4bwy8OkhpsbYvS568w7hr6y(SpsnR2A>UVp}qjaYRjgDAo4W?L8uumLt>{AP$Uul<g$<B+oD105o{DjC^2AmzAC67pU=KgPURDD>sjS2OvOs>c|qtK=%}AeCokAVmmIB9|-ptmvBo=s?`ML5^~xvaBs2Q#)eTp=l41y=Tj;EyXdY%oi1pTpDI_Of0g5nyp=xY<0P*=BcVvgY2q=s-R9(6%MZ4xI;whqqLwgS1|J8rkoWhpHacKR`4z77Vo<-xw*9q6=Dd<<tdB$&~iHGlE=WE0%Y;E1Y8TC_QD(L6}fcD0NBrxWbWSwWKmg@I1qo?fY@sw@Kj>JCM_;7^efWZFqKW5OgO39<3TS%aFxQ-YGFg{nkZnGCGNc#5rard0zy+GplW^?s>NgiHe*gz($m5unwe#qPT+Kh@<S}uGrY-m9vKup;%=tRSSH~@Qf>~ph;Sw>9n&=#na_BT$HqYh8ir+pdPp2)-1_*HB}k0H(4YprwlW%tT6?8Yd2PH)y_c#Klod=TdO&R21HLpf93#NRU6v){8pASu#%%WB>2TUbct#qPWT4N7(gqKdcp>%OX^4iBb7p2*ESrBlF#4ubnHH0~E6`L)0$uT;zvA)+5!w_nO)ZjXDV~_JU_qp^hr;bn1+*XTsZp;(M+e&42OYuGueumI&!#3ZmpR@TrVb$HOf&bI_#L*Q>4~z9U|}&11Mj8PgkQtn#tPgq8V8uQzmvg??dv<3WmM_6RMPHu=4pLqttug~Z?VER>V5^f-bipMkFF&&3ca9WIr6)O(n3=}rgO`LYN=oJOll1^xPM}l=4B!D3O%cb+!j=egYZfdj<g?PyB(nCaGZlY&k#c8F^~m9t_GyZZz#8SUc!r8E{`bE8?V|FN(oAC&>5sFh#gLbRW)^)U#Ee2=M>bEXEm$~)O#$@Ys$bF2S@O=zK3XS2oCA2Tv$;fsPvGdpP2WhRVOl%wZo%WjF|$f?*$=S*DVR({9sQppowxZIV+dP>k4pOp#NJUUjq7}_td@zG(l=wCmCOjlcB%XLq`TkHLOfI0>adC+}oQe0qc0}4}`nfY7+NPkv|hlPlQJkH-_v3_JOt+m{`79x~d3Q?~p|Y?iJLUT{;wN5gUo^m<Y}|l+&@7WBPvENj3;4I_x!pjH^v#zs}X1hWfVC0ubvW3fiUghtf<i^FX?uwx4B|NjUc3kU^KB@UnGOC|+dtQIF+KlY-SjXpEsk9^yna?DQJBdWoTaC#QoVt2Zg~BB{M=To!udw~xqmfV>G({X9~DVG%?$n8*RQmHz%5nu;pNs=R6MT-BzgG!sIqd>CqV5`1KCV`h0GtIni5v7Qz&ICBrBwevd24f~;BgTKt(oz`4vwy~ro-)N*RpR%r=qjf;yNdtS7K;d8j!<Y`d2{3LUoplIt#ab)`%RuSK@qj2Q4HN$$4A3Z`pz4{W<akgUp#@-Uabmj}Y?-1Bu|uS|S=AdU`#OS!Z9rWV?%=2dSC^&kVXJCKx=eKswEH+Ld99P`f{JRF_w>+NXHpSF>paJB+P#6zos8%id+&K7J8Z_Ryc;u{iQ4FWK5rr!hgSZJGW==K(539_zHYf7)5G0C(9m#X^J}G?TORGn)Hx^bXUaf}yP*_oYVCp#y;u2yZ^D6lao86LqUcuoLKAV*126_~i$B7H*ZN!eRQk2n0@yxU0N|0ijm(YFmw?HJ<gRE8!;=<j*qMe^Nv{nXGHh)a3rSO}W~9GdgsI_xG7*OqbyN$aEI)m%ZlNQS_p0!BHOpPtwwA&K0dF-yr)n{AkUjw^Yya+Zl0Qf~k@7DkYs?HuJT;{#q=T0iaf;2xaFviwYOS8A%UewnD#@oR2zxH(8E6UHD2VM`?SRq|#9RAr4cY0jkpkZ3aI#(hMKU-V&M8?PIPJ;kLu&2rgU){OLIsT}jCWkCFqOIhIdI(@E#}?o*SSy+;_c)p6iuK6w!6~j(g^ps{1prZ8dtD(hF`mr!=eMd3UG)1B_QaKkwKp7YBZF8xHKv#4|)y`ltSj^Eb#C1xzS~8|B>nO_zykX{uP%Ri)%`(J(XLX?Tl}PQiq=ky5`)8MH?qKHU|P?6u9Fl#r)k!BRgZ!gRb1MJ_UR65r?k);d_T%eF=sLA1bKuwk-*`{mz28z2r;AAjOMAqmb@HtyLDO-4LH9Bhg-9n2Mb7;!dsRIdIiB^L!BX)2qFWTU+Gb4NT@`wOLv{%wlEN9Wbv!rv~Tl{F!>^1kiOUjM|4G+$RCd2tTNMNKe!qn!;d!;<GiDkJMCKO-oU=Uf=}4Jb)N4)|4WR+?Fw4t)v~L0<0dWYtP%+)hm#m8@ePN<!CUWaWu_ZD@k&c7xiA%)B681xcXS8Vj$jWJNrtA)S2qSsl}kp_%Ijo(lShqSKR)tpLuE(txFht+3>BdhaY{*dcfgF-P<hjKe4ega3vUi3r6?mZdITFH&N$#tzGoxxan#NM>cRG5?&Yx5BC>yI8&sx`~WIzFoEl@;SfNHs5dTiNMSz?bRQblwu_5giDj(z%lXLN!_%o6$F?rcsB^Ry6S^}f+bYZ61WIE!N8$`m#9mLy8#0eNmwxQXKG`V~I7%ToGa3tc(y<^|JMr#x08~+otP&zWtF9P<)oY7onu@*%<@xTJ$13kNyP74r@0-%1{AzP`Yl)iztx`j08<k1&RHTm_-C(Vbs;&cAO42?s&$=mvvj{Hep*xk?iQe7NUr;bh8{%9rCgfT-y`Xm7<_2juzuA|<^#+0{A>TgpkJ}@U?+;=x&}kQ&>Pciq-Yf2#rZr4P?Q)KfA5Ptgo%FjoaK{UBbe*O1Zz<RptF=gmCOF}Ae*x6{j4G~5rjgx<W>F*VxtGBA)O!3ZUO?CuzvqPUY30dK$UL;>%O~*E023VHZol_q%U>?&_hG~Wr`tV;*eNgf+3U>N_nYg#uE~Co+3vPD+SLj*M%HO{CBIXQpo${Zh9RjSW~UFm^Got37HGoGh>O)2>A}McY91_AW-0JB?BwTTqKdq?P&olVHv*#`=jlZ<zqQ6j!N_-)WqFSJHFChL(zRr`&MZOmnOFsi7GjcEAX&+{6U2_v$f`|MS2BUSX$FM>Py-Wy;;%B&7h{Da@@reFJ*Q-MF1u^7d?Yl;HkEDAy+DstT)OpN8X)9<8qvR+QmgY)p}aNXw<WkZ9kg}Ra&q1sZ2t6%F0R{=_FU50Ep->ylcGT4yM1huQl55RZI=@R+Ks~ez1Bom0Wx7Qz5>LgCxa0<#w%ob=YFmh)EeFJYe6+!dUY)>>B66DQTCKuDXD>e&}wt_wd`JzZUIf{z7uzQq1*Q`SGQV*s+QC``%HA8-HvZTd$gCYI?q<PEAbXpdt=j{m{!M*>u{%iS@OEI>a%hcOTEfBCg%i+9dq=nZk=T3cG{7n1>a6<8iLXZai9S>=+&bHzosz1<kxrG(7wWtV{fQ6SayPS;a)iHO;C0%tz~7@<QFnB8e=yb>(qM*JyG5lri^luzL0hRGP~|^Bbp&hlhu!U2pWK&r>3(>-~GYQRjITa4}Bhs;wNh4U8<B~PwMLW1C5&xsT%v`*&kVig20u$O2!NE-bu&>F6Q}$$TblMR%WR`QnwE*N{=tpC@Ae6v)ym9xe&lYPU`GSL=krGleoaA-^nj;N$UO*jQj~0Vf0a)7s<58b89$^gIyOgbsWa+Z_S7%n$Ovoeu!!&B-^iMi^QY5Xh-~qk6_Y>djLj;yl$iQVlZc(h>nbjo<%Vgf)QSH9ODOi#7CWA=dOfz^QTaR`9X2FMBWHR38W&%+y=gJQzQ5V5sZ+$lfNXIsPq_tBo&0DfWQ2mr5o)x?)IcOs9cfv%BK;%oY^$uL?}Ghot!J;$OF5)N^?pEJceZ2!hWU8$R~H(Dg6emJLAPjz}B)Z#=A0hhK`JC<#gZ0Y1oaX1FLF>j8ihEyLP!9yfZUB4x6g=z?ryF0&VE&9v!2V;p*9Gh3d)H`kpvSthMClZc7zz*Nj0*+mXKobTyJcLE25b&GeL8y*Q%%s^zG{Lo0LUP&2y~7-m+h`V}Bqr_oLKRqIopxuJ9e$gB7af1Z8Up8R<z$OFn-FBM`4(`fkD#^4b1ub2LEgh(K#1M?a-76W_*n@T{#d&sd5LO^%0Ia2zG-&AZs@o$OUP%yS~W)V{`)~E=W?w~?7p_$kGp3~`_51OrY<-X@kvg1Lsi1g>)NjfRhALvP7Q~lbz*=7kk5+*&D(Odd4y%(?0cqeDOeycv|6xa}cYCd`n4DyfkAm9OpOoT`f^_c0=?#jL!zEvl1zr-a_VOJd<l2ina9-MF+0Z8cRAxMxFkOB}WUlNyU9M#!{<Ro%Xsv)){Hf~KTYq`o;Q-#El!(8uqndhR2N8XkF3{}3gqBf9kf!6}=s$SNAuCHcqzOwr<U7);h6wFa+G`^zB=Yxx$JJVnp#3|7aUq4ugZZtExl+;g&ALM}(g%>dDDiK#}0^aU=!`Ol|7$Ix~sppV<pdAvE=(Q801-W+&H9lx4P5(z79Dj7@XN>p$(f&4BQ?8*~0ihq~{uOY0$}J;}Mi6J;GR$%Gf#66f%8ivjh3>7@X-eFzspqueMz=?28xH;#nM~J{',
 'pubmed_resume_snapshot.py': 'c-qBU>u=n+5&yn_g`E2!>7r<J0a_Frs6cw>pe~ZsaGb*xTSJf~u9s)MKvIsKApd)3h7Xau>vgU`(FKxSQA2Vbzj={<_UzfG@kUe)i-x{^h}4!v_0Wx#;@2&UPTw+BZzCnDUXFuMSu_eJnYiQqO~mfGRQTh!_wS;rZ=$Mgqoz_-RBu^*I}W^8GJp2$*@AaCIjW?xKX1P9(f;06a@+E2`>4NQAfFU(?db1v=ocF?bP@1|<c2Ofhet~!xBe8xJEo^hqQ&*bN=Rns!I&Idt`>{)Hy0mIFUq%{KKy)n_RkL=qasS;`>rh&yH{}*#qcGnT9ty;gJ^JsRw~&BwG*d{#iC&w*u6?n{B#uwrY6x_$#Qt}>Q8^xFY_(CZ}<(9DoJUYU{#Z#Q1d(#HgAWjkqL^Y-{we2JSD8JhXyoX#FN^*_)DDXl`1uKye8@_x*nQ`bhU`!g99Mcyp$V7m9Spr-EG4~q90^&DJGVYb<5$Fp7adGx)Ii)iJ!UJmU7x`_&u$e>)+4gTy<kS)90TBSFF5#P)sJo)BH-Xv8`%mctQz?*a&b}wXpFjkSZa~G*dUqgx+VYmy=*+C2P)$w^b`y#`^}uC|;${<A3*Ynv=Fzl3F{F@kCh<J()G}G^GS%ARz!%gN5GZ4Wk8SWh3ck>TAV^UIP8sKJTlJt!N^0m-N6z-A)Z~rj+2<6Ns<bOQX#oqCVn1?Uv&X5#K~y@?NT{2YvJzuJhJx@C)0d-)Kvq>Ia2xmf&`{#LNY)YDRYvjmWAvVE<Rt*{KkNNaFWgO7vdi<LJa%Gm+IyZtQ4K`ITgJJFfBZ>gc}TjK&elWi@gGb6Ql6tOnJE-k#%Vi9TCyc+1Lms4Ltoi6QJvopZTthH*DEQ_J>O(&Kt9Ts6V}Qnxj^7ZW+VhYdAtqQ3%}^^g$*5XZrXS&5FE3=@Py5X#5{aqa>f^aX{{J0|#sYu>X+LjiSNR}dPTNCy%L(Y69}3PxrHZxBPxqy&Oy5=gFvrVxr-z>e09;inP5>iGtO3A~^{0xy=a9n8^Pj1dWE5ZQ=8R^8Y#dfmn*vS=4gEz&TWk!PF~cDc2gSL%2cUmb$g1@RndN{WC{$7NT6*({#)J9M127wKBw8|T1Z2&Y5Nms8MA$g`16q|GI`*qEeOTF{4*H|Ugd9rFg5)V?8cMYFSWr<U~z(Swxu+q_RB<idLnbii6xE7p9E_-DV!Wj##2+S9?<@rFVNASlNNja0}=3GfYH@m0FVH?nN=)=ZXf2dU5niy|JU>y9;fUtjaQZ@awb+x%v@i?zg}`dktO3Am+&A#m|6gMUPWKxcOxLMTKcO<@r3r4mAsez}fuRUAiF0I@@R#|(_TphY=?OW3mouef8S8WN2~m>zo*(<531Qq2KOl)z#nrrs*l>_x9}L(rNNcjg~In7XI{$%heaOo=5H*MO%=MgS#Nb;!f#LjNZ3NYG60-~3iyoPIoq$aY%(?O&ItAD{n}21^@A!Q}L>hi4p-4kIwTuNecANdAQZwmYtT`S3%v70?&xpt#X41D4>V@D21GE{obd=&$zU^tnczT_FzaW*NcQtBR1dj6j~QUtZ1YxCv1+)UYem8yLF6l^6eWaZGr3JUHmP7~fUn2r+oy1+*^~6gKi3rVIuB#DJ3qcr<~4%jm-ZQ4JlMMvU`Xsq}0dI)?wbu7=36{B-f|plq;fOr<ON04-1DIRVD#kltU-rBTQSzO^b&S34MZgg2b?q`tN6x_$yxkh21lIuPjCWup?rgqwD_PU0U;7F&6XAw@A4&bt8s5&{FjfdB3r=ZJdWoS*qB#tH<(>hkQDgEAQ)JcgJ#W`Y+K2g1CMgFpnIXV%|9hKs>Kv8jC!Lq5e;sj-C0QB4xbUcG#|OJNj?+am>jf2UDVBL+c$U@AVTJ!}~yKwCLtmC^=r%DGjsn@N0<!jTc1$kwB3Fd|^qvZ6w^>vRdLC0p$VLRj8%ZGCSS<VU^Dtp4KsJpjN8au5luowu`F!J~DvedmhPnS`sB?a|QS_qrZ3xWw#1K6r>hFWawe2*BFt%Ir#nb=40&6s>LX@(~aH|GB2)?T<L<lb-pAuNBVMb_@Wc0zpuC`t(wO*6ChCcGG-#RQ-dmfLn^dhR%$T0d7hA8Z;Tjip)=dSs1H#5OWkYPNmD8xOYgEj`(@;)A8_qjZ@h%j1a{m5TSS48qh;Yjx$dvc>a2N@ez8;C+a7fADOAxc)7DIUz&|e!_;Qr9NXJSs3~?0IH!6Wv&qhYVtPEnl4j>fu6FT}D8qJp?le|@FOtPB0>>+7mn?=_7JJN)_{B^RNM)_%ErruOT*E$i28LME-K?c?wOVT1`$?Z5>W;W)pG$1qGCW4u@S809F^j-W!32#yOLG14KL(H!T^eeTrKZLbydtNBO>qu@jEmQ>Dm?4sA=^DKfsv&_+fu?(7@SR)CjFkd2s>s~2{2{hV^({6Df?=a+X2pg9%9`g;z1Hu57gf18L;&@$x?Hxq3xf|7(rLNg%220r>#}oa7i?AaHtb(mDAxHG~+AZH+EyNCDLcF5KoQ==_j+v&sC3{7;AzVT6R;_57=Z;&xFSh`{KWjCdsVD9)hH!5AL+SGmDG#h-#1qb~Bp!UBkmuzA=~Kurvgo70ieZF8+KvEAR(q?J+o%v(4{Jq+^g*YXDz8xtDJeL%b%=ZBKGSt(lYeP<S}&6%%3_v5K<#)_Rgr!#9)%l&!DOJ%ak)K&5cI!=d~gAHXj5ReTlZIM7A_u++B5Za51>nb&y|HHSpe@R$Uhe?e-g`Wt-Pv-^<(S{vYGXdBX0LHKoi;4q5roh|H~Qh6s;T6qD<EMu^wwNiG_cK?;Q#!fbT92FSRw%Rx40Tb=(=YWvAda$&Roa|~r0c|6l?d=Xo__4a1YkMct!Ojkkt<5fYE~fA{2oh+p<K$hG*m4}xfEOb7Sh%$JO-{vhsw$@u2jzV8WIK?lmzF1c9-ZB?2eke=iEGFdEu8rK@jaVIjV`jp26uGzJ#sr}srL#k`heJi4}y#+hHJL7h#50$T&;G^M;?IOhiLb#xdyQgo_c)hE$o0(y!v{?cB?3UPCG1IRUTf4co5osHOI6g#`7-{G)Fo`R9G6gM1fpprlWM%i~8`~*xVZQhv*%g57W&gE`TAyrK1fOl1|Ry9trV)M3V69*36?bKn&G^y^cn%(A~0j1TF!TP<lfb3rF+0?SvZY9CX^TiY&JPyFom}%Z+IeN=I3K4TAnRS3{UA=m>Wb;SY0{Vk#gy4fojeAr#r;(?|^@tiGSn3dUz!s!`H5^1-;k34<0+kD6-c;4X(5H+O}PPRjJcA#fL_`z+iQkXui`(?U~)zoUpmf0-)-h1S@=fWiA0BJi_T(Qp8(%WO6<g~NWp%7W-vn2obTF*D5V!EMQ*98`q#RH_|_^+n<HzgImH<D`O2S0qJTPXetGXAbD4>3T?)mltn7z_IhiWoh~*%_;gel~bENP%n*`UxPHVhhv0|T_tm5M4>NlAN81^bky5;x4E;mP!tGq`Wo81z5S-^yv$OjLB;1(gvznCS~e&AA`mN^bAD)`ET}pPC+U8)->bP?h|U5m`VSez{~HK2(zR3tS=HB{U!>X^C0#sKPwTk>r~)<uWvy*pdsVnzGX5?W9>=2z+RYyTq;4KfwkoYR>MpWjk_*=OcdM$xD{~j%p{;#!K3%`ZSO-&V1NnOXJC?s+UY_IOByTyLzSte;#?|uc8bz|V>lB}C^ujjHDMqtxZaINX@0i06IIdCwpl9@l4EmN2X5?UjZnyOVvXdgL;bMv<rpPh-(vbO0g~CZei$7J=!lBd)*KAM{vb98T5sjq-tol5DETPYTFn}k*#wSszod9cD+>z+Yro_vwSG^cZ!I4>u%6xDMvsw0o=-}ZC)8zqp*EmS8cs2UYJ*M{dEQrCqVIS^#=Hfr{KxCf'}

EMBEDDED_SHA256 = {'ask120_snapshot.py': 'f0ee1f7de72e32e17f00ce5405734fe8ed66ab628e94531a8107ce6b819f0382',
 'audit_snapshot.py': 'cb0ed8dcdf9f7209ea732219a14905f9cd034de29dec0696a71fee5290631002',
 'checkpoint_snapshot.py': '8ecff6c935274743509f8ca1b1a0a85dd612b58f02240c79e6ae9cab03e18ff6',
 'cnkang_network_snapshot.py': '6ec4a7de1f96221cbac97312ef9def117afb867732612c8c11dfd1b66739c70c',
 'cnkang_request_worker_snapshot.py': '0dcae995e981cffa7eb1308c619756b9fd97aa8480a2f4aac64683ec9e4fa052',
 'cnkang_snapshot.py': 'f0958ecde31eef754e884f4408b0bed691af522382b75add80ac502151f5bedf',
 'cnkang_transport_snapshot.py': '8d888e810bd5efcd7a605fca913d6136a52e445490b0661be492483cfe296d12',
 'corpus_runner_snapshot.py': '1e762d2fe385cf3053772beed339a8e668c768abdf60b26bab004038894746a0',
 'crawler_snapshot.py': '34e75e0cedb052e422c039d35553c81ec77f04fd3cf3513a45b2e802b92b510a',
 'diagnostics_snapshot.py': '0d45ea2439d7d11f42d7203b3628d18b05a54f8f86b746af235ed3e354af316c',
 'export_snapshot.py': 'e924cfa0a104928aa7d90826118955bbe49bbdca833994bb42746bf7e9c8ba30',
 'longchau_snapshot.py': '287b8cd009ff5e558f2a0e7c5592dd1b430a5e144da23619087c745f57287a3f',
 'postprocess_snapshot.py': '377d747c79f82e52d4fd9f16e36512af540d985536aa0a70b86855eff76842b8',
 'priority_probe_snapshot.py': '626e493ad05d4f6c96b257167c2cb81a815614122ce35272317ced8d5240f37d',
 'pubmed_efetch_snapshot.py': '983c7c3c1350c4e3a157a5b1b6b004a85f81074454ced5ffb7476f531f4cc3c7',
 'pubmed_resume_snapshot.py': '2c43d37a923335a61cc03d41632c340668ac8cc45afca15f24ebdcddc36220db'}


RUN_DIR.mkdir(parents=True, exist_ok=True)
POSTPROCESS_FILES = {'pubmed_resume_snapshot.py', 'checkpoint_snapshot.py', 'audit_snapshot.py', 'export_snapshot.py', 'diagnostics_snapshot.py', 'postprocess_snapshot.py'}

HAS_CNKANG_SAMPLE = False
if (RUN_DIR / 'sample.jsonl').exists():
    from urllib.parse import urlsplit
    with (RUN_DIR / 'sample.jsonl').open(encoding='utf-8') as stream:
        HAS_CNKANG_SAMPLE = any(
            (urlsplit(json.loads(line)['url']).hostname or '').lower() == 'cnkang.com'
            or (urlsplit(json.loads(line)['url']).hostname or '').lower().endswith('.cnkang.com')
            for line in stream if line.strip())
if not POSTPROCESS_ONLY and HAS_CNKANG_SAMPLE and (RUN_DIR / 'corpus_config.json').exists():
    saved_cnkang = json.loads((RUN_DIR / 'corpus_config.json').read_text(encoding='utf-8'))
    for name, key in (('cnkang_snapshot.py','cnkang_sha256'),
                      ('cnkang_transport_snapshot.py','cnkang_transport_sha256'),
                      ('cnkang_network_snapshot.py','cnkang_network_sha256'),
                      ('cnkang_request_worker_snapshot.py','cnkang_network_worker_sha256'),
                      ('crawler_snapshot.py','crawler_sha256'), ('corpus_runner_snapshot.py','runner_sha256')):
        if saved_cnkang.get(key) != EMBEDDED_SHA256[name]:
            raise ValueError('Run cnkang cũ cần RUN_TAG mới + RECOVERY_RUN_PATH; giữ nguyên raw/checkpoint cũ.')

HAS_ASK120_SAMPLE = False
if (RUN_DIR / 'sample.jsonl').exists():
    from urllib.parse import urlsplit
    with (RUN_DIR / 'sample.jsonl').open(encoding='utf-8') as stream:
        for line in stream:
            if not line.strip():
                continue
            ask120_url = urlsplit(json.loads(line)['url'])
            ask120_id = ask120_url.path[len('/question/'):-len('.htm')]
            if ((ask120_url.hostname or '').lower() in ('120ask.com', 'www.120ask.com')
                    and ask120_url.path.startswith('/question/') and ask120_url.path.endswith('.htm')
                    and ask120_id.isdigit()):
                HAS_ASK120_SAMPLE = True
                break
if not POSTPROCESS_ONLY and HAS_ASK120_SAMPLE and (RUN_DIR / 'corpus_config.json').exists():
    saved_ask120_runtime = json.loads((RUN_DIR / 'corpus_config.json').read_text(encoding='utf-8'))
    for name, key in (('ask120_snapshot.py', 'ask120_sha256'),
                      ('crawler_snapshot.py', 'crawler_sha256'),
                      ('corpus_runner_snapshot.py', 'runner_sha256')):
        if saved_ask120_runtime.get(key) != EMBEDDED_SHA256[name]:
            raise ValueError('Run 120ask cũ cần RUN_TAG mới + RECOVERY_RUN_PATH; '
                             'không tự thay text/checkpoint đã trích bằng parser cũ.')

HAS_VOV_SAMPLE = False
if (RUN_DIR / 'sample.jsonl').exists():
    from urllib.parse import urlsplit
    with (RUN_DIR / 'sample.jsonl').open(encoding='utf-8') as stream:
        HAS_VOV_SAMPLE = any((urlsplit(json.loads(line)['url']).hostname or '').lower() == 'vov.vn'
            or (urlsplit(json.loads(line)['url']).hostname or '').lower().endswith('.vov.vn')
            for line in stream if line.strip())
if not POSTPROCESS_ONLY and HAS_VOV_SAMPLE and VOV_USE_CURL and (RUN_DIR / 'corpus_config.json').exists():
    saved_vov_runtime = json.loads((RUN_DIR / 'corpus_config.json').read_text(encoding='utf-8'))
    for name,key in (('priority_probe_snapshot.py','probe_sha256'),('corpus_runner_snapshot.py','runner_sha256')):
        if saved_vov_runtime.get(key) != EMBEDDED_SHA256[name]:
            raise ValueError('Run VOV cũ cần RUN_TAG mới + RECOVERY_RUN_PATH để chuyển sang curl/Chrome; giữ raw/records/cooldown.')

HAS_LONGCHAU_SAMPLE = False
if (RUN_DIR / 'sample.jsonl').exists():
    from urllib.parse import urlsplit
    with (RUN_DIR / 'sample.jsonl').open(encoding='utf-8') as stream:
        HAS_LONGCHAU_SAMPLE = any(urlsplit(json.loads(line)['url']).hostname in
            ('nhathuoclongchau.com.vn', 'www.nhathuoclongchau.com.vn') for line in stream if line.strip())
if not POSTPROCESS_ONLY and HAS_LONGCHAU_SAMPLE and (RUN_DIR / 'corpus_config.json').exists():
    saved_runtime = json.loads((RUN_DIR / 'corpus_config.json').read_text(encoding='utf-8'))
    for name, key in (('longchau_snapshot.py', 'longchau_sha256'),
                      ('corpus_runner_snapshot.py', 'runner_sha256'), ('priority_probe_snapshot.py', 'probe_sha256')):
        if saved_runtime.get(key) != EMBEDDED_SHA256[name]:
            raise ValueError('Run Long Châu cũ cần recovery: dùng ZIP đầy đủ, RECOVERY_RUN_PATH + RUN_TAG mới; '
                             'không ép RESUME vì parser/runner đã đổi. Raw và records cũ được giữ nguyên.')

for name, encoded in EMBEDDED_FILES.items():
    target = RUN_DIR / name
    if name in POSTPROCESS_FILES or (name == 'ask120_snapshot.py' and POSTPROCESS_ONLY and HAS_ASK120_SAMPLE) or (name == 'longchau_snapshot.py' and POSTPROCESS_ONLY and HAS_LONGCHAU_SAMPLE) or (not target.exists() and not POSTPROCESS_ONLY):
        body = zlib.decompress(base64.b85decode(encoded))
        if hashlib.sha256(body).hexdigest() != EMBEDDED_SHA256[name]:
            raise ValueError(f'Snapshot bị hỏng: {name}')
        target.write_bytes(body)
resume_spec = importlib.util.spec_from_file_location('pubmed_resume_upgrade', RUN_DIR / 'pubmed_resume_snapshot.py')
PUBMED_RESUME = importlib.util.module_from_spec(resume_spec)
resume_spec.loader.exec_module(PUBMED_RESUME)
if not POSTPROCESS_ONLY and PUBMED_USE_EFETCH and PUBMED_UPGRADE_EFETCH_SNAPSHOT:
    upgrade_status = PUBMED_RESUME.upgrade(RUN_DIR,
        zlib.decompress(base64.b85decode(EMBEDDED_FILES['pubmed_efetch_snapshot.py'])),
        {'batch_size': PUBMED_EFETCH_BATCH_SIZE, 'workers': PUBMED_EFETCH_WORKERS,
            'gap_seconds': PUBMED_EFETCH_GAP_SECONDS, 'attempts': PUBMED_EFETCH_ATTEMPTS,
            'timeout': [PUBMED_EFETCH_CONNECT_TIMEOUT, PUBMED_EFETCH_READ_TIMEOUT], 'budget': PUBMED_EFETCH_BUDGET,
            'min_batch_size': PUBMED_MIN_BATCH_SIZE, 'max_split_requests': PUBMED_MAX_SPLIT_REQUESTS,
            'fallback_budget': PUBMED_FALLBACK_BUDGET, 'pending_retry_passes': PUBMED_PENDING_RETRY_PASSES,
            'max_consecutive_failed_batches': PUBMED_MAX_FAILED_BATCHES,
            'checkpoint': {'enabled': CHECKPOINT_ENABLED, 'upload': CHECKPOINT_UPLOAD_ENABLED,
                           'repo_id': CHECKPOINT_HF_REPO_ID, 'every_urls': CHECKPOINT_EVERY_URLS,
                           'every_seconds': CHECKPOINT_EVERY_SECONDS, 'token_env': 'HF_TOKEN'}})
    print('EFetch snapshot:', json.dumps(upgrade_status, ensure_ascii=False), flush=True)

CRAWLER = RUN_DIR / 'crawler_snapshot.py'
CORPUS_RUNNER = RUN_DIR / 'corpus_runner_snapshot.py'
AUDITOR = RUN_DIR / 'audit_snapshot.py'
EXPORTER = RUN_DIR / 'export_snapshot.py'
DIAGNOSTICS = RUN_DIR / 'diagnostics_snapshot.py'
import importlib.util
postprocess_spec = importlib.util.spec_from_file_location('notebook_postprocess', RUN_DIR / 'postprocess_snapshot.py')
POSTPROCESS = importlib.util.module_from_spec(postprocess_spec)
postprocess_spec.loader.exec_module(POSTPROCESS)
diagnostics_archive = WORKDIR / f'{RUN_DIR.name}_diagnostics.zip'
from contextlib import contextmanager
import traceback
LOG_DIR = RUN_DIR / 'logs'
LOG_DIR.mkdir(exist_ok=True)

def refresh_diagnostics(strict=False):
    try:
        result = subprocess.run(
            [sys.executable, '-u', str(DIAGNOSTICS), '--out', str(RUN_DIR), '--archive', str(diagnostics_archive)],
            check=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding='utf-8', errors='replace',
        )
        with (LOG_DIR / 'diagnostics.log').open('a', encoding='utf-8') as stream:
            stream.write(result.stdout)
    except (OSError, subprocess.CalledProcessError) as exc:
        with (LOG_DIR / 'diagnostics.log').open('a', encoding='utf-8') as stream:
            stream.write(str(exc) + '\n' + str(getattr(exc, 'output', '') or '') + '\n')
        if strict:
            raise
        print(f'Tạo ZIP chẩn đoán thất bại: {exc}', flush=True)

def save_checkpoint_archive(failed_step=None, error=None):
    archive = POSTPROCESS.package_run(
        RUN_DIR, WORKDIR, id_from=ID_FROM, id_to=ID_TO, dataset_revision=DATASET_REVISION,
        source_sha256=SOURCE_SHA256, packages=versions, refresh_diagnostics=refresh_diagnostics,
        failed_step=failed_step, error=error)
    print('ZIP checkpoint đầy đủ:', archive, flush=True)
    print('ZIP chẩn đoán:', diagnostics_archive, flush=True)
    return archive

@contextmanager
def diagnostic_step(name):
    try:
        yield
    except BaseException as exc:
        with (LOG_DIR / 'notebook_errors.log').open('a', encoding='utf-8') as stream:
            stream.write(f'Bước: {name}\n' + traceback.format_exc() + '\n')
        try:
            save_checkpoint_archive(failed_step=name, error=f'{type(exc).__name__}: {exc}')
        except Exception as rescue_error:
            print(f'Đóng gói cứu dữ liệu lỗi: {rescue_error}; checkpoint vẫn ở {RUN_DIR}', flush=True)
            with (LOG_DIR / 'notebook_errors.log').open('a', encoding='utf-8') as stream:
                stream.write('Đóng gói cứu dữ liệu: ' + traceback.format_exc() + '\n')
        raise
    else:
        refresh_diagnostics()
with diagnostic_step('snapshot_validation'):
    if (RUN_DIR / 'crawl_config.json').exists():
        saved_config = json.loads((RUN_DIR / 'crawl_config.json').read_text(encoding='utf-8'))
        if hashlib.sha256(CRAWLER.read_bytes()).hexdigest() != saved_config['script_sha256']:
            raise ValueError('Snapshot không khớp hash của run cũ; không được ép resume.')
    if not requirements_path.exists():
        requirements_path.write_text(''.join(f'{k}=={v}\n' for k, v in versions.items()), encoding='utf-8')
print('Đã chuẩn bị snapshot và ghi phiên bản dependencies.')
refresh_diagnostics()


## 4. Tải hoặc đọc danh sách URL

Nguồn được cố định theo revision và kiểm tra SHA-256. Chỉ đọc corpus URL;
không dùng query làm nội dung hay nhãn. File Input có thể ở vị trí khác nhưng
được copy vào đường dẫn ổn định trong `/kaggle/working` để dùng lại khi resume.


In [ ]:
with diagnostic_step('source'):
    import requests

    def sha256_file(path):
        digest = hashlib.sha256()
        with Path(path).open('rb') as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b''):
                digest.update(block)
        return digest.hexdigest()

    SOURCE_PATH.parent.mkdir(parents=True, exist_ok=True)
    if SOURCE_PARQUET:
        supplied = Path(SOURCE_PARQUET)
        if not supplied.is_file():
            raise FileNotFoundError(supplied)
        if sha256_file(supplied) != SOURCE_SHA256:
            raise ValueError(f'Parquet Input không khớp {SOURCE_FILENAME} của revision đã chốt; cache hiện tại được giữ nguyên.')
        if supplied.resolve() != SOURCE_PATH.resolve():
            shutil.copyfile(supplied, SOURCE_PATH)
    elif not SOURCE_PATH.exists():
        temporary = SOURCE_PATH.with_suffix('.download')
        try:
            with requests.get(SOURCE_URL, stream=True, timeout=(15, 90)) as response:
                response.raise_for_status()
                with temporary.open('wb') as stream:
                    for block in response.iter_content(1024 * 1024):
                        if block:
                            stream.write(block)
            if sha256_file(temporary) != SOURCE_SHA256:
                raise ValueError('Hash file tải về không khớp nguồn đã chốt.')
            temporary.replace(SOURCE_PATH)
        finally:
            temporary.unlink(missing_ok=True)
    if sha256_file(SOURCE_PATH) != SOURCE_SHA256:
        raise ValueError('Parquet không khớp nguồn đã chốt. Kiểm tra file Input.')
    print(f'Nguồn hợp lệ: {SOURCE_PATH} ({SOURCE_PATH.stat().st_size:,} byte)')


## 5. Chọn ID và tạo manifest

Nếu khoảng chọn có ID không tồn tại trong nguồn, notebook báo số lượng thiếu,
rồi lấy đúng các hàng thực sự có trong khoảng; không tạo ID hoặc URL thay thế.
Bước chuẩn bị hiện quét toàn file để kiểm tra nguồn và thống kê hostname, nên có thể mất một lúc.


In [ ]:
with diagnostic_step('prepare'):
    import pyarrow.parquet as pq
    from collections import Counter
    from urllib.parse import urlsplit
    from IPython.display import display
    from datetime import datetime, timezone
    import time
    import uuid

    selected = pq.read_table(SOURCE_PATH, filters=[('id', '>=', ID_FROM), ('id', '<=', ID_TO)]).to_pylist()
    selected.sort(key=lambda row: row['id'])
    if not selected:
        raise ValueError('Không có URL nào trong khoảng ID đã chọn.')
    if len({row['id'] for row in selected}) != len(selected):
        raise ValueError('Nguồn có ID trùng trong khoảng chọn.')
    missing_count = ID_TO - ID_FROM + 1 - len(selected)
    print(f'Chọn {len(selected):,} URL; có {missing_count:,} ID không tồn tại trong khoảng.')
    print('Phân bố hostname:', dict(Counter(urlsplit(row['url']).hostname for row in selected)))
    display(selected[:5])

    LOG_DIR = RUN_DIR / 'logs'
    LOG_DIR.mkdir(exist_ok=True)

    def log_event(name, event):
        with (LOG_DIR / name).open('a', encoding='utf-8') as stream:
            stream.write(json.dumps(event, ensure_ascii=False) + '\n')

    def run_script(script, *arguments):
        # Output hiển thị trực tiếp để theo dõi tiến độ; không dùng shell.
        invocation_id = uuid.uuid4().hex
        started = time.monotonic()
        operation = str(arguments[0]) if script in (CRAWLER, CORPUS_RUNNER) else Path(script).stem
        log_event('invocations.jsonl', {
            'event': 'start', 'invocation_id': invocation_id, 'operation': operation,
            'at': datetime.now(timezone.utc).isoformat(), 'arguments': list(map(str, arguments)),
        })
        process = subprocess.Popen(
            [sys.executable, '-u', str(script), *map(str, arguments)],
            cwd=str(WORKDIR), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding='utf-8', errors='replace', bufsize=1,
        )
        returncode = None
        try:
            with (LOG_DIR / (Path(script).stem + '.log')).open('a', encoding='utf-8') as log:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    log.write(line)
                    log.flush()
                    if operation == 'crawl':
                        try:
                            progress = json.loads(line)
                        except ValueError:
                            progress = {}
                        if isinstance(progress, dict) and 'completed_this_invocation' in progress:
                            log_event('progress.jsonl', {
                                **progress, 'invocation_id': invocation_id,
                                'at': datetime.now(timezone.utc).isoformat(),
                                'elapsed_seconds': time.monotonic() - started,
                            })
                            refresh_diagnostics()
            returncode = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
        finally:
            process.stdout.close()
            log_event('invocations.jsonl', {
                'event': 'end', 'invocation_id': invocation_id, 'operation': operation,
                'at': datetime.now(timezone.utc).isoformat(),
                'elapsed_seconds': time.monotonic() - started,
                'returncode': process.returncode,
            })
            refresh_diagnostics()
        if returncode:
            raise subprocess.CalledProcessError(returncode, process.args)

    manifest_path = RUN_DIR / 'manifest.json'
    if not manifest_path.exists():
        run_script(CRAWLER, 'prepare', '--source', SOURCE_PATH, '--out', RUN_DIR,
                   '--size', len(selected), '--id-start', ID_FROM, '--id-stop', ID_TO + 1)
    else:
        manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        if manifest['source_sha256'] != SOURCE_SHA256 or manifest['selected'] != len(selected):
            raise ValueError('Manifest không khớp nguồn hoặc số URL đã chọn.')
        if not POSTPROCESS_ONLY and Path(manifest['source']).resolve() != SOURCE_PATH.resolve():
            raise ValueError('Đường dẫn nguồn của run cũ khác notebook này; dùng run mới.')
        sample_bytes = (RUN_DIR / 'sample.jsonl').read_bytes()
        if hashlib.sha256(sample_bytes).hexdigest() != manifest['sample_sha256']:
            raise ValueError('Hash sample của run cũ đã thay đổi.')
        with (RUN_DIR / 'sample.jsonl').open(encoding='utf-8') as stream:
            sample = [json.loads(line) for line in stream if line.strip()]
        if {r['id']: r['url'] for r in sample} != {r['id']: r['url'] for r in selected} or len(sample) != len(selected):
            raise ValueError('ID/URL của run cũ không khớp khoảng chọn.')
        print('Manifest hợp lệ; giữ checkpoint để resume.')


## 6. Crawl — chạy lại ô này để tiếp tục

PubMed mặc định dùng **EFetch GET**, **100 PMID/batch**, **1 worker**, nghỉ ít nhất **0,5 giây giữa request**.
`PUBMED_USE_EFETCH = True` hoạt động cả khi `BROWSER_TEST_MODE = 'off'`; run chỉ có PubMed không khởi động Chrome.
Giữ ID/URL gốc, ghép theo PMID trong XML; `text` có title và toàn bộ abstract cùng nhãn.
Các website ngoài PubMed, Long Châu và VOV dùng Requests rồi Selenium theo cấu hình browser; `RESPECT_ROBOTS` áp dụng lượt crawl website.

Checkpoint XML và batch được lưu trước các record ID. Resume sửa các record còn thiếu mà không GET lại batch đã lưu.
`MAX_NEW` tính số URL corpus mới đã lên lịch, gồm cả PubMed; retry không tính hai lần.
HTTP 400 chỉ retry khi body báo Timeout/Empty Response; HTTP 400 invalid UID/parameter không retry mù quáng.
5 attempts, read timeout 90 giây, budget 300 giây/request batch, backoff + jitter và Retry-After.
Batch còn lỗi chia 100 → 50 → 25 → … để cô lập, giới hạn tổng request/time; sau đó tiếp tục batch khác.
Cuối lượt thử pending thêm một vòng. Cooldown server dài hoặc 3 batch liên tiếp không lấy được bản ghi
thì hoãn để resume; không ném RuntimeError vì riêng một batch lỗi, không đánh dấu complete khi còn pending.
PMID không được API trả về hoặc không có abstract được ghi riêng, không đưa vào corpus accepted.
Xem `pubmed_results.csv`, `pubmed_missing_pmids.csv`, `pubmed_no_abstract.csv`, `pubmed_review.csv`,
`pubmed_pending.csv`, `pubmed_requests.jsonl`, `pubmed_efetch_summary.json` và `raw/efetch/` trong ZIP đầy đủ.

Run mới có hậu tố `_efetch`. Nâng cấp run Selenium cũ bằng `RUN_TAG` mới và `RECOVERY_RUN_PATH`:
reuse raw hợp lệ của domain khác; PubMed HTML cũ được lấy mới qua EFetch.
Muốn thử lại PMID thiếu/no abstract cần run mới; resume giữ những kết quả XML hợp lệ đã hoàn tất.
`POSTPROCESS_ONLY = True` vẫn audit/export run cũ mà không crawl lại.
Resume EFetch cũ tự nâng cấp transport tương thích, giữ nguyên XML/record/schema và các snapshot crawl khác.
Checkpoint full ZIP/HF chạy định kỳ trong lúc crawl, khác với upload corpus cuối cùng sau audit/export.

Tham khảo [NCBI EFetch](https://www.ncbi.nlm.nih.gov/books/NBK25499/).

Long Châu mặc định dùng `curl_cffi`/Chrome, 1 worker, delay 5 giây; không chạy Selenium cho domain này. Parser giữ sapo/nội dung chính; `/benh/` lấy đầy đủ các trường HTML y khoa từ JSON đã xác minh slug/canonical. Xem `longchau_results.csv`, `longchau_summary.json`; parser v2 loại nhóm liên kết “Xem thêm”. Run mới có hậu tố `_longchau_v2`; dùng recovery từ ZIP đầy đủ để tái trích HTML 200 cũ, giữ cooldown. Checkpoint thuộc runner chung: lưu theo cadence cho mọi engine và khi dừng/ngắt; log có tiến độ chung theo ID.

VOV (`vov.vn`, `vov2.vov.vn` và subdomain) mặc định dùng `curl_cffi` với `impersonate="chrome"`, một request VOV tại một thời điểm. Giữ parser hiện có, timeout/budget/robots, raw và ID/URL. Không chạy Selenium cho URL VOV khi `VOV_USE_CURL=True`; 429 dừng nhóm VOV và lưu `vov_request_gate.json` qua resume/recovery. Run VOV cũ dùng RUN_TAG mới + RECOVERY_RUN_PATH.

120ask V3 chỉ định tuyến `/question/<id>.htm`; `text` giữ câu hỏi và từng lượt theo thread. `ask120_threads`, `ask120_text_raw` và `ask120_qc` nằm trong record checkpoint; Parquet corpus không thêm cột. Run parser cũ phải dùng RUN_TAG mới + RECOVERY_RUN_PATH để tái trích raw, không resume ghi đè.

Cnkang dùng chung parser và Requests transport với check_cnkang.com.ipynb: một worker, gap tối thiểu 1,5 giây, HTTPS trước, robots từng origin, retry/cooldown hữu hạn; không chạy Selenium. Giữ encoding/body candidates, bảng, text trước/sau và parser_flags/source_flags riêng. Mã v6 gốc đã lưu trong docs/references/cnkang; chưa có raw 120 trang để chứng minh tương đương baseline cũ. Run cũ dùng RUN_TAG mới + RECOVERY_RUN_PATH để tái trích raw, không ép resume.


In [ ]:
with diagnostic_step('crawl'):
    if POSTPROCESS_ONLY:
        print('POSTPROCESS_ONLY: bỏ qua Requests/Selenium; giữ nguyên cấu hình và snapshot crawl.')
    else:
        # Pass credentials to the subprocess through its environment; settings/log contain none.
        for name, value in [('NCBI_EMAIL', NCBI_EMAIL), ('NCBI_API_KEY', NCBI_API_KEY)]:
            if value:
                os.environ[name] = value
        POSTPROCESS.invalidate(RUN_DIR)
        corpus_settings = {
            'user_agent': USER_AGENT, 'requests_trust_env': REQUESTS_TRUST_ENV, 'delay': DELAY, 'attempts': ATTEMPTS,
            'timeout': [PAGE_CONNECT_TIMEOUT, PAGE_READ_TIMEOUT], 'budget': PAGE_REQUEST_BUDGET,
            'respect_robots': RESPECT_ROBOTS,
            'robots_options': {'robots_attempts': ROBOTS_ATTEMPTS,
                               'robots_connect_timeout': ROBOTS_CONNECT_TIMEOUT,
                               'robots_read_timeout': ROBOTS_READ_TIMEOUT,
                               'robots_retry_budget': ROBOTS_RETRY_BUDGET,
                               'robots_failure_ttl': ROBOTS_FAILURE_TTL},
            'browser_mode': BROWSER_TEST_MODE, 'browser_user_agent': BROWSER_USER_AGENT,
            'pubmed_selenium_first': PUBMED_SELENIUM_FIRST,
            'pubmed_use_efetch': PUBMED_USE_EFETCH,
            'longchau_use_curl': LONGCHAU_USE_CURL,
            'cnkang': {'gap': CNKANG_REQUEST_GAP, 'attempts': CNKANG_ATTEMPTS,
                       'timeout': [CNKANG_CONNECT_TIMEOUT, CNKANG_READ_TIMEOUT],
                       'request_deadline': CNKANG_REQUEST_DEADLINE,
                       'url_deadline': CNKANG_URL_DEADLINE,
                       'user_agent': CNKANG_USER_AGENT, 'trust_env': REQUESTS_TRUST_ENV},
            'vov_use_curl': VOV_USE_CURL,
            'longchau': {'workers': LONGCHAU_WORKERS, 'delay': LONGCHAU_DELAY, 'attempts': LONGCHAU_ATTEMPTS,
                          'timeout': [LONGCHAU_CONNECT_TIMEOUT, LONGCHAU_READ_TIMEOUT],
                          'budget': LONGCHAU_REQUEST_BUDGET, 'impersonate': 'chrome',
                          'retry_failed_on_resume': LONGCHAU_RETRY_FAILED_ON_RESUME},
            'pubmed_efetch': {'batch_size': PUBMED_EFETCH_BATCH_SIZE, 'workers': PUBMED_EFETCH_WORKERS,
                               'gap_seconds': PUBMED_EFETCH_GAP_SECONDS, 'attempts': PUBMED_EFETCH_ATTEMPTS,
                               'timeout': [PUBMED_EFETCH_CONNECT_TIMEOUT, PUBMED_EFETCH_READ_TIMEOUT],
                               'budget': PUBMED_EFETCH_BUDGET},
            'chrome_binary': str(CHROME_BINARY) if CHROME_BINARY else None,
            'browser_load_timeout': BROWSER_LOAD_TIMEOUT, 'browser_wait_seconds': BROWSER_WAIT_SECONDS,
            'browser_attempts': BROWSER_ATTEMPTS,
        }
        PUBMED_CONFIG_CURRENT = {'batch_size': PUBMED_EFETCH_BATCH_SIZE, 'workers': PUBMED_EFETCH_WORKERS,
            'gap_seconds': PUBMED_EFETCH_GAP_SECONDS, 'attempts': PUBMED_EFETCH_ATTEMPTS,
            'timeout': [PUBMED_EFETCH_CONNECT_TIMEOUT, PUBMED_EFETCH_READ_TIMEOUT], 'budget': PUBMED_EFETCH_BUDGET,
            'min_batch_size': PUBMED_MIN_BATCH_SIZE, 'max_split_requests': PUBMED_MAX_SPLIT_REQUESTS,
            'fallback_budget': PUBMED_FALLBACK_BUDGET, 'pending_retry_passes': PUBMED_PENDING_RETRY_PASSES,
            'max_consecutive_failed_batches': PUBMED_MAX_FAILED_BATCHES,
            'checkpoint': {'enabled': CHECKPOINT_ENABLED, 'upload': CHECKPOINT_UPLOAD_ENABLED,
                           'repo_id': CHECKPOINT_HF_REPO_ID, 'every_urls': CHECKPOINT_EVERY_URLS,
                           'every_seconds': CHECKPOINT_EVERY_SECONDS, 'token_env': 'HF_TOKEN'}}
        corpus_settings['pubmed_efetch'] = PUBMED_CONFIG_CURRENT
        if PUBMED_UPGRADE_EFETCH_SNAPSHOT:
            corpus_settings = PUBMED_RESUME.runtime_settings(RUN_DIR, corpus_settings)
        if CHECKPOINT_ENABLED and CHECKPOINT_UPLOAD_ENABLED and not os.environ.get('HF_TOKEN'):
            try:
                from kaggle_secrets import UserSecretsClient
                os.environ['HF_TOKEN'] = UserSecretsClient().get_secret(CHECKPOINT_HF_SECRET_NAME)
            except Exception:
                print('Chưa có HF_TOKEN: checkpoint local vẫn lưu; backup HF chưa khả dụng.')
        settings_path = RUN_DIR / 'corpus_settings.json'
        settings_path.write_text(json.dumps(corpus_settings, ensure_ascii=False, indent=2), encoding='utf-8')
        arguments = ['crawl', '--out', RUN_DIR, '--workers', WORKERS, '--config', settings_path]
        if '--log-every' in CORPUS_RUNNER.read_text(encoding='utf-8'):
            arguments += ['--log-every', LOG_EVERY]
        if RECOVERY_RUN_PATH:
            arguments += ['--recovery-source', RECOVERY_RUN_PATH]
        if MAX_NEW is not None:
            arguments += ['--max-new', MAX_NEW]
        print('ZIP chẩn đoán tự cập nhật:', diagnostics_archive)
        run_script(CORPUS_RUNNER, *arguments)


## 7. Báo cáo, kiểm tra và xuất corpus

Audit chỉ chạy khi đủ checkpoint; raw rỗng hoặc HTML không parse được được ghi cờ chẩn đoán.
Sai hash, ID/URL hoặc thiếu raw của bản ghi extracted vẫn làm bước này thất bại.
Corpus chỉ xuất sau audit đạt. Thành công được ghi riêng trong postprocess_status.json;
run hết pending chưa đủ điều kiện thành công/upload. Nếu lỗi, notebook tạo ZIP checkpoint
đầy đủ và diagnostics trước khi báo lại lỗi gốc. Accepted là QC tự động, vẫn cần review nội dung.


In [ ]:
with diagnostic_step('report_audit_export'):
    summary = POSTPROCESS.run_postprocess(RUN_DIR, SOURCE_PATH, run_script)
    display({k: summary[k] for k in ['selected', 'completed', 'pending', 'accepted', 'review', 'rejected', 'status_counts']})
    if (RUN_DIR / 'access_summary.json').exists():
        display(json.loads((RUN_DIR / 'access_summary.json').read_text(encoding='utf-8')))
        print('Lỗi theo tổng URL dataset giảm dần:', RUN_DIR / 'WEB_ERRORS.md')
        print('Đối chiếu Requests/Selenium:', RUN_DIR / 'requests_vs_selenium.csv')
    if summary['pending']:
        print('Run chưa hoàn tất; chưa audit/export hoặc đánh dấu thành công. Chạy tiếp ô crawl.')
    elif summary['accepted'] > 0:
        corpus = pq.read_table(RUN_DIR / 'corpus.parquet')
        print(corpus.schema)
        display(corpus.select(['id', 'url', 'title']).slice(0, 5).to_pylist())
    else:
        print('Audit đạt; không có tài liệu accepted để xuất corpus. Xem QUALITY_REPORT.md.')


## 8. ZIP chẩn đoán — tích hợp trong Output

Chạy ô này để tạo lại gói debug từ checkpoint hiện có, kể cả khi crawl chưa xong.
Không cần file Python riêng, không tải lại URL. ZIP chứa:
- `overview.html`, `summary.json`: tiến độ, tốc độ/ETA, trạng thái HTTP và QC.
- CSV từng ID, lỗi/pending, request/retry và redirect.
- Log pip/crawl/Chrome, kết quả preflight, traceback notebook, manifest/cấu hình và phiên bản môi trường.
- Snapshot code, bằng chứng robots và tối đa 10 mẫu record/raw để kiểm tra parser.

Raw mẫu có ngân sách 10 MiB; các mẫu bị bỏ qua được ghi trong `sample_manifest.json`.
Giải nén ZIP và mở `overview.html`. Có thể gửi nguyên ZIP này để chẩn đoán lỗi.


In [ ]:
with diagnostic_step('diagnostics_archive'):
    from IPython.display import FileLink
    import zipfile

    refresh_diagnostics(strict=True)
    with zipfile.ZipFile(diagnostics_archive) as diagnostic_bundle:
        diagnostic_summary = json.loads(diagnostic_bundle.read('summary.json'))
    display({key: diagnostic_summary[key] for key in ['selected', 'completed', 'pending', 'quality_status_counts', 'timing']})
    print('ZIP chẩn đoán trong Output:', diagnostics_archive)
    display(FileLink(str(diagnostics_archive.relative_to(WORKDIR))))


## 9. Đóng gói corpus đầy đủ để tải về

ZIP được tạo cả khi xử lý thất bại; manifest ghi complete = False cho run chưa qua audit/export.
ZIP chứa cả raw, records, snapshot code và corpus để truy vết/resume; có thể khá lớn.
ZIP riêng `<run>_diagnostics.zip` nhỏ hơn: giải nén và mở `overview.html` để xem
tiến độ, tốc độ/ETA, HTTP status, lỗi và cờ chất lượng. Có CSV từng ID và log cài dependencies.
Tải file trong Output hoặc qua link bên dưới. Khi đổi session, cần giữ output hoặc
thêm run đã giải nén làm Kaggle Input và đặt `RESUME_RUN_PATH` trước khi chạy lại.
Resume không tự thử lại ID lỗi đã có checkpoint. Dùng `RECOVERY_RUN_PATH`
cùng `RUN_TAG` mới để thử lại lỗi bằng code/cấu hình mới và tái sử dụng raw thành công.


In [ ]:
with diagnostic_step('checkpoint_archive'):
    from IPython.display import FileLink

    archive = save_checkpoint_archive()
    notebook_manifest = json.loads((RUN_DIR / 'notebook_manifest.json').read_text(encoding='utf-8'))
    print('Run đã qua audit/export:', notebook_manifest['complete'])
    display(FileLink(str(Path(archive).relative_to(WORKDIR))))
    display(FileLink(str(diagnostics_archive.relative_to(WORKDIR))))
    if notebook_manifest['complete'] and notebook_manifest['accepted'] > 0:
        display(FileLink(str((RUN_DIR / 'corpus.parquet').relative_to(WORKDIR))))
        display(FileLink(str((RUN_DIR / 'corpus.jsonl').relative_to(WORKDIR))))


## 10. Tự upload output lên Hugging Face Dataset

Repo đích: **AncientWings/AncientWings_R2AI_STAGE3_ds**. Bật Internet và gắn Kaggle Secret
**HF_TOKEN** có quyền ghi vào repo. Run All sẽ tự chạy ô này sau khi xuất corpus/đóng ZIP.
Không cần đăng nhập qua widget hoặc tải output về máy.

Mỗi run được phân biệt bằng `RUN_DIR.name`: nguồn mới mặc định `ids_<ID_FROM>_<ID_TO>_synced_vi_en_zh`.
Recovery có tên riêng; nếu crawl lại cùng khoảng ID với code/cấu hình khác, đặt **RUN_TAG mới**
ở mục 1. Chạy lại ô upload cùng run sẽ cập nhật đúng các đường dẫn của run đó.

```text
corpus/<run>.parquet                       # Đầu vào RAG: chỉ tài liệu accepted
corpus_jsonl/<run>.jsonl                   # Cùng corpus, định dạng JSONL
archives/<run>.zip                         # Chỉ khi bật HF_UPLOAD_ARCHIVE = True
diagnostics/<run>_diagnostics.zip           # Gói chẩn đoán
manifests/<run>/notebook_manifest.json      # Khoảng ID và số lượng bản ghi
manifests/<run>/upload_manifest.json        # Danh sách file, kích thước và SHA-256
```

Mặc định upload corpus, diagnostics và manifest; không upload ZIP raw/checkpoint (`HF_UPLOAD_ARCHIVE = False`).
Mục 9 vẫn tạo ZIP đầy đủ trong Kaggle Output để tải về/resume; chỉ bật upload archive khi chủ động cần.
Đặt `HF_UPLOAD_ENABLED = False` nếu chỉ muốn giữ output Kaggle.
Run còn pending (ví dụ thử `MAX_NEW = 10`) sẽ **chưa upload**; chạy tiếp mục 6 rồi chạy lại mục 7–10.
Run hoàn tất nhưng không có accepted vẫn upload manifest/ZIP; không xuất corpus cũ còn sót lại.
Upload đọc file từ đĩa, không nạp cả ZIP vào RAM. Nếu mạng lỗi, ô thử lại tối đa 3 lần;
có thể chạy lại riêng mục 10 mà không crawl lại. Kết quả commit được lưu tại
`hf_upload_result.json` trong run. Repo đích phải tồn tại; token/repo lỗi sẽ được báo ở ô này.

Tài liệu: [Hugging Face upload](https://huggingface.co/docs/huggingface_hub/en/guides/upload)
và [Kaggle Secrets](https://github.com/Kaggle/kaggle-cli/blob/main/docs/kernels.md#using-secrets-in-kernels).


In [ ]:
# ===== UPLOAD OUTPUT CỦA RUN NÀY LÊN HUGGING FACE =====
HF_UPLOAD_ENABLED = True
HF_REPO_ID = 'AncientWings/AncientWings_R2AI_STAGE3_ds'
HF_SECRET_NAME = 'HF_TOKEN'  # Tạo trong Kaggle: Add-ons > Secrets; không dán token vào code.
HF_UPLOAD_JSONL = True
HF_UPLOAD_ARCHIVE = False  # ZIP raw/checkpoint chỉ giữ trong Kaggle Output.
HF_UPLOAD_DIAGNOSTICS = True
HF_UPLOAD_ATTEMPTS = 3


def upload_current_run_to_hf():
    import hashlib
    import importlib.metadata
    import json
    from datetime import datetime, timezone
    from pathlib import Path
    import re
    import subprocess
    import sys
    import time
    import traceback
    from urllib.parse import quote

    if not HF_UPLOAD_ENABLED:
        print('Đã tắt upload Hugging Face; output vẫn được lưu trong Kaggle Output.')
        return {'status': 'disabled'}

    run_dir = Path(RUN_DIR)
    manifest_path = run_dir / 'notebook_manifest.json'
    if not manifest_path.is_file():
        raise RuntimeError('Chạy mục 7–9 để xuất corpus và đóng ZIP trước khi upload.')
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    if (manifest.get('id_from_inclusive'), manifest.get('id_to_inclusive')) != (ID_FROM, ID_TO):
        raise RuntimeError('Manifest không khớp khoảng ID hiện tại; chạy lại mục 7–9.')
    if manifest.get('dataset_revision') != DATASET_REVISION:
        raise RuntimeError('Manifest không khớp revision nguồn; chạy lại mục 7–9.')
    if manifest.get('pending', 1) != 0:
        print(f"Chưa upload: run còn {manifest.get('pending', '?')} URL pending. "
              'Chạy tiếp mục 6, rồi chạy lại mục 7–10.')
        return {'status': 'incomplete', 'run_name': run_dir.name}
    # Completion requires fresh audit/export evidence, even if a legacy manifest says complete.
    import importlib.util
    helper_path = run_dir / 'postprocess_snapshot.py'
    if not helper_path.is_file():
        raise RuntimeError('Thiếu bằng chứng audit/export mới; chạy lại mục 7–9 với POSTPROCESS_ONLY = True.')
    helper_spec = importlib.util.spec_from_file_location('hf_postprocess_gate', helper_path)
    helper = importlib.util.module_from_spec(helper_spec)
    helper_spec.loader.exec_module(helper)
    helper.require_ready(run_dir, manifest)
    if type(HF_UPLOAD_ATTEMPTS) is not int or HF_UPLOAD_ATTEMPTS < 1:
        raise ValueError('HF_UPLOAD_ATTEMPTS phải là số nguyên dương.')

    run_name = run_dir.name
    files = [(manifest_path, f'manifests/{run_name}/notebook_manifest.json')]
    if manifest.get('accepted', 0) > 0:
        files.append((run_dir / 'corpus.parquet', f'corpus/{run_name}.parquet'))
        if HF_UPLOAD_JSONL:
            files.append((run_dir / 'corpus.jsonl', f'corpus_jsonl/{run_name}.jsonl'))
    else:
        print('Run không có tài liệu accepted; chỉ upload manifest và các ZIP đã chọn.')
    if HF_UPLOAD_ARCHIVE:
        files.append((Path(WORKDIR) / f'{run_name}.zip', f'archives/{run_name}.zip'))
    if HF_UPLOAD_DIAGNOSTICS:
        files.append((Path(WORKDIR) / f'{run_name}_diagnostics.zip',
                      f'diagnostics/{run_name}_diagnostics.zip'))
    for local_path, _ in files:
        if not local_path.is_file() or local_path.stat().st_size == 0:
            raise RuntimeError(f'Thiếu output hoặc file rỗng: {local_path.name}. Chạy lại mục 7–9.')

    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret(HF_SECRET_NAME)
    except Exception:
        raise RuntimeError(f'Tạo và bật Kaggle Secret {HF_SECRET_NAME!r} cho notebook này; '
                           'token cần quyền ghi vào dataset đích.') from None
    if not isinstance(token, str) or not token.strip():
        raise RuntimeError(f'Kaggle Secret {HF_SECRET_NAME!r} đang rỗng.')
    token = token.strip()

    try:
        from huggingface_hub import HfApi, CommitOperationAdd
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                        'huggingface_hub>=0.34,<3'], check=True)
        from huggingface_hub import HfApi, CommitOperationAdd

    def report_upload_error(error, stage, attempt=None):
        # Giữ thông tin chẩn đoán, nhưng che token/header xác thực và query của URL ký.
        def redact(value):
            value = str(value).replace(token, '[REDACTED_TOKEN]')
            value = re.sub(r'\bhf_[A-Za-z0-9]{8,}\b', '[REDACTED_TOKEN]', value)
            value = re.sub(r'(?i)\bBearer\s+[A-Za-z0-9._~+/=-]+',
                           'Bearer [REDACTED_TOKEN]', value)
            value = re.sub(r'(https?://)[^/\s@]+@', r'\1[REDACTED_AUTH]@', value)
            value = re.sub(r'''(https?://[^\s?<>"'\)\]]+)\?[^\s<>"'\)\]]+''',
                           r'\1?[REDACTED_QUERY]', value)
            value = re.sub(r'''(?i)(\bauthorization\b["']?\s*[:=]\s*["']?)'''
                           r'''(?:(?:Bearer|Basic)\s+)?[^\s"',;}]+''',
                           r'\1[REDACTED_AUTH]', value)
            value = re.sub(r'''(?i)(\b(?:access[_-]?token|token|api[_-]?key)\b'''
                           r'''["']?\s*[:=]\s*["']?)[^\s"',;}&]+''',
                           r'\1[REDACTED_TOKEN]', value)
            return value

        chain, seen, current, status = [], set(), error, None
        while current is not None and id(current) not in seen and len(chain) < 8:
            seen.add(id(current))
            message = redact(str(current))
            chain.append(f'{type(current).__name__}: {message}')
            code = getattr(getattr(current, 'response', None), 'status_code', None)
            if status is None and isinstance(code, int) and 100 <= code <= 599:
                status = code
            if status is None:
                # hf_xet có thể bọc HTTP error trong RuntimeError mà không có response.
                match = re.search(r'\b(?:HTTP(?:\s+(?:status|error|code|client|server)){0,4}'
                                  r'|status(?:\s+code)?)\s*[:=]?\s*\(?([1-5]\d{2})\b',
                                  message, re.IGNORECASE)
                if match:
                    status = int(match.group(1))
            current = current.__cause__ if current.__cause__ is not None else current.__context__
        versions = {}
        for package in ('huggingface-hub', 'hf-xet'):
            try:
                versions[package] = importlib.metadata.version(package)
            except importlib.metadata.PackageNotFoundError:
                versions[package] = 'not installed / unavailable'
        detail = '\nCaused by: '.join(chain)
        heading = (f'Hugging Face error | stage={stage} | '
                   f'attempt={attempt or "-"}/{HF_UPLOAD_ATTEMPTS} | HTTP {status or "?"}')
        print(heading)
        print(detail)
        print('SDK:', json.dumps(versions, ensure_ascii=False))
        diagnostic = {
            'time_utc': datetime.now(timezone.utc).isoformat(), 'stage': stage,
            'attempt': attempt, 'http_status': status, 'repo_id': HF_REPO_ID,
            'run_name': run_name, 'sdk_versions': versions, 'error_chain': chain,
        }
        log_path = run_dir / 'logs' / 'hf_upload_error.log'
        try:
            log_path.parent.mkdir(parents=True, exist_ok=True)
            with log_path.open('a', encoding='utf-8') as log:
                log.write(redact(json.dumps(diagnostic, ensure_ascii=False)) + '\n')
                log.write(redact(''.join(traceback.format_exception(
                    type(error), error, error.__traceback__))) + '\n')
            print('Log lỗi upload:', log_path)
        except OSError:
            print('Không ghi được file log; thông báo lỗi đã in ở trên.')
        return status, detail

    api = HfApi(token=token)  # Không login(), không lưu token vào file hay biến môi trường.
    try:
        api.repo_info(repo_id=HF_REPO_ID, repo_type='dataset')
    except Exception as error:
        status, detail = report_upload_error(error, 'repo_info')
        raise RuntimeError(f'Không truy cập được dataset {HF_REPO_ID} (HTTP {status or "?"}). '
                           f'\nLỗi gốc: {detail}\n'
                           'Kiểm tra Internet, repo đích và quyền của HF_TOKEN.') from None

    file_records = []
    for local_path, remote_path in files:
        digest = hashlib.sha256()
        with local_path.open('rb') as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b''):
                digest.update(block)
        file_records.append({'path_in_repo': remote_path, 'bytes': local_path.stat().st_size,
                             'sha256': digest.hexdigest()})
    upload_manifest = {
        'format_version': 1,
        'source_notebook': 'crawl_corpus_kaggle_hf_upload.ipynb',
        'repo_id': HF_REPO_ID, 'run_name': run_name,
        'id_from_inclusive': ID_FROM, 'id_to_inclusive': ID_TO,
        'dataset_revision': manifest['dataset_revision'],
        'source_sha256': manifest['source_sha256'],
        'complete': True, 'corpus_policy': 'accepted_only',
        'counts': {key: manifest[key] for key in ('selected', 'completed', 'pending', 'accepted')},
        'prepared_at_utc': datetime.now(timezone.utc).isoformat(),
        'files': file_records,
    }
    upload_manifest_path = run_dir / 'hf_upload_manifest.json'
    upload_manifest_path.write_text(json.dumps(upload_manifest, ensure_ascii=False, indent=2) + '\n',
                                    encoding='utf-8')
    files.append((upload_manifest_path, f'manifests/{run_name}/upload_manifest.json'))
    print(f'Dataset đích: {HF_REPO_ID} | Run: {run_name} | Accepted: {manifest["accepted"]}')
    for local_path, remote_path in files:
        print(f'  {remote_path} ({local_path.stat().st_size / 1024**2:.2f} MiB)')

    for attempt in range(1, HF_UPLOAD_ATTEMPTS + 1):
        try:
            # Một commit cho corpus, manifest và ZIP của cùng run; chỉ thêm/cập nhật các path này.
            # Tạo lại operations khi retry vì SDK có thể thay đổi chúng trong quá trình upload.
            commit = api.create_commit(
                repo_id=HF_REPO_ID, repo_type='dataset',
                operations=[CommitOperationAdd(path_in_repo=remote, path_or_fileobj=str(local))
                            for local, remote in files],
                commit_message=f'Upload crawl {run_name} ({manifest["accepted"]} accepted)',
            )
            break
        except Exception as error:
            status, detail = report_upload_error(error, 'create_commit', attempt)
            retryable = status is None or status >= 500 or status in (408, 409, 429)
            if not retryable or attempt == HF_UPLOAD_ATTEMPTS:
                raise RuntimeError(f'Upload chưa thành công (HTTP {status or "?"}, '
                                   f'{type(error).__name__}).\nLỗi gốc: {detail}\n'
                                   'Xem logs/hf_upload_error.log để biết chi tiết; '
                                   'output local vẫn còn, có thể chạy lại riêng mục 10.') from None
            print(f'Upload thất bại (lượt {attempt}/{HF_UPLOAD_ATTEMPTS}); đang thử lại...')
            time.sleep(min(2 ** attempt, 30))

    result = {
        'status': 'uploaded', 'repo_id': HF_REPO_ID, 'run_name': run_name,
        'commit_url': commit.commit_url, 'commit_oid': commit.oid,
        'files': [remote for _, remote in files],
    }
    (run_dir / 'hf_upload_result.json').write_text(
        json.dumps(result, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    print('Upload thành công:', commit.commit_url)
    for _, remote in files:
        print(f'https://huggingface.co/datasets/{HF_REPO_ID}/blob/main/{quote(remote, safe="/")}')
    return result


hf_upload_result = upload_current_run_to_hf()
